# ★ V12 是什么：冻结架构 + 序数/软标签目标（B4）

**一句话**：架构保持冻结（mean 池化 + 4 轮），检验 **B4 假设：点回归在「追噪声」**——
标签是 10 名标注者 0–5 整数评分的均值（单例句评分噪声 sd ≈ 0.36–0.50），把每行的
(均值 m, 标准差 s, n=10) 重建成 6 个桶 {0..5} 上的软分布，用软交叉熵训练。

| 臂 | 读出头 | 损失 | 检验什么 |
|---|---|---|---|
| `joint_base` | Linear(1024→2) | huber+guard（冻结配方） | 对照 |
| `joint_ord` | Linear(1024→12)→6 bin | 软 CE + guard 保险丝 | 序数包 vs 点回归 |
| `joint_ordmix` | Linear(1024→12)→6 bin | 软 CE + huber + guard | 混合能否兼得 |

**软标签**：`q_k = Φ((e_hi−m)/s) − Φ((e_lo−m)/s)`（6 桶、两端吃截断尾量、s=0 退化为
单点）；读数 `ŷ = Σ k·p_k`（桶凸组合 ⇒ 值域 [0,5] 由结构保证，提交格式不变）。
**文献**：软标签分布匹配（Müller et al. 2019：熵正则与校准收益）；有序分布回归思想
（与 CORAL 的序数目标同族，未借用其秩一致机制）；Kendall & Gal 2017（aleatoric
噪声显式化）；Northcutt et al. 2021（带噪人类标签）。
**如实披露**：CE 的最优读数是 EV(q) ≠ m（截断分桶把质量拉向两端，宽 s 时 U 形）；
现实 s ∈ [0.36, 0.6] 下 EV(q)−m 的离散化 RMSE 下限 ≈ 0.04–0.08，与 RMSE 判读阈值
0.0463 同量级 ⇒ 本实验的 RMSE 假设是在该结构障碍下的保守检验。

**预算**：3 臂 × 45 min ≈ 2.3 h + 开销 ≈ 2.6 h ≤ 8 h（双口径投影照常打印；
超预算先裁 joint_ordmix）。

**Step10 预注册判读（阈值 0.0652/0.1526/0.0463）**：
- **R4 红线**：三臂指标2 非 nan、退化目标数 = 0、轨迹非空（违反即停）；
- **反向否决**：任一序数臂 Δm1 < −0.0652 或 Δm2 < −0.1526 ⇒ 该臂出局；
- **采纳**：幸存臂中 RMSE 最低者 ΔRMSE ≤ −0.0463 ⇒ 采纳（RMSE 假设），
  或其 Δm1 ≥ +0.0652 ⇒ 采纳（m1 假设）；否则全部不采纳，保持冻结配方。

# ComPartMENT V12：XLM-RoBERTa-large（V12：冻结架构 + 序数/软标签目标）

**任务**：给定一个多词表达（MWE）和一句真实语料，输出该表达在这条语境里的
组合性分数（0–5 连续值，10 名标注者评分的均值）。

**架构 = 上游判读冻结配方**：mean 掩码均值池化；4 轮、t3 全参、有效 batch 16、lr 2e-5。
**V12 检验 B4 假设：点回归在「追噪声」**——标签是 10 名标注者评分的均值（单例句
评分噪声 sd ≈ 0.36–0.50），把每行的 (均值 m, 标准差 s, n=10) 重建成 6 个桶 {0..5}
上的软分布 q，用软交叉熵 CE(p, q) 训练（期望值读数 ŷ = Σ k·p_k 天然落在 [0,5]）。

| 臂 | 头 | 损失 | 检验什么 |
|---|---|---|---|
| `joint_base` | Linear(1024→2) | huber+guard（冻结配方，对照） | — |
| `joint_ord` | Linear(1024→12)→6 bin | **软 CE + guard 保险丝** | 序数包 vs 点回归 |
| `joint_ordmix` | 同上 | **软 CE + huber + guard** | 混合能否兼得 |

**跑什么**：一路执行到底，自动完成 3 个联合臂（4 轮、seed 42；数据切分恒 42 ⇒
验证集逐行对齐，配对差值合法）。Step10 出预注册判读：反向否决（m1/m2 显著变差
的臂出局）→ RMSE 假设（幸存臂中 RMSE 最低者 ΔRMSE ≤ −0.0463 ⇒ 采纳）→
m1 假设（Δm1 ≥ +0.0652 ⇒ 采纳）→ 否则保持冻结配方。

**与 v8–v11 的关系**：同源管线（数据 / 切分 / 指标 / 档位逐字相同）；
唯一变量 = 训练目标（点回归 → 序数分布）与读出头。

## Step1 导入包

**这一节要解决什么**：把「会随机器、随人变」的东西全部钉死在一个地方——
模型下载端点、随机种子、设备、精度、序列长度、批大小、模型 id、学习率、运行计划。
其余 8 个 Step 一律不写死任何常量，只从这里取。

**为什么必须这样**：本项目要跑 2 个模型 × 4 个子集 × 后续若干组消融。
如果 `lr=2e-5` 写在 Step6、`max_length=160` 写在 Step3，
那么「换个模型跑一遍」就要改 6 个地方，改漏一处就会得到一张**无法归因**的对照表。
对照实验的可信度 100% 取决于配置是否只有一个入口。

**先说一件本项目特有的事：模型权重从哪儿下。**

云端 GPU（Kaggle / Colab / 阿里云等海外实例）**可以直连 `huggingface.co`**，
本机直连通常不通。所以默认策略是：

1. **先探测官方源**（`https://huggingface.co`）是否真的可达——用一次真实的 HTTPS 请求，
   不是 ping，也不是 socket 探测（很多网络环境 443 端口能连但 HTTP 被重置）；
2. 官方源可达 → 用官方；不可达 → 换镜像站 `https://hf-mirror.com`；
3. **下载时再兜一层**：即便探测时通了、下载时断了，也会自动换另一个端点重试。

探测与端点切换都发生在 `import transformers` **之前**或**显式传参**，
不依赖 `HF_ENDPOINT` 全局变量的缓存时机——这是「我明明改了镜像怎么还在走官方」
这类现象最常见的来源。

In [ ]:

# ============ 0. 端点选择：必须在 import transformers 之前完成 ============





import os

import urllib.request







os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")



# =============================================================================













# =============================================================================

_GPU_INDEX = int(os.environ.get("COMPARTMENT_GPU", "0"))

os.environ["CUDA_VISIBLE_DEVICES"] = str(_GPU_INDEX)

print(f"[环境] CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} "

      f"（本进程只看见这 1 张卡 → Trainer 不会套 nn.DataParallel → 梯度显存不翻倍）")



OFFICIAL_ENDPOINT = "https://huggingface.co"

MIRROR_ENDPOINT = "https://hf-mirror.com"













HF_MODE = os.environ.get("COMPARTMENT_HF_MODE", "auto").strip().lower()





def probe_http(url, timeout=8):
    """发一次真实的 HTTPS 请求，判断 URL 是否可达。
    【为什么不 ping / 不测 socket】很多网络环境 TCP 443 能握手但 HTTP 被 RST，
    只有真实请求才能反映「能不能下模型」。"""

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "compartment-probe"})

        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return 200 <= resp.status < 500

    except Exception:
        return False





def choose_endpoints(model_id):
    """返回按优先级排序的端点列表；第一个失败会自动尝试下一个。"""

    if HF_MODE == "cache":
        print("端点策略: cache（只用本地缓存，不联网）")

        return []

    if HF_MODE == "mirror":
        print("端点策略: mirror（强制镜像站）")

        return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]

    if HF_MODE == "official":
        print("端点策略: official（强制官方源）")

        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]

    api = f"{OFFICIAL_ENDPOINT}/api/models/{model_id}"

    ok = probe_http(api)

    print(f"官方源连通性探测: {'通' if ok else '不通'}  ({api})")

    if ok:
        print("→ 使用官方 huggingface.co（云端 GPU 直连即走这条）")

        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]

    print("→ 官方源不可达，改用镜像站 hf-mirror.com")

    return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]





os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


In [ ]:

# ============ 1. 导入其余依赖 ============

import gc

import inspect

import json

import math

import platform

import re

import shutil

import time

import unicodedata

import warnings

from collections import Counter

from pathlib import Path



import numpy as np

import pandas as pd

import torch

import torch.nn as nn

import torch.nn.functional as F

from scipy import stats

from scipy.stats import ConstantInputWarning



import matplotlib.pyplot as plt

from matplotlib import font_manager



from datasets import Dataset

from transformers import (

    AutoConfig,

    AutoModel,

    AutoTokenizer,

    Trainer,

    TrainingArguments,

    set_seed,

)

from transformers.modeling_outputs import ModelOutput

from torch.utils.data import Sampler



print("torch", torch.__version__)


In [ ]:

# ============ 2. 绘图字体与图注语言：统一使用英文 ============





_FONT_CANDIDATES = [

    "Helvetica Neue", "Helvetica", "Arial", "Arial Unicode MS",

    "Liberation Sans", "DejaVu Sans", "Verdana", "Tahoma",

]





def _pick_latin_font():
    """挑一个系统里已装、能显示拉丁字母与德语变音的字体。"""

    try:
        installed = {f.name for f in font_manager.fontManager.ttflist}

    except Exception:
        installed = set()

    for _name in _FONT_CANDIDATES:
        if _name in installed:
            return _name

    return "DejaVu Sans"





_CHOSEN_FONT = _pick_latin_font()

plt.rcParams["font.sans-serif"] = [_CHOSEN_FONT] + [f for f in _FONT_CANDIDATES if f != _CHOSEN_FONT]

plt.rcParams["axes.unicode_minus"] = False

print("绘图字体:", _CHOSEN_FONT, "| 图注语言: en (English only)")





PLOT_LANG = "en"



_L10N = {

    "zh": {

        "gold": "人工评分", "pred": "模型预测",

        "gold_std": "人工评分的组内标准差（金标准）", "pred_std": "模型预测的组内标准差",

        "m1": "指标1_ρ", "m2": "指标2_ρ", "m3": "指标3_RMSE",

        "rho_up": "Spearman ρ（越大越好）", "rmse_down": "RMSE（越小越好）",

        "figA": "图A  预测 vs 真值", "figB": "图B  评估标准", "figC": "图C  指标2 的可视化",

        "figD": "图D  五次运行的合计指标对照", "figE": "图E  参考对照",

        "not_standard": "非评估标准", "trained": "训练后模型", "random_init": "随机初始化",

        "oracle": "type-level oracle", "single": "单子集训练", "joint": "联合训练（覆盖全部设置）",

        "joint_short": "联合\n(四子集)", "single_short": "单子集\n{short}",

        "m2rho": "指标2 ρ =", "n_targets": "目标数 =",

        "const_std": "预测 std 恒为 0 的目标数 =", "n_settings": "个评测设置",

        "gold_axis": "横轴金标准、纵轴系统输出", "eval_std": "评估标准",

        "short": {},

    },

    "en": {

        "gold": "Gold score", "pred": "Prediction",

        "gold_std": "Gold within-target std", "pred_std": "Predicted within-target std",

        "m1": "Metric1 rho", "m2": "Metric2 rho", "m3": "Metric3 RMSE",

        "rho_up": "Spearman rho (higher is better)", "rmse_down": "RMSE (lower is better)",

        "figA": "Fig A  Prediction vs Gold", "figB": "Fig B  Evaluation standard",

        "figC": "Fig C  Metric 2", "figD": "Fig D  Aggregate metrics across runs",

        "figE": "Fig E  Reference systems",

        "figF": "Fig F  Training loss curves",

        "loss": "Loss", "step": "Step", "epoch": "Epoch",

        "train_loss": "Train loss", "eval_loss": "Eval loss",

        "loss_all_runs": "Training loss of every run",

        "loss_main": "Train vs eval loss (joint run)",

        "untrained": "untrained", "overfit_gap": "train - eval gap at last epoch",

        "not_standard": "reference only", "trained": "Trained", "random_init": "Random init",

        "oracle": "type-level oracle", "single": "Single-subset", "joint": "Joint (all settings)",

        "joint_short": "Joint\n(4 subsets)", "single_short": "single\n{short}",

        "m2rho": "Metric2 rho =", "n_targets": "targets =",

        "const_std": "targets with constant pred std =", "n_settings": "settings",

        "gold_axis": "gold (x) vs system (y)", "eval_std": "Evaluation standard",

        "short": {"英·复合词": "EN-NN", "德·复合词": "DE-NN",

                  "英·小品词": "EN-PV", "德·小品词": "DE-PV"},

    },

}





def T(key, **kw):
    """取图上文字：有中文字体用中文，没有就用英文。"""

    v = _L10N[PLOT_LANG].get(key, key)

    return v.format(**kw) if kw else v





def short_of(subset):
    """子集的短标签（跟随图上语言）。"""

    zh = SUBSET_SPECS[subset]["short"]

    return _L10N[PLOT_LANG]["short"].get(zh, zh)


In [ ]:

# ============ 全局配置：整个 notebook 只有这一处可以改数值 ============

SEED = 42



_DEFAULT_CFG = {'plan': '', 'epochs': '', 'data_mode': 'orig'}



MODEL_REGISTRY = {

    "xlmr": dict(

        model_id="FacebookAI/xlm-roberta-large",

        attn_impls=("sdpa", "eager"),

    ),

}

MODEL_KEY = (os.environ.get("COMPARTMENT_MODEL_KEY", "").strip() or "xlmr")





SUBSET_SPECS = {

    "en-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], stds=["ModStd", "HeadStd"], lang="en", task="nn", short="英·复合词"),

    "de-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], stds=["ModStd", "HeadStd"], lang="de", task="nn", short="德·复合词"),

    "en-pv": dict(key="ParticleVerb", cols=["Avg"],             stds=["Std"],           lang="en", task="pv", short="英·小品词"),

    "de-pv": dict(key="ParticleVerb", cols=["Avg"],             stds=["Std"],           lang="de", task="pv", short="德·小品词"),

}

SUBSET_ORDER = ["en-nn", "de-nn", "en-pv", "de-pv"]





N_OUTPUTS = 2





RUN_PLAN = [

    ("joint_base", None),

    ("joint_ord", None),

    ("joint_ordmix", None),

]

_RUN_PLAN_ENV_RAW = os.environ.get("COMPARTMENT_RUN_PLAN", "").strip()

_RUN_PLAN_ENV = _RUN_PLAN_ENV_RAW or _DEFAULT_CFG["plan"]





def _filter_run_plan(plan, spec):
    """按环境变量/注入档裁剪运行计划（名字 = 运行名本身）。"""

    if (not spec) or spec.lower() == "all":
        return list(plan)

    want = {x.strip() for x in spec.split(",") if x.strip()}

    out = [(m, s) for (m, s) in plan if (m if s is None else f"single_{s}") in want]

    if not out:
        raise ValueError(f"COMPARTMENT_RUN_PLAN={spec!r} 没有匹配到任何运行；可选：{[m for m, _ in plan]}")

    return out





RUN_PLAN = _filter_run_plan(RUN_PLAN, _RUN_PLAN_ENV)





V12_ARMS = {'joint_base': {'loss_mode': 'base', 'head_kind': 'linear'},

 'joint_ord': {'loss_mode': 'ord', 'head_kind': 'ordinal'},

 'joint_ordmix': {'loss_mode': 'ordmix', 'head_kind': 'ordinal'}}

assert set(V12_ARMS) == {m for m, _ in RUN_PLAN}, "运行计划与三臂定义不一致"

SUBSET_LOSS_PROFILE = {k: dict(v) for k, v in {'en-nn': {'order': 0.0, 'mse': 0.0, 'guard': 0.06, 'xrank': 0.0, 'kappa': 0.15},

 'de-nn': {'order': 0.0, 'mse': 0.0, 'guard': 0.06, 'xrank': 0.0, 'kappa': 0.15},

 'en-pv': {'order': 0.0, 'mse': 0.0, 'guard': 0.06, 'xrank': 0.0, 'kappa': 0.5},

 'de-pv': {'order': 0.0, 'mse': 0.0, 'guard': 0.06, 'xrank': 0.0, 'kappa': 0.5}}.items()}

POOL_MODE = "mean"

LOSS_MODE = "base"

HEAD_KIND = "linear"

print(f"V12 三臂：{list(V12_ARMS)}（对照 = 冻结配方 pool=mean | huber+guard）")









SOFT_BINS     = 6

GUARD_FUSE_W  = 1.0





MAX_LENGTH   = 160

EFFECTIVE_BATCH_BY_MODEL = {"xlmr": 16}

TARGET_EFFECTIVE_BATCH = EFFECTIVE_BATCH_BY_MODEL[MODEL_KEY]

BATCH_SIZE   = 8

GRAD_ACCUM   = max(1, int(round(TARGET_EFFECTIVE_BATCH / BATCH_SIZE)))

PROBE_MEMORY = True

LR           = 2e-05

EPOCHS       = 4

JOINT_EPOCHS = 4

_EP_OVERRIDE = (os.environ.get("COMPARTMENT_EPOCHS", "").strip()

                or str(_DEFAULT_CFG.get("epochs", "") or ""))

if _EP_OVERRIDE:
    EPOCHS = int(float(_EP_OVERRIDE))

    JOINT_EPOCHS = int(float(_EP_OVERRIDE))

    print(f"★ 轮数覆盖生效：EPOCHS = JOINT_EPOCHS = {EPOCHS}")

VAL_RATIO    = 0.2

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

HUBER_DELTA  = 1.0



USE_8BIT_OPT   = "auto"

USE_GRAD_CKPT  = False

FIT_MODE       = "t3"

FIT_REF_PARAMS = 5.6e8

FIT_SAFETY     = 0.90

FIT_PROBE_RESERVE_GB = 0.8

GPU_INDEX      = int(os.environ.get("COMPARTMENT_GPU", "0"))

TIME_BUDGET_H = 8.0

CLOUD_SPS = 0.087



AUTO_BALANCE = True

ORDER_TAU    = 1.0

XRANK_MARGIN = 0.05

XRANK_TAU    = 2.0

BALANCE_EMA_BETA  = 0.99

LOSS_FINITE_CHECK_EVERY = 50

BEST_METRIC_KEY = "合计|_综合分"

Z_M1 = 0.023

Z_M2 = 0.055

Z_M3 = 0.046

GROUPED_BATCHING = True

SMOKE = (os.environ.get("COMPARTMENT_SMOKE", "0").strip() == "1")

SMOKE_N_ROWS = 400

SAVE_FINAL_FOR = set()



if SMOKE:
    MAX_LENGTH = 64

    EPOCHS = 1

    JOINT_EPOCHS = 1

    BATCH_SIZE = 4

    RUN_PLAN = [("single", "en-nn")] + [(m, None) for m, _ in RUN_PLAN]

    if _RUN_PLAN_ENV_RAW and _RUN_PLAN_ENV_RAW.lower() != "all":
        RUN_PLAN = _filter_run_plan(RUN_PLAN, _RUN_PLAN_ENV_RAW)



print(json.dumps(dict(MODEL_KEY=MODEL_KEY, POOL_MODE=POOL_MODE,

                      MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM,

                      TARGET_EFFECTIVE_BATCH=TARGET_EFFECTIVE_BATCH,

                      LR=LR, EPOCHS=EPOCHS, JOINT_EPOCHS=JOINT_EPOCHS,

                      WARMUP_RATIO=WARMUP_RATIO, HUBER_DELTA=HUBER_DELTA,

                      SUBSET_LOSS_PROFILE=SUBSET_LOSS_PROFILE, AUTO_BALANCE=AUTO_BALANCE,

                      ORDER_TAU=ORDER_TAU, XRANK_MARGIN=XRANK_MARGIN, XRANK_TAU=XRANK_TAU,

                      BALANCE_EMA_BETA=BALANCE_EMA_BETA,

                      SOFT_BINS=SOFT_BINS, GUARD_FUSE_W=GUARD_FUSE_W, CLOUD_SPS=CLOUD_SPS,

                      Z_M1=Z_M1, Z_M2=Z_M2, Z_M3=Z_M3,

                      LOSS_FINITE_CHECK_EVERY=LOSS_FINITE_CHECK_EVERY,

                      BEST_METRIC_KEY=BEST_METRIC_KEY,

                      SMOKE=SMOKE, RUN_PLAN=[list(x) for x in RUN_PLAN]),

                 ensure_ascii=False, indent=1))


In [ ]:

# ============ 4. 硬件探测与精度自动降级 ============

def detect_device():
    """返回 (device, use_bf16, use_fp16, use_tf32)，三档自动降级不写死。

    bf16 判据以算力为准：torch.cuda.is_bf16_supported() 会把「软件模拟」也算支持，
    T4 等老卡照样返回 True ⇒ 只有 major ≥ 8 才用原生 bf16，major ≥ 7 用 fp16。"""

    if torch.cuda.is_available():
        _n_gpu = torch.cuda.device_count()

        _gi = GPU_INDEX % max(1, _n_gpu)





        torch.cuda.set_device(_gi)

        _cap = torch.cuda.get_device_capability(_gi)

        _native_bf16 = (_cap[0] >= 8) and torch.cuda.is_bf16_supported()

        _fp16 = (not _native_bf16) and (_cap[0] >= 7)

        return "cuda", _native_bf16, _fp16, True

    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():


        return "mps", False, False, False

    return "cpu", False, False, False





DEVICE, USE_BF16, USE_FP16, USE_TF32 = detect_device()





HW_BF16, HW_FP16 = USE_BF16, USE_FP16





NUM_WORKERS = 0 if platform.system() == "Darwin" else max(1, (os.cpu_count() or 2) // 2)

DATALOADER_PERSISTENT = NUM_WORKERS > 0





if USE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True

    torch.backends.cudnn.allow_tf32 = True



set_seed(SEED)

print({

    "device": DEVICE, "bf16": USE_BF16, "fp16": USE_FP16, "tf32": USE_TF32,

    "workers": NUM_WORKERS, "torch": torch.__version__, "platform": platform.system(),

    "python": platform.python_version(),

})



if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)

        print(f"GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "

              f"compute capability {_p.major}.{_p.minor} | "

              f"原生bf16={_p.major >= 8} | fp16={_p.major >= 7}")

    _total_cards = sum(torch.cuda.get_device_properties(_i).total_memory

                        for _i in range(torch.cuda.device_count())) / (1024 ** 3)

    print(f"物理卡：{torch.cuda.device_count()} 张可见（合计 {_total_cards:.1f} GiB）；"

          f"本 notebook 绑定物理 GPU{GPU_INDEX}，在进程内编号为 cuda:0")

    print("→ 单卡模式：Trainer 看到 n_gpu=1 → **不会**自动套 nn.DataParallel"

          "（DataParallel 的反向传播会额外复制一整份梯度，Qwen-1.5B 要多花 5.75 GiB）")


## Step2 加载数据集

**这一节要解决什么**：用最少的假设把原始 TSV 变成规整的 DataFrame，
并且**在载入这一步就把会泄漏答案的东西扔掉**。

**为什么载入阶段就要处理反泄漏**：本项目实测发现，
同一个 MWE 的各条例句在文件里是**按 ContextID 顺序连续排列**的，
而 ContextID 又与该目标的平均分高度相关（组内 Spearman 达 0.985）。
换句话说，**「这条例句在文件里的位置」本身就带着答案**。
如果先建特征再打乱，任何从行号派生的东西（下标、位次、窗口邻域）都会变成作弊通道。
所以顺序必须是：**先删 ContextID → 再物理打乱**。

**要保留什么**：`Compound` / `ParticleVerb`（MWE 本身）、`Mod`/`Head`/`Base`/`Particle`（成分）、
`ModAvg`/`HeadAvg`/`Avg`（标签）、`Context`（句子）。

In [ ]:

# ============ 数据目录解析：按「环境变量 → Kaggle → 仓库」顺序探测，不改代码即可换数据源 ============

CLOUD_TRAIN_DIR = "/kaggle/input/datasets/kintsugi0v0/compartment-data/train"



CLOUD_V2_DATA_ROOT = "/kaggle/input/datasets/kintsugi0v0/compartment-data-v2/train_data_v2"

CLOUD_AUG_DIR = CLOUD_V2_DATA_ROOT + "/filtered"





def resolve_data_root(start: Path) -> Path:
    """返回第一个同时含 4 个 *-train.tsv 的目录（原始训练数据）。

    兜底用递归 glob 全盘找 en-nn-train.tsv，但排除 filtered/ 目录下的命中——
    那是合成增广数据（与原始文件同名），不能被当成原始数据，否则增广模式会把
    合成数据拼两遍。
    """

    import glob as _glob

    cands = []

    env = os.environ.get("COMPARTMENT_TRAIN_DIR")

    if env:
        cands.append(Path(env).expanduser())

    cands.append(Path(CLOUD_TRAIN_DIR))

    root_env = os.environ.get("COMPARTMENT_ROOT")

    if root_env:
        cands.append(Path(root_env).expanduser() / "train")

    p = start.resolve()

    for cand in [p] + list(p.parents):
        if (cand / "train" / "en-nn-train.tsv").exists():
            cands.append(cand / "train")

            break

    for _hit in sorted(map(Path, _glob.glob("/kaggle/input/**/en-nn-train.tsv", recursive=True))):
        if "filtered" not in _hit.parts:
            cands.append(_hit.parent)

    _need = [f"{_n}-train.tsv" for _n in SUBSET_ORDER]

    for c in dict.fromkeys(cands):
        if all((c / _f).exists() for _f in _need):
            assert "filtered" not in c.parts, f"原始数据目录不能是 filtered/：{c}"

            return c

    raise FileNotFoundError(

        "找不到原始训练数据（需同时含 " + " / ".join(_need) + "）。已依次尝试：\n  "

        + "\n  ".join(str(c) for c in cands)

        + "\n请检查 Kaggle 数据集挂载（原始 TSV 应在 .../train_data_v2/ 根目录下），"

          "或设置环境变量 COMPARTMENT_TRAIN_DIR 指向该目录。")





WORK_DIR = Path.cwd()

DATA_ROOT = resolve_data_root(WORK_DIR)



OUTPUT_ROOT = Path(os.environ.get("COMPARTMENT_OUTPUT", str(WORK_DIR / "outputs")))











DATA_MODE = "orig"

print("DATA_MODE=orig（只用原始数据）")



ORIG_TSV_MD5 = {

    "en-nn": "58695bd9d5a1f1d39ac14a2d3364f00a",

    "de-nn": "0ed2245c5fe2d7b354d97d0a75ef1039",

    "en-pv": "0648c8a441ec731e5d4dd81b774933ee",

    "de-pv": "7c6e353097593d850c21c38885bd0767",

}

if os.environ.get("COMPARTMENT_SKIP_MD5", "") != "1":
    import hashlib

    _bad_md5 = []

    for _n, _want in ORIG_TSV_MD5.items():
        _h = hashlib.md5((DATA_ROOT / f"{_n}-train.tsv").read_bytes()).hexdigest()

        if _h != _want:
            _bad_md5.append((_n, _want, _h))

    if _bad_md5:
        raise RuntimeError("数据身份校验失败（大概率挂成了合成/替换数据集）："

                           + "; ".join(f"{n} 期望 {w} 实得 {h}" for n, w, h in _bad_md5)

                           + "。请挂载官方数据，或设 COMPARTMENT_SKIP_MD5=1 显式跳过。")

    print("数据身份校验通过：4 个官方 TSV 的 md5 指纹逐位匹配")


In [ ]:

# ============ 演示：原始数据长什么样 ============





_demo_name = SUBSET_ORDER[0]

_demo_spec = SUBSET_SPECS[_demo_name]

raw = pd.read_csv(DATA_ROOT / f"{_demo_name}-train.tsv", sep="\t")

print(f"{_demo_name} 原始形状: {raw.shape}")

print("列名:", list(raw.columns))

print(raw[[_demo_spec["key"]] + _demo_spec["cols"] + ["Context"]].head(3).to_string()[:700])





print("\n--- 标签分布 ---")

for c in _demo_spec["cols"]:
    s = raw[c]

    print(f"{c}: 均值 {s.mean():.3f} | 标准差 {s.std():.3f} | 最小 {s.min():.2f} | 最大 {s.max():.2f}")

wlen = raw["Context"].str.split().str.len()

print("\n--- 句子长度（词）分位 ---")

print(dict(zip(["p50", "p90", "p95", "p99", "max"],

                np.percentile(wlen, [50, 90, 95, 99, 100]).round(0).tolist())))

print("→ 句子很短，MAX_LENGTH=160 足够，不会因为截断丢掉关键信息。")


In [ ]:

# ============ 封装：载入一个子集并做反泄漏处理 ============

def load_subset(name, seed=SEED):
    """载入一个子集：去重 → 删 ContextID（行序泄漏）→ 物理打乱 → 统一两列标签与标准差。"""

    sp = SUBSET_SPECS[name]

    df = pd.read_csv(DATA_ROOT / f"{name}-train.tsv", sep="\t")

    df = df.drop_duplicates(subset=[sp["key"], "Context"]).reset_index(drop=True)

    if "ContextID" in df.columns:
        df = df.drop(columns=["ContextID"])

    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    df["y0"] = df[sp["cols"][0]].astype(float)

    df["m0"] = 1.0

    df["s0"] = df[sp["stds"][0]].astype(float)

    if len(sp["cols"]) > 1:
        df["y1"] = df[sp["cols"][1]].astype(float)

        df["m1"] = 1.0

        df["s1"] = df[sp["stds"][1]].astype(float)

    else:
        df["y1"] = 0.0

        df["m1"] = 0.0

        df["s1"] = 0.0

    if SMOKE and SMOKE_N_ROWS:
        _n_keep = max(1, SMOKE_N_ROWS // 5)

        keep = df[sp["key"]].drop_duplicates().head(_n_keep)

        df = df[df[sp["key"]].isin(set(keep))].reset_index(drop=True)

    return df





for _n in SUBSET_ORDER:
    _d = load_subset(_n)

    _sp = SUBSET_SPECS[_n]

    print(f"[{_n}] 形状 {_d.shape} | 目标数 {_d[_sp['key']].nunique()} | "

          f"每目标例句数中位 {_d.groupby(_sp['key']).size().median()} | "

          f"ContextID 是否还在: {'ContextID' in _d.columns}")


## Step3 数据预处理

这一节做四件事，每件都有明确的动机：

| 环节 | 做什么 | 为什么 |
|---|---|---|
| ① 定位 MWE | 在句子里标出 MWE 出现的那个 span | 任务问的是「这个 MWE 在**这句**里是不是被当成整体理解」。不告诉模型句中哪个 span 对应 MWE，它只能靠整句语义猜 |
| ② 拼输入模板 | `MWE: <mwe> \| Context: <ctx>` | 两个模型必须看到**逐字相同**的输入，否则「模型差异」里会混进「模板差异」 |
| ③ 目标级互斥切分 | 按 MWE 切 train/val，并复刻官方「先分箱后分层抽样」 | 按行切会把同一 MWE 的例句分到两边，模型等于「见过这个 MWE」再去预测它，指标被高估 |
| ④ 分词 + 动态 padding | 批量分词，batch 内补齐到 8 的倍数 | 定长补齐到 `max_length` 会把每条的注意力开销抬到 O(L²) 的上限，动态补齐只按本批最长序列付费 |

**定位为什么难**：本项目实测过定位率——
英语复合词 98.6%、德语复合词 95.9% 可以字面命中；
但英语小品词动词只有 33.6%（`crack down` 在句子里常写成 `cracking down`），
德语更只有 20.8%（德语可分动词写作 `abgehauen`，粒子粘在动词前，且中间插前缀 `ge-`）。
所以定位必须做**分级级联**，而不是一次精确匹配。

In [ ]:

# ============ 文本归一化与词干工具（定位级联的零件）============







def fold(s, ascii_mode=False):
    """小写 + NFKD 归一 + 去标点；ascii_mode=True 时把 ä/ö/ü 换成 ae/oe/ue。"""

    s = unicodedata.normalize("NFKD", str(s).lower()).replace("ß", "ss")

    s = re.sub(r"[^a-z0-9äöüß']", "", s)

    if ascii_mode:
        for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue")]:
            s = s.replace(a, b)

    return s





def stem_de(b):
    """德语词干：去掉常见后缀（ern/eln/en/er/e/t/ten/n）。"""

    for suf in ["ern", "eln", "en", "er", "e", "t", "ten", "n"]:
        if b.endswith(suf) and len(b) - len(suf) >= 4:
            return b[:-len(suf)]

    return b





def stem_en(w):
    """英语词干：去掉 ing/edly/ed/es/s/y/e。"""

    for suf in ["ing", "edly", "ed", "es", "s", "y", "e"]:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]

    return w





def stem_keys(base, lang):
    """取词干的前 5/4/3 个字符作为宽松匹配键。
    【为什么要多个长度】词干可能因屈折被切掉一截（crack → cracking），
    只用完整词干会漏；只用 3 个字符又太松。3/4/5 三档一起用，命中任一即算。"""

    ks = set()

    for v in (fold(base), fold(base, True)):
        st = stem_de(v) if lang == "de" else stem_en(v)

        for L in (5, 4, 3):
            if len(st) >= L:
                ks.add(st[:L])

    return ks





WORD_RE = re.compile(r"[a-zA-ZäöüÄÖÜß]+(?:[-'][a-zA-ZäöüÄÖÜß]+)*")

print("归一化与词干工具就绪")


In [ ]:

# ============ 演示：一条样本的定位过程 ============



_demo_df = load_subset(_demo_name)

_sample_row = _demo_df.iloc[0].to_dict()

_mwe = _sample_row[_demo_spec["key"]]

_context = _sample_row["Context"]

print("MWE   :", _mwe)

print("原句  :", _context[:170])





_toks = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(_context.lower())]

print(f"\n句子切成 {len(_toks)} 个词:", [t[0] for t in _toks][:14])





print("\n归一化与词干:")

print("  fold(MWE) =", fold(_mwe))

if _demo_spec["task"] == "nn":
    print("  词干键(复合词) =", sorted(stem_keys(_mwe, _demo_spec["lang"])))

else:
    print("  fold(基干)   =", fold(_sample_row["Base"]))

    print("  词干键(基干) =", sorted(stem_keys(_sample_row["Base"], _demo_spec["lang"])))

    print("  粒子         =", _sample_row["Particle"])





_m = re.search(re.escape(_mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"),

               _context.lower())

print("\n最小版本（只做 S0 精确匹配）:", ("命中 → " + repr(_context[_m.start():_m.end()])) if _m else "未命中")

print("→ 精确匹配在复合词上够用，但在小品词动词上只能覆盖约三分之一；")

print("   所以下一格把级联封装成一个函数，逐级放宽。")


In [ ]:

# ============ 封装：定位 + 标记 + 输入模板 ============

def _mark(text, start, end):
    """在 text 的 [start, end) 两侧插入 [[ ]] 标记。
    【为什么用 ASCII 方括号而不是 « »】DeBERTa 用 SentencePiece，
    没有收录的字符会被拆成 byte fallback，既浪费 token 又干扰模型；
    ASCII 方括号在两套 tokenizer 里都是正常字符。"""

    return text[:start] + "[[" + text[start:end] + "]]" + text[end:]





def locate_target(row, sp):
    """在上下文中定位 MWE，返回 (标记后的句子, 策略标签)。
    级联：S0 精确匹配 → S1 历史转写还原 → S2a 德语前缀屈折 → S2b 德语后置分离
    → S3 英语词干前缀 → S4 相邻窗口 → S5 未定位（与 analysis/run_eda.py 同源）。
    精确匹配覆盖率视子集差异极大（复合词 96–99%，英语小品词动词仅 34%），
    必须逐级放宽；策略标签标出每条样本的定位置信度，出问题能一眼定位是哪一环。"""

    ctx = str(row["Context"])

    low = ctx.lower()

    mwe = str(row[sp["key"]])

    if sp["task"] == "nn":
        base, part = mwe, ""

    else:
        base, part = str(row["Base"]), str(row["Particle"])





    m = re.search(re.escape(mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"), low)

    if m:
        return _mark(ctx, m.start(), m.end()), "S0 精确匹配"



    toks = [(mm.group(0), mm.start(), mm.end()) for mm in WORD_RE.finditer(low)]

    if not toks:
        return ctx, "S5 未定位"





    if fold(mwe, True) and fold(mwe, True) in fold(low, True):
        return ctx, "S1 历史转写还原"



    ks = stem_keys(base, sp["lang"])

    p = fold(part)

    pw = {fold(part), fold(part, True)}

    fw = [fold(t[0]) for t in toks]

    fa = [fold(t[0], True) for t in toks]



    if sp["lang"] == "de":


        for i2, t in enumerate(toks):
            if not fw[i2].startswith(p):
                continue

            r, ra = fw[i2][len(p):], fa[i2][len(p):]

            if any(k and (k in r or k in ra) for k in ks):
                return _mark(ctx, t[1], t[2]), "S2a 德语前缀型(屈折)"



        for i2 in range(len(toks)):
            if not any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
                continue

            for j2 in range(i2 + 1, min(i2 + 4, len(toks))):
                if fw[j2] in pw or fa[j2] in pw:
                    return _mark(ctx, toks[i2][1], toks[j2][2]), "S2b 德语后置型(动词分离)"

        return ctx, "S5 未定位"





    for i2, t in enumerate(toks):
        if any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
            return _mark(ctx, t[1], t[2]), "S3 英文词干前缀"



    for i2 in range(len(toks) - 1):
        two = fw[i2] + fw[i2 + 1]

        if any(k and k in two for k in ks):
            return _mark(ctx, toks[i2][1], toks[i2 + 1][2]), "S4 相邻窗口"

    return ctx, "S5 未定位"





def build_input(mwe, marked_context):
    """最终送进模型的文本。
    【为什么模板固定成这样】
      1) 两个模型必须收到**逐字相同**的输入，否则对照实验的变量就不干净；
      2) MWE 放在前面：分词截断从尾部截，MWE 在前可保证它一定进得了模型；
      3) 用 `|` 而不是换行：换行在两套 tokenizer 里的切分行为不同。"""

    return f"MWE: {mwe} | Context: {marked_context}"













_marked, _strat = locate_target(_sample_row, _demo_spec)

print("策略  :", _strat)

print("标记后:", _marked[:200])

print("最终输入:", build_input(_mwe, _marked)[:220])


In [ ]:

# ============ 定位覆盖率统计：把「标注好不好」变成可量化的数字 ============



for _n in SUBSET_ORDER:
    _sp = SUBSET_SPECS[_n]

    _d = load_subset(_n)

    _cnt = Counter()

    for _row in _d.to_dict("records"):
        _, _st = locate_target(_row, _sp)

        _cnt[_st] += 1

    _total = sum(_cnt.values())

    _loc = _total - _cnt.get("S5 未定位", 0) - _cnt.get("S1 历史转写还原", 0)

    print(f"{_n} 定位覆盖（n={_total}）成功标出 span {_loc / _total:.1%} | " +

          " | ".join(f"{k} {v}" for k, v in _cnt.most_common()))


In [ ]:

# ============ 切分：目标级互斥 + 复刻官方「先分箱后分层抽样」 ============

def split_by_target(df, sp, val_ratio=VAL_RATIO, seed=SEED, n_bins=5):
    """按「目标」互斥切分，并复刻官方的「先分箱后分层抽样」协议。
    【为什么不能按行切分】同一 MWE 的例句一旦分散在训练集和验证集，
       模型就等于「见过这个 MWE」再去预测它的另一句话，指标会被显著高估。
    【为什么要分箱】各目标的平均组合性差别很大（本项目实测德语复合词修饰语
       79.4% 的目标落在 [4,5]，几乎是天花板标签），直接随机切会让验证集的
       难度分布偏离训练集，分数不可比。
    【为什么分箱在目标层面做】官方协议是先按组合性分箱、再在目标层面按箱配额抽样；
       在目标层面分箱才能保证同一箱里的目标难度可比。"""

    key = sp["key"]

    tgt_mean = df.groupby(key)[sp["cols"]].mean().mean(axis=1)

    try:
        bins = pd.qcut(tgt_mean, n_bins, labels=False, duplicates="drop").to_numpy()

    except ValueError:
        bins = np.zeros(len(tgt_mean), dtype=int)



    rng = np.random.default_rng(seed)

    val_targets = []

    for b in np.unique(bins):
        idx = tgt_mean.index[bins == b].to_numpy()

        rng.shuffle(idx)

        k = max(1, int(round(len(idx) * val_ratio)))

        val_targets.extend(idx[:k].tolist())



    is_val = df[key].isin(set(val_targets)).to_numpy()

    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)


In [ ]:

# ============ 模型权重的下载与解析（端点自动回退）============







_ALLOW_PATTERNS = ["*.json", "*.txt", "*.model", "*.safetensors", "*.bin", "*.py", "*.tiktoken"]





def resolve_model_dir(repo_id):
    """把模型仓库下载到本地目录，返回本地路径。端点失败会自动回退。"""

    from huggingface_hub import snapshot_download

    endpoints = choose_endpoints(repo_id)

    if not endpoints:
        return repo_id

    last_err = None

    for ep in endpoints:
        try:
            path = snapshot_download(repo_id, endpoint=ep, allow_patterns=_ALLOW_PATTERNS)

            print(f"权重就绪: {repo_id} @ {ep}\n         -> {path}")

            return path

        except Exception as e:
            last_err = e

            print(f"从 {ep} 下载失败: {type(e).__name__}: {str(e)[:180]}")

    raise last_err







if SMOKE:
    _SMOKE_IDS = {"xlmr": "FacebookAI/xlm-roberta-large"}

    MODEL_SOURCE = _SMOKE_IDS[MODEL_KEY]

    print("SMOKE=True：使用微型随机主干，不下载权重 ->", MODEL_SOURCE)

else:
    MODEL_SOURCE = resolve_model_dir(MODEL_REGISTRY[MODEL_KEY]["model_id"])


In [ ]:

# ============ 分词与数据集构建（含软标签重建）============

def load_tokenizer(model_key, source):
    """加载 tokenizer（右补齐）。"""

    tok = AutoTokenizer.from_pretrained(source)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    tok.padding_side = "right"

    tok.model_max_length = 512

    return tok





def _soft_target(m, s, bins=SOFT_BINS):
    """(均值 m, 标准差 s, n=10) → bins 个桶 {0..bins-1} 上的软分布（求和恒 1）。

    截断正态分桶：q_k = Φ((e_hi−m)/s) − Φ((e_lo−m)/s)，两端桶吃掉截断尾量；
    s=0（标注者全一致）退化为最近桶上的单点分布。m 先夹到 [0, bins−1]。"""

    m = float(min(max(float(m), 0.0), float(bins - 1)))

    if s <= 0:
        q = [0.0] * bins

        q[int(round(m))] = 1.0

        return q

    edges = torch.tensor([-1e9] + [k + 0.5 for k in range(bins - 1)] + [1e9],

                         dtype=torch.float64)

    z = (edges - m) / float(s)

    cdf = 0.5 * (1.0 + torch.erf(z / 1.4142135623730951))

    q = (cdf[1:] - cdf[:-1]).clamp(min=0.0)

    q = q / q.sum().clamp(min=1e-12)

    return [float(x) for x in q]





def build_dataset(frame, tokenizer, max_length=MAX_LENGTH):
    """DataFrame → datasets.Dataset（下游用；文本 = 模板输入，soft = 6 桶软标签）。"""

    records = []

    for row in frame.to_dict("records"):
        sp = SUBSET_SPECS[row["subset_name"]]

        mwe = str(row[sp["key"]])

        marked, _ = locate_target(row, sp)

        records.append({

            "text": build_input(mwe, marked),

            "labels": [float(row["y0"]), float(row["y1"])],

            "loss_mask": [float(row["m0"]), float(row["m1"])],

            "soft": [_soft_target(row["y0"], row["s0"]), _soft_target(row["y1"], row["s1"])],

            "group_index": 0,

            "group_key": f'{row["subset_name"]}||{mwe}',

            "subset_index": int(row["subset_id"]),

        })

    ds = Dataset.from_list(records)

    codes = {k: i for i, k in enumerate(sorted(set(frame["group_key"].astype(str))))}

    ds = ds.map(lambda b: {"group_index": [codes[k] for k in b["group_key"]]}, batched=True)

    ds = ds.remove_columns(["group_key"])

    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_length),

                batched=True, remove_columns=["text"])

    return ds





def ds_labels(ds, n_out=None):
    """labels 列 → [N, n_out] float 数组（容忍底层 arrow 的展平行为）。"""

    n_out = n_out if n_out is not None else N_OUTPUTS

    arr = np.asarray(ds["labels"], dtype=float)

    if arr.ndim == 1:
        arr = arr.reshape(-1, n_out)

    return arr





tokenizer = load_tokenizer(MODEL_KEY, MODEL_SOURCE)

print("tokenizer:", type(tokenizer).__name__, "| pad:", tokenizer.pad_token,

      "| padding_side:", tokenizer.padding_side)


In [ ]:

# ============ 关键封装：一次运行的全部数据准备 ============









def prepare_run(subset_list, tokenizer):
    """按给定的子集列表准备一次运行所需的全部数据。
    返回 dict：train_df / val_df / train_ds / val_ds / eval_cols。
    【切分在子集内部做，再合并】官方的分层协议是**在子集内部**做的；
      而且四个子集的目标难度分布差异极大（德语复合词修饰语 79.4% 落在 [4,5]，
      英语小品词动词均值只有 2.46），混在一起切会得到难度分布错乱的验证集。
    【group_key 带子集名】保证 group_index 在跨子集时也不会撞号。"""

    frames_tr, frames_va = [], []

    for name in subset_list:
        sp = SUBSET_SPECS[name]

        df = load_subset(name)

        df["subset_name"] = name

        df["subset_id"] = SUBSET_ORDER.index(name)

        df["group_key"] = name + "||" + df[sp["key"]].astype(str)

        tr, va = split_by_target(df, sp)

        frames_tr.append(tr)

        frames_va.append(va)

    train_df = pd.concat(frames_tr, ignore_index=True)

    val_df = pd.concat(frames_va, ignore_index=True)

    train_ds = build_dataset(train_df, tokenizer)

    val_ds = build_dataset(val_df, tokenizer)



    eval_cols = [(n, c, j) for n in subset_list for j, c in enumerate(SUBSET_SPECS[n]["cols"])]

    return dict(subset_list=list(subset_list), train_df=train_df, val_df=val_df,

                train_ds=train_ds, val_ds=val_ds, eval_cols=eval_cols)







_demo_run = prepare_run(["en-nn"], tokenizer)

_tr, _va = _demo_run["train_df"], _demo_run["val_df"]

print(f"en-nn 训练集 {_tr.shape} | 验证集 {_va.shape}")

print("目标交集:", len(set(_tr["group_key"]) & set(_va["group_key"])), "（必须为 0）")

for _n, _c, _j in _demo_run["eval_cols"]:
    _col = "y0" if _j == 0 else "y1"

    print(f"  {_n}·{_c}: train 均值 {_tr[_col].mean():.3f} | val 均值 {_va[_col].mean():.3f}")

print("示例解码:", tokenizer.decode(_demo_run["train_ds"][0]["input_ids"])[:220])


In [ ]:

# ============ 动态 padding 的省算力实测 ============







_lens = [len(x) for x in _demo_run["train_ds"]["input_ids"]]

_fixed_total = len(_lens) * MAX_LENGTH

_dynamic_total = 0

for _i in range(0, len(_lens), BATCH_SIZE):
    _dynamic_total += min(BATCH_SIZE, len(_lens) - _i) * max(_lens[_i:_i + BATCH_SIZE])

print(f"序列长度: 中位 {int(np.median(_lens))} | 均值 {int(np.mean(_lens))} | 最长 {max(_lens)}")

print(f"定长补齐到 {MAX_LENGTH} 的 token 总量: {_fixed_total:,}")

print(f"动态补齐的 token 总量            : {_dynamic_total:,}")

print(f"→ 动态补齐省下 {(1 - _dynamic_total / _fixed_total):.1%} 的 token，"

      f"注意力开销按平方计，实际省得更多。")


In [ ]:

# ============ Collator：把样本拼成 batch ============

class CompartmentCollator:
    """labels/loss_mask/soft/group_index/subset_index 手工叠张量（连续值回归标签不能走
    -100 填充），input_ids/attention_mask 交给 tokenizer.pad（动态补齐到 8 的倍数）。"""



    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer

        self.pad_to_multiple_of = pad_to_multiple_of



    def __call__(self, features):
        model_inputs = [{k: v for k, v in f.items()

                         if k not in ("labels", "loss_mask", "soft",

                                      "group_index", "subset_index")}

                        for f in features]

        batch = self.tokenizer.pad(model_inputs, padding=True,

                                   pad_to_multiple_of=self.pad_to_multiple_of,

                                   return_tensors="pt")

        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.float32)

        batch["loss_mask"] = torch.tensor([f["loss_mask"] for f in features], dtype=torch.float32)

        batch["soft"] = torch.tensor([f["soft"] for f in features], dtype=torch.float32)

        batch["group_index"] = torch.tensor([f["group_index"] for f in features], dtype=torch.long)

        batch["subset_index"] = torch.tensor([f["subset_index"] for f in features], dtype=torch.long)

        return batch





collator = CompartmentCollator(tokenizer)

_demo_batch = collator([_demo_run["train_ds"][i] for i in range(4)])

for k, v in _demo_batch.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")

_sm = (_demo_batch["soft"].sum(dim=-1) - 1.0).abs().max()

assert float(_sm) < 1e-4, f"软标签分布求和偏离 1（实测 {_sm}）"

print("→ labels 是 [B, 2] 的 float 张量；soft 是 [B, 2, 6] 的软分布（每行求和=1）；"

      "pv 第 2 列由 loss_mask=0 屏蔽。")


## Step4 创建模型

**这一节要解决什么**：把「编码器」和「解码器」两个完全不同的架构，
装配成**同一个形状的模型**：`(input_ids, attention_mask) → 两个 0–5 的连续分数`。

### 两种「读出方式」

两条基线各自验证一个机制：DeBERTa 用回归头，Qwen 完全不加回归头。
两者的**输入端完全一样**，只有末端不同：

```
【A】回归头（DeBERTa，head_mode = "regression"）

AutoModel 主干 ──► last_hidden_state [B, L, H]
                          │
              掩码均值池化(编码器) / 取最后一个真实 token(解码器)
                          │
                       [B, H]
                          │
                  Dropout → Linear(H, n_out)          ← 新建的回归头
                          │
                    logits [B, n_out]
                          │
                  × 5 并落在 (0,5)

【B】无回归头（Qwen，head_mode = "lm_head"）

Qwen2ForCausalLM ──► 主干 ──► last_hidden_state [B, L, H]
                                        │
                            取锚点（"\nModifier: "）最后一个 token 处
                                        │
                                    [B, H]
                                        │
                   主干**自带**的 LM 头（不新建任何层，与 embedding 共享权重）
                                        │
                              全词表 logits [B, V]
                                        │
                        只取 "0"–"5" 这 6 个 token（id 15…20）
                                        │
                             p = softmax(6 个 logit)
                                        │
                    分数 = Σ_k k·p_k  ∈ [0,5]     ← 6 个数的凸组合
```

**为什么「期望值」这个读法是对的**：标签本来就是 0–5 的**连续均值**
（实测：每个标签是 10 个整数评分的均值，粒度 0.1，唯一值 47–51 个）。
所以「标签 3.5」对应的天然分布就是「3 与 4 的概率混合」，
而不是「3 或 4」—— 模型输出一个数字 token 的分布、再取期望，
恰好与标签的生成方式**同构**。

**为什么不用 `AutoModelForSequenceClassification`**：两条理由，都很硬。

1. **任务类型不匹配**。强行用 `num_labels=6` + 交叉熵，会把「4.2 分」和「4.0 分」的差别完全抹平，
   而指标 1（组合性相关）恰恰只看**排序**——抹平排序等于直接放弃指标 1。
   （注意：lm_head 模式里出现的「6」**不是**把标签离散化成 6 类，
   而是模型本来就要生成的 6 个 token；两回事。）
2. **接口不统一**。DeBERTa 的 `DebertaV2ForSequenceClassification` 与
   Qwen 的 `Qwen2ForSequenceClassification` 在 pooling、损失、logits 形状上都不一致。
   自己写一个 20 行的头，两个模型才能**逐行同构**，
   对照实验里唯一的变量才真的是「架构」而不是「框架细节」。

**为什么回归头那条要 ×5**：标签的量纲是 0–5，模型是无约束实数输出。
用 `sigmoid × 5` 把输出**先验地**限制在合法区间，
这样即使训练早期发散，预测值也不会跑到 [-100, 100] 这种让 RMSE 直接爆掉的范围。
（lm_head 模式不需要这一步：期望值是 6 个概率的凸组合，**天然**落在 [0,5]。）

In [ ]:

# ============ 统一：把模型 config 缩到小尺寸（并同步所有派生维度）============

def shrink_config(cfg, num_layers=2, hidden_size=64, num_heads=4, tokenizer=None):
    """把 config 缩小到一个小尺寸，**同时同步所有派生维度**。
    不同架构的「注意力内部宽度」不都等于 hidden_size：
      · DeBERTa-v2 的注意力内部维度 = attention_head_size × num_attention_heads，
        只改 heads 不同步它，前向会抛 mat1/mat2 形状不匹配；
      · Qwen2 的 head_dim、DeBERTa-v2 的 embedding_size 同理必须跟着改。
    防御约定：缩尺寸只走这一个函数，任何地方不许直接改 cfg 的尺寸字段。
    """

    cfg.num_hidden_layers = num_layers

    cfg.hidden_size = hidden_size

    cfg.num_attention_heads = num_heads

    cfg.intermediate_size = hidden_size * 2



    if hasattr(cfg, "attention_head_size"):
        cfg.attention_head_size = hidden_size // num_heads

    if hasattr(cfg, "head_dim"):
        cfg.head_dim = hidden_size // num_heads

    if hasattr(cfg, "embedding_size"):
        cfg.embedding_size = hidden_size

    if hasattr(cfg, "num_key_value_heads"):
        cfg.num_key_value_heads = min(int(getattr(cfg, "num_key_value_heads", 1)), num_heads)

    if tokenizer is not None:




        cfg.vocab_size = max(cfg.vocab_size, len(tokenizer))

    return cfg





# ============ 演示：主干一次前向，看看 last_hidden_state 长什么样 ============







_demo_cfg = shrink_config(AutoConfig.from_pretrained(MODEL_SOURCE), tokenizer=tokenizer)

_demo_backbone = AutoModel.from_config(_demo_cfg)

_demo_backbone.eval()



_demo_ids = tokenizer("MWE: account book | Context: He read his [[account book]] last night.",

                      return_tensors="pt")

with torch.no_grad():
    _demo_out = _demo_backbone(input_ids=_demo_ids["input_ids"],

                               attention_mask=_demo_ids["attention_mask"])

_h = _demo_out.last_hidden_state

print("主干类型:", type(_demo_backbone).__name__)

print("config: hidden_size=%d num_attention_heads=%d attention_head_size=%s head_dim=%s" % (

    _demo_cfg.hidden_size, _demo_cfg.num_attention_heads,

    getattr(_demo_cfg, "attention_head_size", "-"), getattr(_demo_cfg, "head_dim", "-")))

print("last_hidden_state shape:", tuple(_h.shape))



assert _h.shape[-1] == _demo_cfg.hidden_size, (

    f"主干输出宽度 {_h.shape[-1]} ≠ hidden_size {_demo_cfg.hidden_size}；"

    "多半是 config 里有派生宽度字段（attention_head_size / head_dim / embedding_size）没同步")

print("→ [batch, 序列长度, 隐层维度]。真正的「句子向量」还需要池化，这就是下面要做的。")


In [ ]:

# ============ 统一的读出模型：掩码均值池化 + linear / ordinal 两种读出头 ============

BIN_CENTERS = torch.arange(SOFT_BINS, dtype=torch.float32)





def ev_readout(ord_logits):
    """序数头 → 期望值读数 ŷ_j = Σ_k k·p_jk（6 桶凸组合 ⇒ 值域 [0,5] 由结构保证）。"""

    p = torch.softmax(ord_logits.to(torch.float32), dim=-1)

    centers = BIN_CENTERS.to(ord_logits.device).view(1, 1, -1)

    return (p * centers).sum(dim=-1)





class CompositionalityRegressor(nn.Module):
    """主干 + 掩码均值池化 + 读出头。

    head_kind="linear"：Linear(hidden→n_outputs) + 5×sigmoid（冻结配方，值域 [0,5]）。
    head_kind="ordinal"：Linear(hidden→SOFT_BINS·n_outputs) → [B, n_out, 6] 有序桶
      logits；ModelOutput 同时带 logits（期望值读数）与 ord_logits（软 CE 用）。"""



    supports_gradient_checkpointing = True



    def __init__(self, backbone, hidden_size, n_outputs=1, dropout=0.1, head_kind="linear"):
        super().__init__()

        self.backbone = backbone

        self.dropout = nn.Dropout(dropout)

        self.n_outputs = int(n_outputs)

        self.head_kind = head_kind

        if head_kind == "ordinal":
            self.head = nn.Linear(hidden_size, SOFT_BINS * self.n_outputs)

        else:
            self.head = nn.Linear(hidden_size, self.n_outputs)



    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
        """转发给主干（本模型是自定义 nn.Module，Trainer 会直接调这个成员）。"""

        if not hasattr(self.backbone, "gradient_checkpointing_enable"):
            raise RuntimeError("当前主干不支持 gradient checkpointing，无法开启")

        self.backbone.gradient_checkpointing_enable(

            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs)



    @property

    def is_gradient_checkpointing(self):
        return bool(getattr(self.backbone, "is_gradient_checkpointing", False))



    def forward(self, input_ids=None, attention_mask=None, labels=None, group_index=None, **kwargs):
        """掩码均值池化；labels / group_index / 其余数据列只接收不使用（损失在 Trainer 里算）。"""

        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        m = attention_mask.unsqueeze(-1).to(out.last_hidden_state.dtype)

        pooled = (out.last_hidden_state * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)

        z = self.head(self.dropout(pooled))

        if self.head_kind == "ordinal":
            ol = z.view(-1, self.n_outputs, SOFT_BINS)

            return ModelOutput(logits=ev_readout(ol), ord_logits=ol)

        return ModelOutput(logits=5.0 * torch.sigmoid(z))





def build_tiny_backbone(model_key, source, tokenizer):
    """冒烟模式：真实 config 缩成 2 层随机主干（链路全真、权重随机、不下权重）。"""

    from transformers import XLMRobertaModel

    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)

    cfg.use_cache = False

    return XLMRobertaModel(cfg)





_AUTO_PRETRAINED_PARAMS = set(inspect.signature(AutoModel.from_pretrained).parameters)

_DTYPE_KW = "dtype" if "dtype" in _AUTO_PRETRAINED_PARAMS else "torch_dtype"

print("from_pretrained 的精度参数名:", _DTYPE_KW)





def load_backbone():
    """加载主干：官方权重（冒烟 = 2 层随机）。显式 fp32 + 注意力实现自动回退。"""

    if SMOKE:
        return build_tiny_backbone(MODEL_KEY, MODEL_SOURCE, tokenizer)

    mcfg = MODEL_REGISTRY[MODEL_KEY]

    backbone, last_err = None, None

    for impl in mcfg["attn_impls"]:
        try:
            backbone = AutoModel.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},

                                                 attn_implementation=impl)

            print(f"主干加载成功: {mcfg['model_id']} | attn_implementation={impl}")

            break

        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise

            last_err = e

            print(f"attn_implementation={impl} 不支持该架构，自动回退")

    if backbone is None:
        raise last_err

    backbone.config.use_cache = False

    return backbone


In [ ]:

# ============ 训练档位：只训练顶部若干层（显存不够时的最后一档）============

FIT_SCOPE_FRAC = 1.0





def _find_layer_list(backbone):
    """找到主干里的 transformer 层列表（DeBERTa 与 Qwen 的存放路径不同）。"""

    for path in ("encoder.layer", "layers", "layer", "decoder.layers", "encoder.decoder.layer"):
        obj = backbone

        for part in path.split("."):
            if not hasattr(obj, part):
                obj = None

                break

            obj = getattr(obj, part)

        if obj is not None and isinstance(obj, (nn.ModuleList, list)) and len(obj) > 0:
            return obj

    return None





def apply_fit_scope(model, scope_frac=None):
    """按档位冻结底部的 transformer 层，只训练顶部若干层。
    【为什么需要这一档】全参微调的固定显存开销 =
        权重 4 B/param（**全部**参数都要占显存） + 梯度与优化器状态 12 B/param（只算可训部分）。
        只训顶部一半的层 → 16 B/param 降到 4 + 12×0.5 = 10 B/param。
    【为什么必须做成单一入口】冻结与否直接决定「哪些参数有梯度」，
        如果有的运行冻了、有的没冻，两次运行之间就不可比了。
        所以只能由 build_model() 统一调用，别的地方不许私自改 requires_grad。"""

    frac = FIT_SCOPE_FRAC if scope_frac is None else scope_frac

    if frac >= 1.0:
        return model

    layers = _find_layer_list(model.backbone)

    if layers is None:
        print(f"⚠️ 找不到主干的层列表，本档位（只训顶部 {frac:.0%} 层）无法生效 → 退回全参微调")

        return model

    n_total = len(layers)

    n_keep = max(1, int(round(n_total * frac)))

    for p in model.backbone.parameters():
        p.requires_grad_(False)

    for layer in layers[n_total - n_keep:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)

    n_all = sum(p.numel() for p in model.parameters())

    print(f"档位生效：只训练顶部 {n_keep}/{n_total} 层 | 可训练参数 {n_tr / 1e6:.1f}M / "

          f"{n_all / 1e6:.1f}M ({n_tr / n_all:.1%})")

    return model







In [ ]:

# ============ 装配并体检 ============

def build_model():
    """每次运行都重新构造模型；读出头由全局 HEAD_KIND 决定（臂在 runloop 里设定）。"""

    bb = load_backbone()

    m = CompositionalityRegressor(

        backbone=bb, hidden_size=bb.config.hidden_size,

        n_outputs=N_OUTPUTS, dropout=0.1, head_kind=HEAD_KIND,

    ).to(DEVICE)

    apply_fit_scope(m)

    return m





model = build_model()

n_all = sum(p.numel() for p in model.parameters())

n_head = sum(p.numel() for p in model.head.parameters())

print(f"主干       : {MODEL_REGISTRY[MODEL_KEY]['model_id']}（SMOKE={SMOKE}）")

print("池化方式   : mean（全句掩码均值；上游判读冻结）")

print(f"读出头     : {HEAD_KIND} | Linear({model.backbone.config.hidden_size} → "

      f"{model.head.out_features})，{n_head / 1e3:.1f}K 参数")

print(f"总参数     : {n_all / 1e6:.1f}M | 设备: {DEVICE}")



model.eval()

with torch.no_grad():
    _b = collator([_demo_run["train_ds"][i] for i in range(2)])

    _out = model(input_ids=_b["input_ids"].to(DEVICE),

                 attention_mask=_b["attention_mask"].to(DEVICE))

print("前向 logits shape:", tuple(_out.logits.shape),

      "| 是否含 NaN:", bool(torch.isnan(_out.logits).any().item()))

assert tuple(_out.logits.shape) == (2, N_OUTPUTS)

if HEAD_KIND == "ordinal":
    assert float(_out.logits.min()) >= 0.0 and float(_out.logits.max()) <= 5.0, "期望值读数超出 [0,5]（桶凸组合不应越界）"

print("→ 前向自检通过")


## Step5 创建评估函数

**这一节要解决什么**：把官方的三项指标**逐字实现**出来，并让本地实现和
未来用 Trial 的 `scores.txt` 对得上。

| 指标 | 官方口径 | 一句话含义 |
|---|---|---|
| 指标 1 | 组合性相关 ρ | 全部例句上 Spearman(预测, 真值)——整体排序对不对 |
| 指标 2 | 语境方差相关 ρ | 每个目标跨例句算 std，再在**目标之间** Spearman(预测 std, 人工 std)——模型有没有真的「因句而异」 |
| 指标 3 | RMSE | 全部例句上的均方根误差——数值校准准不准 |

**指标 2 为什么是硬门槛**：如果模型对同一目标的所有例句给出同一个分数，
那它的组内 std 恒为 0，相关系数的输入数组是常数 → **相关系数未定义（nan）**。
所以指标 2 不是「加分项」，而是**及格线**：一个常数输出系统直接出局。

**指标 1 与指标 2 存在张力**：指标 1 奖励跨目标的数值校准，
指标 2 奖励目标内部的离散程度。过度压缩输出方差 → 指标 2 掉分；
过度放大方差 / 加噪声 → 指标 1 的排序被破坏、RMSE 变差。
本 notebook 不做后处理调参（那属于刷分），而是让模型真正对语境敏感。

**这一节末尾有一组断言自检**：用「完美预测 / 常数预测 / 反向预测」三种人造系统
把指标函数验一遍。指标实现一旦写错，训练完才发现就太晚了。

In [ ]:

# ============ 演示：为什么 type-level（常数输出）系统会直接出局 ============



_demo_v = _demo_run["val_df"]

_demo_key = SUBSET_SPECS["en-nn"]["key"]

_type_level_pred = _demo_v.groupby(_demo_key)["ModAvg"].transform("mean")

_gold = _demo_v["ModAvg"].to_numpy()

_g = _demo_v[_demo_key].to_numpy()



_df = pd.DataFrame({"g": _g, "p": _type_level_pred.to_numpy(), "y": _gold})

_ps = _df.groupby("g")["p"].std(ddof=1)

_gs = _df.groupby("g")["y"].std(ddof=1)

_ok = _ps.notna() & _gs.notna()

print(f"type-level 基线：指标1 Spearman = {stats.spearmanr(_type_level_pred, _gold)[0]:.3f}"

      f" | 指标3 RMSE = {np.sqrt(((_type_level_pred.to_numpy() - _gold) ** 2).mean()):.3f}")

print(f"组内预测 std 的唯一取值个数 = {_ps[_ok].nunique()}（0 表示每个目标内完全相同）")

print("→ 指标 2 的输入是常数数组，相关系数**未定义（nan）**：该系统在指标 2 上直接出局。")

print("→ 这就是为什么本任务必须「逐例句预测」，不能先聚合成目标级分数再回归。")


In [ ]:

# ============ 封装：三项指标 ============

def strip_eval_prefix(metrics):
    """Trainer.evaluate() 会给所有返回的指标加上 'eval_' 前缀，
    这里统一剥掉，好让它们和 compute_metrics 里的原始键名对齐。"""

    return {k[len("eval_"):] if k.startswith("eval_") else k: v for k, v in metrics.items()}





def _three_metrics(p, g, grp):
    """在一个「子集 × 标签列」的切片上算官方三项指标。
    指标1 组合性相关  ：全部例句上 Spearman(预测, 真值)
    指标2 语境方差相关：每个目标跨例句算 std，再在目标之间 Spearman(预测 std, 人工 std)
    指标3 RMSE        ：全部例句上的均方根误差
    【指标 2 的三个边界条件，缺一不可】
      · 组内至少 2 条例句，std 才有定义；
      · 人工 std 与预测 std 都必须非空；
      · 预测 std 必须**不全是同一个值**，否则相关系数未定义（返回 nan 而不是 0）。
    【为什么用 Spearman 而不是 Pearson】官方协议用秩相关：
       组合性评分是有界的离散均值，秩相关对个别离群点更稳健；
       而指标 2 比较的是「离散程度的大小」，本身就不是正态分布，秩相关更合适。"""





    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConstantInputWarning)

        m1 = float(stats.spearmanr(p, g)[0])

        frame = pd.DataFrame({"g": grp, "p": p, "y": g})

        ps = frame.groupby("g")["p"].std(ddof=1)

        gs = frame.groupby("g")["y"].std(ddof=1)

        ok = ps.notna() & gs.notna()

        if ok.sum() > 2 and ps[ok].nunique() > 1:
            m2 = float(stats.spearmanr(ps[ok], gs[ok])[0])

        else:
            m2 = float("nan")

    rmse = float(np.sqrt(((p - g) ** 2).mean()))

    return m1, m2, rmse, int((ps[ok] == 0).sum())





def compute_compartment_metrics(pred, gold, group_index, subset_ids, eval_cols):
    """按「子集 × 标签列」分别算三项指标，并给出各设置值的平均（「合计」）。
    【为什么必须按子集分开算】四个子集的语言、任务、标签难度完全不同
       （德语复合词修饰语 79.4% 落在 [4,5]，英语小品词动词均值只有 2.46），
       混在一起算一个总 ρ 会被大子集主导，既看不出哪个设置出了问题，
       也无法与官方「6 个评测设置」逐项对照。
    【「合计」怎么算】取各评测设置指标的**平均**。
       【为什么不能把各设置的行拼起来算】官方是 6 个评测设置分别打分，没有「总分」。
       如果把不同标签列的行拼成一组算，同一目标的「修饰语列」和「中心语列」会被
       并进同一个组，组内 std 被人为撑大，指标 2 就不再是官方口径
       （实跑时正是这样把 type-level oracle 的 nan 算成了 0.96）。
       所以合计一律取各设置值的平均，nan 的设置自动跳过。"""

    pred = np.asarray(pred, dtype=float).reshape(len(pred), -1)

    gold = np.asarray(gold, dtype=float).reshape(len(gold), -1)

    group = np.asarray(group_index)

    subset_ids = np.asarray(subset_ids)

    out = {}

    _m1s, _m2s, _rmses, _ndegs = [], [], [], []

    for name, cname, j in eval_cols:
        m = subset_ids == name

        if m.sum() < 3:
            continue

        m1, m2, rmse, ndeg = _three_metrics(pred[m, j], gold[m, j], group[m])

        out[f"{name}|{cname}_指标1_ρ"] = m1

        out[f"{name}|{cname}_指标2_ρ"] = m2

        out[f"{name}|{cname}_指标3_RMSE"] = rmse

        out[f"{name}|{cname}_组内std退化目标数"] = ndeg

        out[f"{name}|{cname}_样例数"] = int(m.sum())

        _m1s.append(m1); _m2s.append(m2); _rmses.append(rmse); _ndegs.append(ndeg)



    def _nanmean(vals):
        """对 nan 安全的平均：nan 的设置不参与平均，全 nan 时返回 nan。"""

        v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)

        return float(v.mean()) if len(v) else float("nan")



    out["合计|_指标1_ρ"] = _nanmean(_m1s)

    out["合计|_指标2_ρ"] = _nanmean(_m2s)

    out["合计|_指标3_RMSE"] = _nanmean(_rmses)

    out["合计|_组内std退化目标数"] = int(np.sum(_ndegs)) if _ndegs else 0

    out["合计|_评测设置数"] = len(_m1s)

    out["合计|_样例数"] = int(len(pred))

    out["合计|_目标数"] = int(len(np.unique(group)))







    _m1 = out["合计|_指标1_ρ"]

    _m2 = out["合计|_指标2_ρ"]

    _m3 = out["合计|_指标3_RMSE"]

    _c1 = 0.0 if not np.isfinite(_m1) else float(_m1)

    _c2 = float(_m2) if np.isfinite(_m2) else -1.0

    _c3 = float(_m3) if np.isfinite(_m3) else float("nan")

    out[BEST_METRIC_KEY] = (_c1 / Z_M1) + (_c2 / Z_M2) - (_c3 / Z_M3 if np.isfinite(_c3) else 0.0)

    return out





def make_compute_metrics(eval_group_index, eval_subset_index, eval_cols, subset_names):
    """生成 Trainer 的 compute_metrics 钩子。
    【为什么闭包里要带 group_index 和 subset_index】指标 2 需要知道
      「哪些例句属于同一个目标」「每条例句来自哪个子集」。
      Trainer.predict 按顺序吐出预测，验证集不打乱，顺序与闭包里的数组一致。"""

    groups = np.asarray(eval_group_index)













    _max_id = int(np.max(eval_subset_index)) if len(eval_subset_index) else -1

    assert _max_id < len(subset_names), (

        f"subset_index 最大值 {_max_id} 超出 subset_names 长度 {len(subset_names)}："

        "subset_index 是全局编号，subset_names 必须传全局的 SUBSET_ORDER")

    subsets = np.asarray([subset_names[i] for i in eval_subset_index])



    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        preds = np.asarray(preds)

        labels = np.asarray(labels)

        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)

        if labels.ndim == 1:
            labels = labels.reshape(-1, 1)

        return compute_compartment_metrics(preds, labels, groups, subsets, eval_cols)



    return compute_metrics


In [ ]:

# ============ 指标函数的断言自检：三组人造系统，错了就当场崩 ============







_gold = ds_labels(_demo_run["val_ds"])

_grp = np.asarray(_demo_run["val_ds"]["group_index"])

_sub = np.asarray(["en-nn"] * len(_grp))

_cols = _demo_run["eval_cols"]





_m = compute_compartment_metrics(_gold, _gold, _grp, _sub, _cols)

for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"

    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"

    _k3 = f"{_c[0]}|{_c[1]}_指标3_RMSE"

    assert abs(_m[_k1] - 1.0) < 1e-9, f"完美预测的指标1 应为 1，实际 {_m[_k1]}"

    assert abs(_m[_k2] - 1.0) < 1e-9, f"完美预测的指标2 应为 1，实际 {_m[_k2]}"

    assert _m[_k3] < 1e-9, f"完美预测的 RMSE 应为 0，实际 {_m[_k3]}"

print("① 完美预测      : 指标1=1.000 指标2=1.000 RMSE=0.000  ✓")





_m = compute_compartment_metrics(np.full_like(_gold, 3.0), _gold, _grp, _sub, _cols)

for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"

    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"

    assert np.isnan(_m[_k1]), f"常数预测的指标1 应为 nan，实际 {_m[_k1]}"

    assert np.isnan(_m[_k2]), f"常数预测的指标2 应为 nan，实际 {_m[_k2]}"

print("② 常数预测      : 指标1=nan 指标2=nan（未定义）              ✓")





_m = compute_compartment_metrics(-_gold, _gold, _grp, _sub, _cols)

for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"

    assert abs(_m[_k1] + 1.0) < 1e-9, f"反向预测的指标1 应为 -1，实际 {_m[_k1]}"

print("③ 反向预测      : 指标1=-1.000                              ✓")





_flat = np.full_like(_gold, 2.5)

_m = compute_compartment_metrics(_gold, _flat, _grp, _sub, _cols)

for _c in _cols:
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"

    assert np.isnan(_m[_k2]), f"人工 std 恒为 0 时指标2 应为 nan，实际 {_m[_k2]}"

print("④ 人工 std 恒为 0: 指标2=nan（被解释变量无方差）           ✓")

print("→ 指标实现通过四组断言，接下来所有结果都由它产出。")


## Step6 配置训练参数

**这一节要解决什么**：把超参组装成一个**函数**，而不是散落的字面量。

**为什么必须是函数**：transformers 的 `Trainer` 会**就地改写** `TrainingArguments` 对象
（`per_device_train_batch_size`、`max_steps`、`label_names`、`_n_gpu` 都会被动过）。
如果两个 Trainer 共用同一个 args 对象，第二个 Trainer 拿到的就是「已经被训练过」的脏 args，
它的 `num_batches` 与实际 dataloader 长度对不上，直接抛
`ValueError: zip() argument 2 is longer than argument 1`。
所以「超参相同」的正确实现是：**每次调用 `build_training_args()` 重新构造一个对象**，
字段默认值相同即等价，而不是复用同一个变量。

**超参取值与理由**：

| 超参 | 值 | 为什么 |
|---|---|---|
| `learning_rate` | 2e-5 | 微调大模型的常规档位。两个模型**用同一个值**，对照实验里 lr 不成为混淆变量 |
| `per_device_train_batch_size` | 8 | 序列长 160、模型 900M～1.5B；显存不够时用 `gradient_accumulation_steps` 把有效 batch 补回去（数学等价、几乎无损，第一选择） |
| `num_train_epochs` | 3 | 本项目每个目标只有 5–10 条例句，全量数据仅 1.5k–3.5k 条；轮数太多会过拟合到「记住句子」 |
| `warmup_steps` | 总步数 × 10% | transformers 5 **删除了 `warmup_ratio`**，只保留绝对步数，所以这里自己算 |
| `lr_scheduler_type` | linear | 线性衰减是回归任务的稳定默认；余弦/常数留到消融时再比 |
| `weight_decay` | 0.01 | 常规正则量级 |
| `eval_strategy` / `save_strategy` | epoch | transformers 5 把 `evaluation_strategy` 改名为 `eval_strategy` |
| `load_best_model_at_end` | True | 必须与 `save_strategy="epoch"` 同时满足，否则初始化阶段直接抛 ValueError |

In [ ]:

# ============ 封装：训练参数 ============











_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)

_EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"

_TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)

_TOKENIZER_KW = "processing_class" if "processing_class" in _TRAINER_PARAMS else "tokenizer"

_HAS_SAVE_ONLY_MODEL = "save_only_model" in _TA_PARAMS

print("transformers 兼容层:", {"eval_key": _EVAL_STRATEGY_KEY, "tokenizer_kw": _TOKENIZER_KW,

                                "save_only_model": _HAS_SAVE_ONLY_MODEL})





# ============ 训练「档位」体系：按显存自动降级；默认写死 t3 ============

def _bnb_available():
    """bitsandbytes 是否可用（8bit 优化器的前提）。"""

    try:
        import bitsandbytes

        return True

    except Exception:
        return False





_HAS_BNB = _bnb_available()





def _use_8bit_allowed():
    """8bit 优化器档位是否可用。
    【★ 判定只依赖「环境」，不依赖「模型」】同一个环境里解析出的 optim 必然相同
       ⇒ 不会出现「一条基线 fp32、另一条 8bit」的无声分叉。
       "auto" 的语义就是「环境里有 bitsandbytes 就允许 8bit 档」——
       同一环境里解析结果必然相同 ⇒ 口径可复现。"""

    mode = USE_8BIT_OPT

    if isinstance(mode, str):
        low = mode.strip().lower()

        if low in ("true", "1", "yes", "8bit"):
            return True

        if low in ("false", "0", "no", "fp32"):
            return False

        return _HAS_BNB

    return bool(mode)



































FIT_ACT_MIN_GB = 0.3



FIT_TIERS = [

    {"name": "t0", "label": "全参微调 · fp32 AdamW · AMP", "optim": "adamw_torch",

     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},

    {"name": "t1", "label": "全参微调 · 8bit AdamW · AMP", "optim": "adamw_bnb_8bit",

     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},

    {"name": "t2", "label": "全参微调 · Adafactor · AMP", "optim": "adafactor",

     "amp": True, "scope": 1.00, "ckpt": True, "act_gb": 1.5},

    {"name": "t3", "label": "全参微调 · Adafactor · 无 AMP", "optim": "adafactor",

     "amp": False, "scope": 1.00, "ckpt": True, "act_gb": 1.5},

    {"name": "t4", "label": "顶部 50% 层 · Adafactor · AMP", "optim": "adafactor",

     "amp": True, "scope": 0.50, "ckpt": True, "act_gb": 1.5},

    {"name": "t5", "label": "顶部 50% 层 · Adafactor · 无 AMP", "optim": "adafactor",

     "amp": False, "scope": 0.50, "ckpt": True, "act_gb": 1.5},

    {"name": "t6", "label": "顶部 25% 层 · Adafactor · 无 AMP", "optim": "adafactor",

     "amp": False, "scope": 0.25, "ckpt": True, "act_gb": 1.5},

]



_OPTIM_STATE_BYTES = {"adamw_torch": 8.0, "adamw_bnb_8bit": 2.0, "adafactor": 0.2}





def tier_fixed_bytes(tier):
    """每参数的**固定**显存开销（bytes），与 batch 大小无关。
    【为什么固定开销是关键】权重 4 B/param + 梯度 4 B/param 是硬的；
    剩下的全是优化器状态与 AMP 的 fp16 副本。
    所以「降 batch / 加梯度累积」能省的只是激活，**救不了这一块**。"""

    w = 4.0

    amp = 2.0 if tier["amp"] else 0.0

    st = _OPTIM_STATE_BYTES[tier["optim"]]

    return w + amp + tier["scope"] * (4.0 + st)





def tier_estimate_gb(n_params, tier, act_gb=None):
    """返回 (固定开销 GiB, 加上激活后的 GiB)。"""

    fixed = n_params * tier_fixed_bytes(tier) / (1024 ** 3)

    act = tier["act_gb"] if act_gb is None else act_gb

    return fixed, fixed + act





def _tier_available(tier):
    """该档位在当前环境是否可用（只与环境有关，与模型无关）。"""

    if tier["optim"] == "adamw_bnb_8bit":
        return _use_8bit_allowed() and _HAS_BNB

    return True





def _vram_total_gb():
    """当前**正在用**的那张卡的容量。
    【为什么不用 device 0】本 notebook 可以通过 GPU_INDEX 指定用哪张卡
    （两条基线同时跑时各占一张），显存预算必须按**实际占用的那张卡**算。"""

    if not torch.cuda.is_available():
        return None

    return torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / (1024 ** 3)





def resolve_fit_tier(vram_gb, ref_params):
    """选出「装得下」的最靠前档位，并打印全部档位的估算。

    auto 模式用参考参数量（两条基线中较大者）统一选档，保证两条落在同一档；
    默认 FIT_MODE="t3" 写死，这条自动路径只是兜底。
    估算只是候选：激活、碎片、CUDA context 都没算进去，
    真正的判定由 Step8 的「真跑 2 步」完成，OOM 自动降下一档。"""

    if vram_gb is None:
        return FIT_TIERS[0], []

    budget = vram_gb * FIT_SAFETY

    rows, chosen = [], None

    for t in FIT_TIERS:
        if not _tier_available(t):
            rows.append((t["name"], t["label"], None, None, None,

                         "不可用（需要 bitsandbytes）" if t["optim"] == "adamw_bnb_8bit" else "不可用"))

            continue



        fixed, need_min = tier_estimate_gb(ref_params, t, act_gb=FIT_ACT_MIN_GB)

        _fixed8, need8 = tier_estimate_gb(ref_params, t)

        ok = need_min <= budget

        rows.append((t["name"], t["label"], fixed, need_min, need8,

                     "✓ 装得下" if ok else f"✗ 超出预算 {budget:.1f} GiB"))

        if ok and chosen is None:
            chosen = t



    if FIT_MODE != "auto":
        forced = next((t for t in FIT_TIERS if t["name"] == str(FIT_MODE).lower()), None)

        if forced is not None and _tier_available(forced):
            chosen = forced

    if chosen is None:


        avail = [t for t in FIT_TIERS if _tier_available(t)]

        chosen = avail[-1] if avail else FIT_TIERS[0]

    return chosen, rows





FIT_TIER_NAME = FIT_TIERS[0]["name"]

OPTIM = FIT_TIERS[0]["optim"]

USE_GRAD_CKPT = False

FIT_SCOPE_FRAC = 1.0





def apply_tier(tier):
    """把档位写进全局变量：5 次运行必须用同一套设置。
    【★ 混合精度也是档位的一部分】开 AMP 会多占 2 B/param（fp16 权重副本），
       但训练快得多；关 AMP 省下这 2 B/param，代价是纯 fp32 训练慢（T4 没有 TF32）。
       这一项由档位统一决定 —— 5 次运行之间不能一个开 AMP 一个不开（否则曲线不可比）。"""

    global OPTIM, USE_GRAD_CKPT, FIT_SCOPE_FRAC, FIT_TIER_NAME, USE_BF16, USE_FP16

    OPTIM = tier["optim"]

    USE_GRAD_CKPT = tier["ckpt"]

    FIT_SCOPE_FRAC = tier["scope"]

    FIT_TIER_NAME = tier["name"]

    if tier["amp"]:
        USE_BF16, USE_FP16 = HW_BF16, HW_FP16

    else:
        USE_BF16, USE_FP16 = False, False





def check_vram_budget(n_params, model_name, optim=None):
    """对比各档位的估算需求与本机实际显存；不够就**当场**给出可执行的建议。
    返回 True=有档位装得下 / False=全部装不下 / None=非 CUDA 环境无法判断。"""

    if not torch.cuda.is_available():
        print("非 CUDA 环境：跳过显存预算检查（按标准全参微调跑）")

        return None

    n_gpu = torch.cuda.device_count()

    for _i in range(n_gpu):
        _p = torch.cuda.get_device_properties(_i)

        print(f"  GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "

              f"compute capability {_p.major}.{_p.minor}")

    total_gb = _vram_total_gb()

    print(f"显存预算：{model_name} | {n_params / 1e6:.0f}M 参数（实际） | "

          f"选档参考 {FIT_REF_PARAMS / 1e6:.0f}M（仅 auto 模式用；当前 FIT_MODE={FIT_MODE}） | "

          f"单卡容量 {total_gb:.1f} GiB | 安全系数 {FIT_SAFETY:.0%}")

    tier, rows = resolve_fit_tier(total_gb, FIT_REF_PARAMS)

    print("  档位表（固定开销与 batch 无关，降 batch 救不了）：")

    print(f"    {'档':4s}{'含义':34s}{'固定':>9s}{'bs=1 需':>10s}{'bs=8 需':>10s}  判定")

    for name, label, fixed, need_min, need8, verdict in rows:
        if fixed is None:
            print(f"    {name:4s}{label:34s}{'-':>9s}{'-':>10s}{'-':>10s}  {verdict}")

        else:
            print(f"    {name:4s}{label:34s}{fixed:8.1f}G{need_min:9.1f}G{need8:9.1f}G  {verdict}")

    fixed_act, need_act = tier_estimate_gb(n_params, tier)

    print(f"  → 采用档位 {tier['name']}（{tier['label']}）；"

          f"本模型固定开销 {fixed_act:.1f} GiB"

          f"（+ 激活：bs=1 约 {tier['act_gb'] * 0.2:.1f} / bs=8 约 {tier['act_gb']:.1f} GiB，"

          f"实际能跑多大由探测决定）")

    if n_gpu > 1:
        print(f"  ⚠️ 本机有 {n_gpu} 张卡，但**单进程默认只用 GPU0**。多卡要用对方式才有效：")

        print("     · DDP（数据并行）：每张卡各存**一整份**模型 → **不降低单卡显存需求**，只是把数据分半")

        print("     · ZeRO-3 / FSDP（参数分片）：把参数+梯度+优化器状态切到多卡 → "

              "这才能真正用满多卡显存（需装 deepspeed）")

        print("     · 最省事：两条基线**各占一张卡并行跑**（两个独立实验，互不干扰）")

    _fixed_m, need_min = tier_estimate_gb(n_params, tier, act_gb=FIT_ACT_MIN_GB)

    if need_min <= total_gb:
        return True

    print("=" * 78)

    print(f"⚠️ 警告：档位 {tier['name']} 即使按 micro-batch=1 也要约 {need_min:.1f} GiB，"

          f"本卡只有 {total_gb:.1f} GiB。")

    print("   Step8 会用真实模型逐档试探；若仍 OOM 会自动再降一档，"

          "并在全部档位都失败时给出精确诊断（而不是一句 out of memory）。")

    print("=" * 78)

    return False





def ds_int_col(ds, name):
    """把数据集里的整型列取成**一维** int64 数组。
    【为什么需要】`datasets` 取「列表型列」时可能返回嵌套 list，也可能返回展平后的 list
       （取决于底层 arrow 的存储类型）。group_index 若被读成 (N,1)，
       后面 `np.max` / `index_add_` 的形状都会对不上，所以这里统一压平。
    """

    arr = np.asarray(ds[name])

    if arr.ndim > 1:
        arr = arr.reshape(-1)

    return arr.astype(np.int64)





def _assert_args_consistent(args, n_train):
    """把「配置常量」与「TrainingArguments 实际生效的口径」对齐，开训前执行。
    口径清单若只打印配置变量，就可能出现「清单说 8、实际跑 16」的静默漂移
    ⇒ 这里断言 batch / 累积 / warmup 与推导值逐项一致，漂移当场报错。

    transformers 5 的步数语义（与 4.x 不同）：
      `train_batch_size = per_device × n_gpu`（不含梯度累积）；
      `num_update_steps_per_epoch = ceil(len(dataloader) / grad_accum)`；
      `max_steps = ceil(epochs × num_update_steps_per_epoch)`。
    """

    _bs = int(args.per_device_train_batch_size)

    _ga = int(args.gradient_accumulation_steps)

    _ep = int(args.num_train_epochs)

    assert _bs == BATCH_SIZE, (

        f"per_device_train_batch_size={_bs} ≠ 配置 {BATCH_SIZE}：口径漂移，拒绝开训")

    assert _ga == GRAD_ACCUM, (

        f"gradient_accumulation_steps={_ga} ≠ 配置 {GRAD_ACCUM}：口径漂移，拒绝开训")

    _loader = math.ceil(n_train / _bs)

    _max_steps = math.ceil(_loader / _ga) * _ep

    _expected_warmup = max(1, int(math.ceil(n_train / (_bs * _ga)) * _ep * WARMUP_RATIO))

    assert int(args.warmup_steps) == _expected_warmup, (

        f"warmup_steps={args.warmup_steps} ≠ 按同一口径算出的 {_expected_warmup}：口径漂移，拒绝开训")

    _wf = float(args.warmup_steps) / max(1, _max_steps)





    _tol = max(0.02, 2.0 / max(1, _max_steps))

    assert abs(_wf - WARMUP_RATIO) <= _tol, (

        f"warmup 实际占比 {_wf:.3f} 与设定 {WARMUP_RATIO} 相差过大"

        f"（warmup_steps={args.warmup_steps} / max_steps={_max_steps}）：口径漂移，拒绝开训")

    print(f"   口径自检 ✓ 有效 batch = {_bs}×{_ga} = {_bs * _ga} | "

          f"每轮批数 {_loader} | 总更新步 {_max_steps} | "

          f"warmup {args.warmup_steps} 步（占 {_wf:.1%}）")





def build_training_args(output_dir, n_train, epochs=None, **overrides):
    """构造一份 TrainingArguments。每次调用都新建对象：
    Trainer 会就地改写传入的 args，共用对象会让第二个 Trainer 拿到脏状态。
    warmup_steps 自己算（transformers 5 删除了 warmup_ratio）；
    n_train 必须传入：5 次运行数据量不同，步数按各自数据量算。
    epochs 可按运行覆盖（联合运行用 JOINT_EPOCHS），warmup 跟着重算。"""

    _epochs = int(epochs) if epochs else int(EPOCHS)

    steps_per_epoch = math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM))

    total_steps = steps_per_epoch * _epochs

    base = dict(

        output_dir=str(output_dir),

        per_device_train_batch_size=BATCH_SIZE,

        per_device_eval_batch_size=BATCH_SIZE,

        gradient_accumulation_steps=GRAD_ACCUM,

        learning_rate=LR,

        num_train_epochs=_epochs,

        weight_decay=WEIGHT_DECAY,

        warmup_steps=max(1, int(total_steps * WARMUP_RATIO)),

        lr_scheduler_type="linear",





        logging_steps=max(1, steps_per_epoch // 8),

        save_strategy="epoch",

        save_total_limit=1,





        load_best_model_at_end=True,





        metric_for_best_model=BEST_METRIC_KEY,

        greater_is_better=True,

        remove_unused_columns=False,

        dataloader_pin_memory=(DEVICE == "cuda"),

        dataloader_num_workers=NUM_WORKERS,

        dataloader_persistent_workers=DATALOADER_PERSISTENT,

        dataloader_multiprocessing_context=None if NUM_WORKERS == 0 else "fork",

        dataloader_prefetch_factor=None if NUM_WORKERS == 0 else 4,

        seed=SEED,

        data_seed=SEED,

        report_to=[],

        disable_tqdm=True,

        bf16=USE_BF16,

        fp16=USE_FP16,

        optim=OPTIM,

        gradient_checkpointing=USE_GRAD_CKPT,

    )

    base[_EVAL_STRATEGY_KEY] = "epoch"

    if _HAS_SAVE_ONLY_MODEL:
        base["save_only_model"] = True

    base.update(overrides)







    if base.get("gradient_checkpointing") and "gradient_checkpointing_kwargs" in _TA_PARAMS:
        base["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    args = TrainingArguments(**base)







    try:
        args._n_gpu = 1

    except Exception:
        pass

    return args


## Step7 创建训练器

**这一节要解决什么**：写出损失函数，并把它接进 `Trainer`。

### 三种损失模式（V12 三臂）

| 臂 | loss_mode | 读出头 | 损失 |
|---|---|---|---|
| `joint_base` | base | Linear(1024→2) | huber(δ=1) + guard(0.06 保险丝)——**冻结配方** |
| `joint_ord` | ord | Linear(1024→12)→6 bin | **软 CE** + guard 保险丝（固定权重 1.0） |
| `joint_ordmix` | ordmix | 同上 | **软 CE + huber(δ=1)** + guard 保险丝 |

### 软标签与软交叉熵（序数臂）

每行手里的 (均值 m, 标准差 s, n=10) 重建成 6 个桶 {0..5} 上的软分布
`q_k = Φ((e_hi−m)/s) − Φ((e_lo−m)/s)`（两端桶吃掉截断尾量；s=0 退化为单点），
训练目标 `L_CE = −Σ_k q_k·log p_k`，读数 `ŷ = Σ_k k·p_k ∈ [0,5]`。
依据：软标签分布匹配（Müller et al. 2019：熵下界与校准收益；本项目 v4 已实测
CE 最小值 = H(q)）；有序分布回归思想（与 CORAL 的序数目标同族，未借用其机制）；
标注噪声显式化（Kendall & Gal 2017；本项目实测单例句评分噪声 sd ≈ 0.36–0.50）。
**结构偏置披露**：CE 的最优读数是 EV(q) ≠ m（截断分桶两端吃尾量，宽 s 时 U 形）；
现实 s ∈ [0.36, 0.6] 下 |EV(q)−m| 的 RMSE 下限 ≈ 0.04–0.08，与 RMSE 判读阈值
0.0463 同量级 ⇒ RMSE 假设是该障碍下的保守检验（figL 叠加了 EV(q) 标记）。

### base 模式的权重「按占比自动标定」

`SubsetLossBalancer` 把辅助项权重定义为「该项应占总损失的百分之几」，用各项实时
量级（EMA）在线反解。序数臂不走 balancer（CE 是主项；guard 保险丝固定权重 1.0、
κ 与 base 相同——nn 0.15 挡真塌陷 / pv 0.50 顶住离散度，并非只在塌缩时触发；
每 50 步做 loss 有限性检查，与 base 臂同一纪律）。

### 实现的三条硬约束

- 小品词动词第 2 列是占位常数（loss_mask=0）⇒ CE/huber/guard 均按列掩码跳过；
- 组内 std 的矩用 float64 累加，`sqrt` 有 eps 与 cnt≥2 掩码护栏 ⇒ 梯度不产生 NaN；
- 序数臂每次 `compute_loss` 刷新 `_ord_fixed_terms`，口径快照记录固定权重（不是 balancer EMA）。

In [ ]:

# ============ 损失引擎（占比自动标定；项与 SUBSET_LOSS_PROFILE 一一对应）============

N_SUB = len(SUBSET_ORDER)





def profile_vectors(device, dtype=torch.float32):
    """把 SUBSET_LOSS_PROFILE 展开成按子集编号的张量向量。"""

    def vec(key):
        return torch.tensor([float(SUBSET_LOSS_PROFILE[n].get(key, 0.0)) for n in SUBSET_ORDER],

                            device=device, dtype=dtype)

    return dict(order=vec("order"), mse=vec("mse"), guard=vec("guard"),

                xrank=vec("xrank"), kappa=vec("kappa"))





def _agg_by_subset(values, index, n_sub):
    """按子集聚合（index_add_ 输出长度固定 ⇒ 全程零 GPU 同步）。"""

    return torch.zeros(n_sub, device=values.device, dtype=values.dtype).index_add_(0, index, values)





def order_parts(pred, gold, group_index, sidx, tau, valid, order_vec, n_sub):
    """组内成对排序（指标 1 的组内部分）。softplus(-s·Δ/τ) 光滑、Δ=0 梯度最大；
    只用符号 ⇒ 独立噪声不改期望梯度方向；真实值并列的对不提供信息。"""

    B = pred.shape[0]

    iu = torch.triu(torch.ones(B, B, dtype=torch.bool, device=pred.device), diagonal=1)

    mf = iu & (group_index[:, None] == group_index[None, :])

    if valid is not None:
        ok = valid > 0

        mf = mf & ok[:, None] & ok[None, :]

    diff_p = pred[:, None] - pred[None, :]

    diff_y = gold[:, None] - gold[None, :]

    sign = torch.sign(diff_y)

    mf = mf & (sign != 0)

    mff = mf.to(pred.dtype)

    w = order_vec[sidx]

    num = F.softplus(-sign * diff_p / tau) * mff * w[:, None]

    den = mff * w[:, None]

    idx = sidx[:, None].expand(B, B).reshape(-1)

    return _agg_by_subset(num.reshape(-1), idx, n_sub), _agg_by_subset(den.reshape(-1), idx, n_sub)





def mse_parts(pred, gold, valid, sidx, mse_vec, n_sub):
    """指标 3（RMSE）的直接对应项（掩码二次项）。"""

    w = mse_vec[sidx]

    if valid is not None:
        w = w * valid.to(pred.dtype)

    num = (pred - gold) ** 2 * w

    return _agg_by_subset(num, sidx, n_sub), _agg_by_subset(w, sidx, n_sub)





def guard_parts(std_pred, std_gold, cnt, group_subset, kappa_vec, n_sub):
    """反收缩 hinge：组内 std 不得低于人工 std 的 κ 倍（指标 2 的 nan 红线保险丝）。

    κ：nn 0.15（只挡真塌陷）/ pv 0.50（顶住离散度）；cnt<2 的组无 std 必须排除。"""

    kg = kappa_vec[group_subset]

    ok = (cnt >= 2) & (kg > 0)

    okf = ok.to(std_pred.dtype)

    gap = (kg * std_gold - std_pred).clamp(min=0.0)

    num = gap * gap * okf

    return _agg_by_subset(num, group_subset, n_sub), _agg_by_subset(okf, group_subset, n_sub)





def xrank_parts(std_pred, std_gold, cur_ids, group_subset, xrank_vec, margin, tau, hist_mask, n_sub):
    """跨目标「组内 std 的序」（指标 2 的可微代理）。

    当前目标与历史累积统计（detach）比较；只在同子集内、只取人工 std 差 > margin 的对。"""

    N = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)

    D = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)

    G = std_pred.numel()

    ar = torch.arange(G, device=std_pred.device)

    for g in cur_ids:
        sg = std_gold[g]

        sp = std_pred[g]

        sub_g = group_subset[g]

        allowed = (group_subset == sub_g) & hist_mask & (ar != g) & ((std_gold - sg).abs() > margin)

        af = allowed.to(std_pred.dtype)

        sign = torch.sign(sg - std_gold)

        pen = F.softplus(-sign * (sp - std_pred) / tau)

        w = xrank_vec[sub_g].to(std_pred.dtype)

        idx = sub_g.repeat(G)

        N = N.index_add_(0, idx, pen * af * w)

        D = D.index_add_(0, idx, af * w)

    return N, D





def soft_ce_parts(ord_logits, soft, mask):
    """软标签交叉熵（按有效列掩码平均）。ord_logits [B, n_out, SOFT_BINS]；
    soft [B, n_out, SOFT_BINS]；mask [B, n_out] 无效列（pv 第 2 列）整体跳过。"""

    p = torch.softmax(ord_logits.to(torch.float32), dim=-1)

    ce = -(soft * p.clamp(min=1e-9).log()).sum(dim=-1)

    w = mask.to(ce.dtype)

    return (ce * w).sum() / w.sum().clamp(min=1.0)





def _ordinal_loss(trainer, model, inputs, return_outputs=False):
    """序数臂（ord / ordmix）的损失：软 CE（+ 混合臂的 Huber）+ 塌缩保险丝。

    固定权重、不走自适应标定；每次调用刷新 trainer._ord_fixed_terms 供口径快照。
    保险丝只看当前 batch 内 cnt≥2 的组；κ 与 base 相同（nn 0.15 挡真塌陷 /
    pv 0.50 顶住离散度），权重固定 1.0 而非占比标定。
    return_outputs=True 时返回 (loss, outputs)（Trainer 的评估路径要求成对返回）。"""

    labels = inputs["labels"]

    mask = inputs["loss_mask"]

    gidx = inputs["group_index"]

    soft = inputs["soft"]

    out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

    ol = getattr(out, "ord_logits", None)

    assert ol is not None, "ordinal 臂要求 HEAD_KIND=ordinal（模型需返回 ord_logits）"

    ce = soft_ce_parts(ol, soft, mask)

    terms = {"ce": ce.detach()}

    loss = ce

    if LOSS_MODE == "ordmix":
        hub = (F.huber_loss(out.logits, labels, delta=HUBER_DELTA,

                            reduction="none") * mask).sum() / mask.sum().clamp(min=1.0)

        loss = loss + hub

        terms["huber"] = hub.detach()

    n_g = int(gidx.max().item()) + 1 if gidx.numel() else 1

    gs = GroupStats(n_g, out.logits.device)

    gsub = torch.zeros(n_g, dtype=torch.long, device=out.logits.device)

    gsub.scatter_(0, gidx, inputs["subset_index"])

    prof_g = torch.tensor([float(SUBSET_LOSS_PROFILE[n]["kappa"]) for n in SUBSET_ORDER],

                          device=out.logits.device, dtype=out.logits.dtype)

    hinge_terms = []

    for j in range(int(labels.shape[1])):
        cnt, s1, s2, gc, gs1, gs2 = gs.stats(out.logits[:, j], labels[:, j],

                                             gidx, mask[:, j])

        sp = std_from_moments(cnt, s1, s2, out_dtype=out.logits.dtype)

        sg = std_from_moments(gc, gs1, gs2, out_dtype=out.logits.dtype)

        gd_n, gd_d = guard_parts(sp, sg, cnt, gsub, prof_g, N_SUB)

        hinge_terms.append((gd_n / gd_d.clamp(min=1.0)).mean())

    hinge = torch.stack(hinge_terms).mean()

    loss = loss + GUARD_FUSE_W * hinge

    trainer._ord_step = getattr(trainer, "_ord_step", 0) + 1

    if trainer._ord_step % LOSS_FINITE_CHECK_EVERY == 0:
        assert np.isfinite(float(loss.detach())), (f"序数臂第 {trainer._ord_step} 步 loss 非有限：t3 档无 GradScaler 兜底，"

             "坏梯度会被直接写进权重（与 base 臂同一纪律）")

    terms["guard_fuse"] = hinge.detach()

    trainer._ord_fixed_terms = {f"w_{k}": round(float(v), 5) for k, v in terms.items()}

    if return_outputs:
        return loss, ModelOutput(logits=out.logits)

    return loss


In [ ]:

# ============ 演示：软标签重建 / 软 CE / 熵下界 / 塌缩保险丝 ============

_qnarrow = _soft_target(3.5, 0.4)

_qflat = _soft_target(2.5, 3.0)

_qonehot = _soft_target(2.0, 0.0)

print("q(3.5, s=0.4):", [round(v, 3) for v in _qnarrow], "（窄分布，质量集中在 3-4 桶）")

print("q(2.5, s=3.0):", [round(v, 3) for v in _qflat],

      "（宽 s ⇒ U 形：两端桶吃截断尾量，EV(q)≠m 的偏置来源）")

print("q(2.0, s=0.0):", _qonehot, "（标注者全一致 ⇒ 单点分布）")

for _name, _q in [("窄", _qnarrow), ("宽", _qflat), ("单点", _qonehot)]:
    _qt = torch.tensor([_q], dtype=torch.float32).unsqueeze(1)

    _mk = torch.ones(1, 2)

    _ce_self = soft_ce_parts(torch.log(_qt.clamp(min=1e-9)), _qt, _mk[:, :1])

    _ce_unif = soft_ce_parts(torch.full_like(_qt, 1.0 / SOFT_BINS).log(), _qt, _mk[:, :1])

    print(f"  {(_name + '分布').strip()}: CE(q,q)={float(_ce_self):.4f}（=熵 H(q)）"

          f" | CE(均匀,q)={float(_ce_unif):.4f}（≈log 6={1.7918:.4f}）")

_ol = torch.full((1, 1, SOFT_BINS), 1.0)

print("ev_readout(均匀 logits) =", float(ev_readout(_ol)[0, 0]), "（应为 2.5）")

_ol_peak = torch.zeros(1, 1, SOFT_BINS)

_ol_peak[0, 0, 4] = 8.0

print("ev_readout(尖峰@4)      =", round(float(ev_readout(_ol_peak)[0, 0]), 4), "（≈4）")

_sp = torch.tensor([2.05, 1.77])

_sg = torch.tensor([2.12, 2.12])

_gsub = torch.tensor([0, 2])

_c = torch.tensor([2.0, 2.0])

_prof = profile_vectors("cpu")

_n, _d = guard_parts(_sp, _sg, _c, _gsub, _prof["kappa"], N_SUB)

print("guard（组内 std 健康）  :", (_n / _d.clamp(min=1.0)).tolist(), "（⇒ 不触发）")

_n, _d = guard_parts(torch.zeros(2), _sg, _c, _gsub, _prof["kappa"], N_SUB)

print("guard（塌缩 std=0）     :", (_n / _d.clamp(min=1.0)).tolist(), "（⇒ 保险丝触发拉回）")

print("→ 序数臂损失 = 软 CE（+ 混合臂 huber）+ GUARD_FUSE_W·hinge；CE 下界 = H(q)。")


In [ ]:

def stats_dtype_for(device):
    """组统计矩的累加精度：能用 float64 就用 float64（MPS 不支持，退回 float32）。

    组内方差用「平方和 − 和的平方」算，是灾难性抵消：fp32 下误差可达 1.6%、
    组内差异极小时甚至变号 ⇒ sqrt 导数在 0 附近剧烈跳动，加上 CUDA 原子累加
    顺序随机，会产生 inf/NaN 梯度；float64 下误差降到 ~1e-9 量级。
    按设备能力探测：MPS 不支持 float64，直接写死会让本机冒烟崩溃。
    """

    try:
        torch.zeros(1, dtype=torch.float64, device=device)

        return torch.float64

    except Exception:
        return torch.float32





class GroupStats:
    """跨 batch 累积的目标级统计量（把组内 std 的估计噪声降下来）。

    一个 batch 通常只有 1–2 个目标、每组约 8 条，组内 std 的单次估计相对误差
    ≈ 27%（k=8）⇒ 每个目标的「计数/和/平方和」用 EMA 跨 batch 累积，
    历史部分 detach（梯度不穿历史），估计噪声随累积下降。
    累加精度由 stats_dtype_for 决定（矩相减必须 float64，见该函数说明）。"""



    def __init__(self, n_groups, device, dtype=None, decay=0.8):
        self.n = int(n_groups)

        self.decay = float(decay)

        self.device = device

        if dtype is None:
            dtype = stats_dtype_for(device)

        self.dtype = dtype

        self.p_cnt = torch.zeros(self.n, device=device, dtype=dtype)

        self.p_s1 = torch.zeros(self.n, device=device, dtype=dtype)

        self.p_s2 = torch.zeros(self.n, device=device, dtype=dtype)

        self.g_cnt = torch.zeros(self.n, device=device, dtype=dtype)

        self.g_s1 = torch.zeros(self.n, device=device, dtype=dtype)

        self.g_s2 = torch.zeros(self.n, device=device, dtype=dtype)



    def _batch_stats(self, x, gidx, valid):
        """把 (cnt, Σx, Σx²) 按目标组累加。x 必须先升精度再算平方：
        若先在 float32 里算平方再升精度，舍入误差已经固化。"""

        xd = x.to(self.dtype)

        vd = valid.to(self.dtype)

        cnt = torch.zeros(self.n, device=x.device, dtype=self.dtype)

        s1 = torch.zeros(self.n, device=x.device, dtype=self.dtype)

        s2 = torch.zeros(self.n, device=x.device, dtype=self.dtype)

        cnt.index_add_(0, gidx, vd)

        s1.index_add_(0, gidx, vd * xd)

        s2.index_add_(0, gidx, vd * xd * xd)

        return cnt, s1, s2



    def stats(self, pred, gold, gidx, valid):
        """返回 (cnt, s1, s2, gcnt, gs1, gs2) = 历史(detach) + 当前 batch。"""

        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)

        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)

        return (self.p_cnt + pc, self.p_s1 + ps1, self.p_s2 + ps2,

                self.g_cnt + gc, self.g_s1 + gs1, self.g_s2 + gs2)



    def commit(self, pred, gold, gidx, valid):
        """把本 batch 的贡献并入历史（**detach**，不留计算图）。"""

        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)

        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)

        d = self.decay

        self.p_cnt = d * self.p_cnt + pc.detach()

        self.p_s1 = d * self.p_s1 + ps1.detach()

        self.p_s2 = d * self.p_s2 + ps2.detach()

        self.g_cnt = d * self.g_cnt + gc.detach()

        self.g_s1 = d * self.g_s1 + gs1.detach()

        self.g_s2 = d * self.g_s2 + gs2.detach()





def std_from_moments(cnt, s1, s2, out_dtype=None):
    """由（计数, 和, 平方和）还原组内 std（ddof=1，与官方指标 2 口径一致）。

    四道安全网，保证梯度永不 inf/NaN：
    float64 输入（抵消误差可忽略）/ sqrt 加 eps（var=0 时导数有界）/
    cnt≥2 掩码（单例组的 std 数学上未定义，不能当「离散度最小」参与比较）/
    出口 nan_to_num 兜底。
    """

    n = cnt.clamp(min=1.0)

    mean = s1 / n

    var = (s2 / n - mean * mean) * (cnt / (cnt - 1.0).clamp(min=1.0))

    var = var.clamp(min=0.0)

    safe = torch.sqrt(var + 1e-12)

    out = safe * (cnt >= 2).to(var.dtype)

    out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    if out_dtype is not None:
        out = out.to(out_dtype)

    return out





class SubsetLossBalancer:
    """按「子集 × 标签列」分别在线标定辅助项的权重（全程不同步）。

    把权重定义成「该项应占总损失的百分之几」，用各项实时量级（EMA）反解：
        w_t = (frac_t / (1 − frac_t)) · EMA(main) / (EMA(term_t) + eps)
    无论两项量级怎么变，占比都稳定在设定值附近。
    按子集分开标定：nn 与 pv 的量级/目标占比都不同，共用 EMA 会互相污染。
    全程不调用 .item()（每次同步都会打断 GPU 流水）。
    「缺席即冻结」：某项缺席的步里 EMA 不衰减，否则下次出现时权重会爆炸。
    """



    MAX_WEIGHT = 5.0



    def __init__(self, n_sub, mix, beta=BALANCE_EMA_BETA, device="cpu"):
        self.n = int(n_sub)

        self.beta = float(beta)

        self.mix = {k: v.to(device) for k, v in mix.items()}

        self.main_ema = torch.zeros(self.n, device=device)

        self.term_ema = {k: torch.zeros(self.n, device=device) for k in mix}

        self.weights = {}





        self.w_ema = {k: torch.zeros(self.n, device=device) for k in mix}



    def update_main(self, main_vec, active):
        """更新主项（按子集）的 EMA。active 是 0/1 向量：该子集本步是否出现。"""

        new = self.beta * self.main_ema + (1.0 - self.beta) * main_vec * active

        self.main_ema = torch.where(active > 0, new, self.main_ema)

        return self.main_ema



    def weight(self, name, term_vec, active):
        """返回该项的权重（[n_sub] 张量：不同步、不带梯度）；本步缺席的子集权重为 0。"""

        frac = self.mix[name]

        ema = self.term_ema[name]

        new_ema = self.beta * ema + (1.0 - self.beta) * term_vec * active

        ema = torch.where(active > 0, new_ema, ema)

        self.term_ema[name] = ema

        w = (frac / (1.0 - frac).clamp(min=1e-3)) * self.main_ema / (ema + 1e-6)

        w = w.clamp(max=self.MAX_WEIGHT).detach()

        w = torch.where(active > 0, w, torch.zeros_like(w))

        self.weights[name] = w



        _we = self.w_ema[name]

        _we = torch.where(active > 0, self.beta * _we + (1.0 - self.beta) * w, _we)

        self.w_ema[name] = _we

        return w





class CompartmentTrainer(Trainer):
    """只覆写 compute_loss 的 Trainer。

    compute_loss 的签名必须带 num_items_in_batch（transformers 5 新增，
    子类不写会参数不匹配）；return_outputs 返回 {"logits": ...} 而不是
    ModelOutput（后者会被展开成不等长张量，评估抛 inhomogeneous shape）。"""



    def __init__(self, *args, grouped_batching=True, n_groups=None, group_subset=None, **kwargs):
        super().__init__(*args, **kwargs)

        self.grouped_batching = grouped_batching

        self.n_groups = n_groups





        self._gs = [GroupStats(n_groups or 1, torch.device("cpu"))

                    if n_groups else None for _ in range(N_OUTPUTS)]

        self._gs_ready = False



        self._group_subset = group_subset

        self._gsub_t = None

        self._prof = None

        self._bals = None



        self._step = 0

        self._finite_check_every = int(LOSS_FINITE_CHECK_EVERY)



    def _ensure_prof(self, device, dtype):
        """把组统计缓存、损失配置、标定器搬到训练设备上（第一次调用时）。"""

        if self._gs_ready:
            return



        _sdtype = stats_dtype_for(device)

        for st in self._gs:
            if st is not None:
                st.dtype = _sdtype

                st.p_cnt = st.p_cnt.to(device=device, dtype=_sdtype)

                st.p_s1 = st.p_s1.to(device=device, dtype=_sdtype)

                st.p_s2 = st.p_s2.to(device=device, dtype=_sdtype)

                st.g_cnt = st.g_cnt.to(device=device, dtype=_sdtype)

                st.g_s1 = st.g_s1.to(device=device, dtype=_sdtype)

                st.g_s2 = st.g_s2.to(device=device, dtype=_sdtype)

        self._prof = profile_vectors(device, dtype)

        mix = {k: self._prof[k] for k in ("order", "mse", "guard", "xrank")}



        self._bals = [SubsetLossBalancer(N_SUB, mix, BALANCE_EMA_BETA, device)

                      for _ in range(N_OUTPUTS)]



        self._main_vec_col0 = None

        if self._group_subset is not None:
            self._gsub_t = torch.as_tensor(self._group_subset, device=device, dtype=torch.long)

        self._gs_ready = True







    def _get_train_sampler(self, train_dataset=None):
        """让同一目标的例句尽量落在同一个 batch（组内损失项的前提：
        随机打散时组内配对几乎永远为空）。副作用：一个 batch 通常只有 1–2 个目标
        ⇒ 跨目标比较要靠 xrank 的跨 batch 累积统计。"""

        if not self.grouped_batching:
            return super()._get_train_sampler(train_dataset)

        ds = train_dataset if train_dataset is not None else self.train_dataset

        return TargetGroupedSampler(ds["group_index"], seed=self.args.seed)



    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if LOSS_MODE in ("ord", "ordmix"):
            return _ordinal_loss(self, model, inputs, return_outputs=return_outputs)



        labels = inputs["labels"]

        mask = inputs.get("loss_mask")

        gidx = inputs.get("group_index")

        sidx = inputs.get("subset_index")

        outputs = model(input_ids=inputs["input_ids"],

                        attention_mask=inputs["attention_mask"])

        logits = outputs.logits







        per = F.huber_loss(logits, labels, delta=HUBER_DELTA, reduction="none")

        if mask is not None:
            main = (per * mask).sum() / mask.sum().clamp(min=1.0)

        else:
            main = per.mean()

        loss = main

        if gidx is None or sidx is None or not self.n_groups:
            return (loss, {"logits": logits}) if return_outputs else loss

        self._ensure_prof(logits.device, logits.dtype)

        gsub = self._gsub_t



        cur_ids = torch.unique(gidx)

        hist_any = torch.stack([(st.g_cnt > 0).any() for st in self._gs]).any()

        for j in range(logits.shape[1]):




            _valid = (mask[:, j] if mask is not None else

                      torch.ones_like(logits[:, j]))

            _valid = _valid.to(logits.dtype)



            _mnum = _agg_by_subset(per[:, j] * _valid, sidx, N_SUB)

            _mden = _agg_by_subset(_valid, sidx, N_SUB)

            _main_vec = _mnum / _mden.clamp(min=1.0)

            _act = (_mden > 0).to(logits.dtype)

            bal = self._bals[j]

            bal.update_main(_main_vec, _act)

            if j == 0:
                self._main_vec_col0 = (_main_vec, _act)



            _share = _mden / _mden.sum().clamp(min=1.0)



            st = self._gs[j]

            cnt, s1, s2, gcnt, gs1, gs2 = st.stats(logits[:, j], labels[:, j], gidx, _valid)

            std_p = std_from_moments(cnt, s1, s2, out_dtype=logits.dtype)

            std_g = std_from_moments(gcnt, gs1, gs2, out_dtype=logits.dtype)

            for name in ("order", "mse", "guard", "xrank"):
                _frac = self._prof[name]

                if name == "order":
                    _N, _D = order_parts(logits[:, j], labels[:, j], gidx, sidx,

                                         ORDER_TAU, _valid, _frac, N_SUB)

                elif name == "mse":
                    _N, _D = mse_parts(logits[:, j], labels[:, j], _valid, sidx, _frac, N_SUB)

                elif name == "guard":
                    _N, _D = guard_parts(std_p, std_g, cnt, gsub, self._prof["kappa"], N_SUB)

                else:
                    if cur_ids.numel() == 0 or not bool(hist_any):
                        continue





                    _hist_ok = (st.g_cnt > 0) & (gcnt >= 2)

                    _N, _D = xrank_parts(std_p, std_g, cur_ids, gsub, _frac,

                                         XRANK_MARGIN, XRANK_TAU, _hist_ok, N_SUB)

                _tv = _N / _D.clamp(min=1.0)

                _tact = (_D > 0).to(logits.dtype)

                if AUTO_BALANCE:
                    _w = bal.weight(name, _tv, _tact)

                else:
                    _w = torch.ones(N_SUB, device=logits.device)





                _cap = (_frac / (1.0 - _frac).clamp(min=1e-3)) * _main_vec * _share

                _contrib = torch.minimum(_w * _tv * _tact, _cap)

                loss = loss + _contrib.sum()



            st.commit(logits[:, j], labels[:, j], gidx, _valid)







        self._step += 1

        if self._finite_check_every > 0 and self._step % self._finite_check_every == 0:
            if not bool(torch.isfinite(loss.detach())):
                raise RuntimeError(

                    f"训练第 {self._step} 步的 loss 变成 NaN/Inf（当前档位无 AMP ⇒ 没有 GradScaler "

                    "兜底，坏梯度会直接写进权重）。请检查：该步的组内 std 是否退化"

                    "（analysis/diagnose_std_gradient.py 可复现 fp32 抵消误差）、"

                    "以及是否有目标只带 1 条例句（en-nn 实测存在 1 个这样的目标）。")

        return (loss, {"logits": logits}) if return_outputs else loss



    def loss_weight_snapshot(self):
        """训练结束后取一次「自适应标定出的实际权重」（允许同步，只在训练后调用）。
        取权重全程 EMA（最后一步可能恰好是缺席步，打印出 0 不代表该项被关）。
        每一项都记一份：「配置里写了但实际没生效」的项也能看出来。"""

        if getattr(self, "_ord_fixed_terms", None) is not None:
            return dict(self._ord_fixed_terms)

        out = {}

        for j, bal in enumerate(self._bals or []):
            for k, v in bal.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[col{j}·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)

        return out





class TargetGroupedSampler(Sampler):
    """按目标分组的采样器：先打乱目标顺序，再把同一目标的例句连着吐出。
    DataLoader 会按顺序把连续下标切成一个 batch，于是同一目标的例句自然聚在一起。"""

    def __init__(self, group_index, seed=SEED, shuffle=True):
        self.groups = np.asarray(group_index)

        self.seed = seed

        self.shuffle = shuffle

        self.epoch = 0

        self._indices = self._build()



    def _build(self):
        rng = np.random.default_rng(self.seed + self.epoch)

        uniq = np.unique(self.groups)

        if self.shuffle:
            rng.shuffle(uniq)

        order = []

        for g in uniq:
            idx = np.where(self.groups == g)[0]

            if self.shuffle:
                rng.shuffle(idx)

            order.extend(idx.tolist())

        return order



    def __iter__(self):
        return iter(self._indices)



    def __len__(self):
        return len(self._indices)



    def set_epoch(self, epoch):
        """Trainer 每轮会调用它，让每个 epoch 的目标顺序不同。"""

        self.epoch = epoch

        self._indices = self._build()





print("CompartmentTrainer / TargetGroupedSampler 已定义")


## Step8 模型训练（自动跑完全部运行）

**这一节不需要任何手动切换**：它按 Step1 的 `RUN_PLAN` 自动依次执行
「四个子集各单独训练一次」+「四个子集联合训练一次」，共 5 次。

**每一次运行都做四件事**（顺序不能换）：

1. **准备数据**：按本次参与的子集切分、构造 Dataset（`prepare_run`）。
2. **训练前评估随机初始化模型** —— 给 Step9 的「参考对照」区提供一条基线。
   少了它，「训练后 ρ=0.6」无法回答「它到底学到了什么」。
3. **正式训练**。
4. **训练后评估**：与随机初始化**完全相同的口径**（同一份验证集、同一个
   `compute_metrics`），唯一区别是模型权重。口径不对齐的对照没有意义。

**第一轮额外做三项开训前检查**（只做一次，因为它们与数据规模无关）：

- **显存探测**：逐档真跑 2 步，量峰值显存，取能跑通的最大 micro-batch，
  再用梯度累积把有效 batch 补回 8。
- **梯度流验证**：手动 `backward()` 一次，打印「有多少参数真的拿到了梯度」。
  能抓到的典型 bug 是「主干被冻结 / 回归头没接上」——这些**不会报错**，
  只会让模型训半天纹丝不动。
- **冒烟测试**：`max_steps=5` 跑通「前向 → 损失 → 反向 → 参数更新 → 评估」
  整条链路。直接开长训，一旦中途报错，要么白等几十分钟，要么分不清是 bug 还是超时。

**为什么每次运行都要重建模型**：冒烟测试那 5 步、显存探测那 2 步探针都会改动权重。
正式训练必须从**干净的随机初始化**开始，否则基线不干净、不可复现。

In [ ]:

# ============ 8.1 组装 Trainer ============

def should_check_consistency(overrides, check_consistency=None):
    """这个 Trainer 要不要跑「口径一致性断言」？

    口径断言按 epochs 推 warmup_steps，而体检/冒烟 trainer 会故意把
    num_train_epochs 改成 1 ⇒ 断言必然拿错口径报「漂移」。
    规则：只要 overrides 动了 max_steps / num_train_epochs 就自动关闭断言；
    正式训练路径永远保留。判定收敛在这一个函数里，不靠调用点自觉。
    """

    if check_consistency is not None:
        return bool(check_consistency)

    return not (("max_steps" in overrides) or ("num_train_epochs" in overrides))





def make_trainer(m, out_dir, data, epochs=None, check_consistency=None, **overrides):
    """组装一个 Trainer。
    每次都重新构造 args（Trainer 会就地改写 TrainingArguments，共用对象会在
    第二个 train() 时抛 zip 长度错误）；data 里的 train_ds / val_ds / eval_cols
    必须与本次运行严格对应。epochs 可按运行覆盖（joint 用 JOINT_EPOCHS）。
    口径断言的开关由 should_check_consistency 自动判定：体检/冒烟 trainer
    （动了 max_steps / num_train_epochs）自动关闭，正式训练永远保留。"""



    _probed = ("max_steps" in overrides) or ("num_train_epochs" in overrides)

    check_consistency = should_check_consistency(overrides, check_consistency)

    if _probed and check_consistency is not True:
        print(f"   （体检用途的 Trainer：覆盖了 {sorted(set(overrides) & {'max_steps', 'num_train_epochs'})} "

              "⇒ 自动关闭口径断言；正式训练路径仍保留）")

    _args = build_training_args(out_dir, len(data["train_ds"]), epochs=epochs, **overrides)





    _n_gpu = int(getattr(_args, "n_gpu", 1) or 1)

    assert _n_gpu == 1, (

        f"args.n_gpu={_n_gpu} > 1：Trainer 会自动套 nn.DataParallel，"

        "其反向传播会复制一整份梯度导致 OOM。请用 COMPARTMENT_GPU=<卡号> 只暴露一张卡。")







    if check_consistency:
        _assert_args_consistent(_args, n_train=len(data["train_ds"]))







    _n_groups = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1





    _gs_map = np.zeros(_n_groups, dtype=np.int64)

    _gs_map[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")

    assert _gs_map.max() < len(SUBSET_ORDER), (

        f"目标组的子集编号最大 {_gs_map.max()}，超出 SUBSET_ORDER 长度 {len(SUBSET_ORDER)}")



    return CompartmentTrainer(

        model=m,

        args=_args,

        train_dataset=data["train_ds"],

        eval_dataset=data["val_ds"],

        data_collator=collator,



        **{_TOKENIZER_KW: tokenizer},

        compute_metrics=make_compute_metrics(

            data["val_ds"]["group_index"], data["val_ds"]["subset_index"],

            data["eval_cols"], SUBSET_ORDER),





        grouped_batching=GROUPED_BATCHING,

        n_groups=_n_groups,

        group_subset=_gs_map,

    )





# ============ 8.2 显存探测：按「档位」逐级试探，OOM 就自动降一档 ============











class FitOOMError(RuntimeError):
    """显存不够（而不是别的 bug）。
    单独的异常类型：把「显存不够」和「代码有 bug」严格分开，
    真 bug 一律原样抛出，绝不能被当成显存不足静默降档。"""





class FitNumericError(RuntimeError):
    """档位级的数值不稳（该档位不可用），但不是代码 bug。

    特征是「loss 正常、grad_norm 非有限」（如 AMP 的 fp16 溢出）⇒ 换档位能解决，
    归入降档链；真正指向代码逻辑的报错（KeyError / AttributeError / 断言失败）
    一律原样抛出，绝不降档掩盖。"""





def _is_oom(exc):
    return "out of memory" in str(exc).lower()





def release_gpu_memory(tag=""):
    """释放显存，并打印「已分配 / 已保留」快照。
    每个关键节点都要显式释放：旧模型的权重/梯度/优化器状态挂着会逐次累积，
    最后在正式训练里以「只差几十 MiB」的形式炸掉；快照让那一步占着显存可查。"""

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        _a = torch.cuda.memory_allocated() / (1024 ** 3)

        _r = torch.cuda.memory_reserved() / (1024 ** 3)

        _c = _vram_total_gb() or 0.0

        print(f"[显存·{tag or '释放'}] 已分配 {_a:.2f} GiB | 已保留 {_r:.2f} GiB"

              f" | 卡 {_c:.2f} GiB")

    else:
        print(f"[显存·{tag or '释放'}] 非 CUDA 环境，跳过")





def _probe_reserve_gb():
    """探测阶段给后续步骤留出的显存余量（GiB）。
    探针的判据是「峰值 + 余量 ≤ 卡容量」而不是「不 OOM」：preflight 还会
    多持一份计算图、一份稠密梯度、一次 Trainer。
    余量 0.8 GiB 只覆盖碎片与 CUDA context——探针与正式训练同模型结构、同梯度累积。
    关键判断：余量不足 ≠ 这档不行。峰值由固定开销主导、与 batch 几乎无关，
    降 micro-batch 降不下峰值；把「余量不足」当降档理由会让连 bs=1 都被否
    ⇒ 自动降到只训部分层的档位 = 改掉了训练范围 ⇒ 保留档位、打印告警、继续用。"""

    try:
        return float(FIT_PROBE_RESERVE_GB)

    except Exception:
        return 0.8





def probe_fit_tier(model, data, tier, candidates=(8, 4, 2, 1), steps=2):
    """在给定档位内逐个 batch 真跑，返回 (bs, peak_gb, sps)。
    返回 None 只表示「所有 batch 档位都真的 OOM」⇒ 调用方据此降档。
    判据是「跑通且尽量留有余量」：优先选「峰值 + 余量 ≤ 卡容量」的 batch；
    找不到就选能跑通的最大 batch 并打印告警，保留档位（只有全 OOM 才降档）。"""

    if not torch.cuda.is_available():
        print("非 CUDA 环境（CPU/MPS）：跳过显存探测，沿用配置中的 batch size")

        return None

    _card = _vram_total_gb() or 0.0

    _reserve = _probe_reserve_gb()

    _fallback = None

    for bs in candidates:
        _tr = None

        try:
            _tr = make_trainer(model, OUTPUT_ROOT / "_probe", data,

                               per_device_train_batch_size=bs,







                               gradient_accumulation_steps=GRAD_ACCUM,

                               max_steps=steps,

                               save_strategy="no",

                               load_best_model_at_end=False,

                               logging_steps=steps,





                               check_consistency=False,

                               optim=tier["optim"],

                               gradient_checkpointing=tier["ckpt"],

                               dataloader_num_workers=0, dataloader_persistent_workers=False,

                               dataloader_multiprocessing_context=None,

                               dataloader_prefetch_factor=None,

                               **{_EVAL_STRATEGY_KEY: "no"})

            torch.cuda.reset_peak_memory_stats()

            _t0 = time.time()

            _tr.train()

            _dt = time.time() - _t0

            peak = torch.cuda.max_memory_allocated() / (1024 ** 3)







            _gn = [h.get("grad_norm") for h in _tr.state.log_history if h.get("grad_norm") is not None]

            _bad_gn = []

            for _g in _gn:
                try:
                    _gv = float(_g)

                except (TypeError, ValueError):
                    continue

                if not math.isfinite(_gv):
                    _bad_gn.append(_gv)

            if _bad_gn:
                raise FitNumericError(

                    f"探针阶段的梯度已经是 NaN/Inf（grad_norm={_bad_gn[:3]}），"

                    f"但 loss 与数据都正常 ⇒ 这是**档位的数值精度问题**"

                    f"（{'AMP/fp16 溢出' if tier.get('amp') else 'fp32 精度问题'}）而不是代码 bug。"

                    "处理方式：换下一个档位重试。"

                    "若全部档位都这样，再查：① 组内 std 的 fp32 抵消误差"

                    "（analysis/diagnose_std_gradient.py）；② 只有 1 条例句的目标"

                    "（en-nn 实测存在 1 个）。")





            sps = _dt / max(1, steps * bs)



            _need = peak + _reserve

            _fits = (_need <= _card)

            print(f"  {tier['name']} micro-batch={bs}: 通过，峰值显存 {peak:.2f} GiB，"

                  f"每样本 {sps * 1000:.2f} ms"

                  f"（+余量 {_reserve:.1f} GiB = {_need:.2f} GiB / 卡 {_card:.2f} GiB"

                  f" ⇒ {'余量充足 ✓' if _fits else '余量不足 ✗（但仍保留本档位）'}）")

            if _fits:
                return bs, peak, sps







            if _fallback is None:
                _fallback = (bs, peak, sps)

        except FitNumericError:
            raise

        except RuntimeError as e:
            if not _is_oom(e):
                raise





            _lines = [l for l in str(e).strip().splitlines() if l.strip()]

            _head = _lines[0][:150] if _lines else ""

            _detail = next((l for l in _lines if "Tried to allocate" in l or "GiB is free" in l), "")

            print(f"  {tier['name']} micro-batch={bs}: OOM | {_head}")

            if _detail:
                print(f"      {_detail.strip()[:220]}")

        finally:
            if _tr is not None:
                del _tr

            gc.collect()

            torch.cuda.empty_cache()



    if _fallback is not None:
        _b, _p, _s = _fallback

        print(f"  ⚠️ 档位 {tier['name']}：没有哪个 batch 能同时满足「余量 {_reserve:.1f} GiB」，"

              f"但 micro-batch={_b} 跑得通（峰值 {_p:.2f} GiB / 卡 {_card:.2f} GiB）")

        print("     ⇒ **保留本档位**（余量不足不是降档的理由：Qwen 的峰值由固定开销主导，"

              "降 micro-batch 降不下峰值；降档会让两条基线不同档 ⇒ 对照结论作废）")

        print(f"     ⇒ 采用**能跑通的最大 batch**={_b}（峰值由固定开销主导，降 batch 换不来多少余量，"

              "却要赔上好几倍速度；真 OOM 了有「正式训练自动重试」兜底）")

        print(f"     ⇒ 后续的 preflight 已做三项降峰值处理（开 checkpointing / 样本数减半 / "

              f"释放计算图），并使用探测出的 micro-batch={_b}")

        return _fallback

    return None





# ============ 训练时长投影：开跑前就知道会不会撞上 12 小时上限 ============

def _total_train_rows():
    """四个子集的总行数（只数行，很便宜）→ 训练时长投影的输入。"""

    tot = 0

    for _name in SUBSET_ORDER:
        _p = DATA_ROOT / f"{_name}-train.tsv"

        with open(_p, "r", encoding="utf-8") as _f:
            tot += max(0, sum(1 for _ in _f) - 1)

    return tot





def project_train_time(sec_per_sample):
    """按本运行计划投影下游总训练时长（逐运行取其轮数），1.3 为评估/出图余量系数。"""

    n_rows = _total_train_rows()

    per_epoch = 0.8 * n_rows

    _eps = sum(JOINT_EPOCHS if m.startswith("joint") else EPOCHS for m, s in RUN_PLAN)

    total = per_epoch * _eps

    hours = total * sec_per_sample * 1.3 / 3600

    return n_rows, total, hours





def prepare_for_training(build_fn, data):
    """选档 → 探测 → 开训前检查；任何一步 OOM 就自动降一档重来。

    优化器不许自动切换（切了数值就不可比且没人察觉）；档位可以降，
    但 FIT_MODE 显式指定时锁定档位、禁止降档——降档会改变训练范围
    （只训部分层），而机制实验要求全参微调；这一档跑不通就直接报错。
    preflight 也纳入降档链：它会真的建优化器状态，是必然 OOM 的第一现场。"""

    vram = _vram_total_gb()

    tier0, _rows = resolve_fit_tier(vram, FIT_REF_PARAMS)

    start = FIT_TIERS.index(tier0)







    _locked = ""

    if FIT_MODE != "auto" and str(FIT_MODE).lower() == tier0["name"]:
        _locked = tier0["name"]

    if _locked:
        print(f"★ 档位已锁定为 {_locked}（FIT_MODE 显式指定）⇒ **不自动降档**")

        print("   （降档 = 冻住一半主干 = 换了实验；本版机制要求全参微调。"

              "若这一档真跑不通，请先查「有效 batch / micro-batch / 读出结构」是否与设计一致，"

              "再考虑改档位，并把改动写进口径清单）")

    last_err = ""

    for i in range(start, len(FIT_TIERS)):
        t = FIT_TIERS[i]

        if _locked and t["name"] != _locked:
            break

        if not _tier_available(t):
            print(f"跳过档位 {t['name']}：当前环境不可用")

            continue

        apply_tier(t)

        print("=" * 78)

        print(f"试探档位 {t['name']}：{t['label']} | optim={t['optim']} | "

              f"可训层比例={t['scope']:.0%} | 混合精度={'开' if t['amp'] else '关'} | "

              f"gradient_checkpointing={t['ckpt']} | "

              f"固定开销 {tier_estimate_gb(FIT_REF_PARAMS, t)[0]:.1f} GiB（按参考 {FIT_REF_PARAMS / 1e6:.0f}M 算）")

        print("=" * 78)

        model = None

        try:
            model = build_fn()

            if PROBE_MEMORY and torch.cuda.is_available():
                res = probe_fit_tier(model, data, t)

                if res is None:
                    last_err = f"档位 {t['name']}：所有 batch 档位都 OOM 或余量不足"

                    print(f"→ {last_err}，自动降一档重试")

                    continue

                bs, peak, sps = res

            else:
                print("非 CUDA 环境或 PROBE_MEMORY=False：跳过显存探测，沿用配置中的 batch size")

                bs, peak, sps = BATCH_SIZE, float("nan"), float("nan")







            del model

            model = None

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = build_fn()

            try:
                run_preflight_checks(model, data, OUTPUT_ROOT / "_preflight", micro_bs=bs)

            except FitOOMError as e:
                last_err = f"档位 {t['name']}：开训前检查阶段仍 OOM（{str(e)[:160]}）"

                print(f"→ {last_err}，自动降一档重试")

                print("   ⚠️ 若最终落在与另一条基线**不同**的档位 ⇒ 训练范围不同，结论不可混谈！")

                print("      ⇒ 本版机制要求**全参微调**（t3）；先查有效 batch 与 micro-batch 是否符合设计，"

                      "再决定要不要改档。")

                continue

            return t, bs, peak, sps

        except FitNumericError as e:


            last_err = f"档位 {t['name']}：数值不稳（{str(e)[:200]}）"

            print(f"→ {last_err}，自动降一档重试")

            continue

        finally:
            if model is not None:
                del model

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()



    n_params = FIT_REF_PARAMS

    _lines = [f"档位 {t['name']}：{t['label']} → 固定开销 "

              f"{tier_estimate_gb(n_params, t)[0]:.1f} GiB" for t in FIT_TIERS]

    _lock_txt = (f"\n  ⚠️ 档位已锁定为 {FIT_MODE}（FIT_MODE 显式指定）⇒ 不会自动降档。\n"

                 f"    ★ 若要换档，请记住：本版机制要求**全参微调**（t3）；"

                 f"换档后训练范围改变，这一版的数字就不能和别的档位的结果放在一起说。"

                 if _locked else "")

    raise RuntimeError(

        "所有训练档位都跑不通。\n"

        f"  本卡容量          : {vram:.1f} GiB\n"

        f"  参考参数量        : {n_params / 1e6:.0f}M（两条基线中较大者）\n"

        f"  最后一次失败原因  : {last_err}\n" + _lock_txt + "\n"

        "  各档位固定开销    : " + " | ".join(_lines) + "\n"

        "  可选方案：\n"

        "    ① 选更大显存的实例（如 L4 24GB / A100 40GB）—— 档位会自动回到 t0/t1\n"

        "    ② 换更小的同族模型（deberta-v3-base / Qwen2.5-0.5B-Instruct），管线一行不用改\n"

        "    ③ ZeRO-3 / FSDP 分片（deepspeed）\n"

        "    ④ 手动指定档位：把 FIT_MODE 写成 't0'~'t6'"

        "（本版要求 t3 = 全参微调；换档会改变训练范围，结果不可跨档比较）")





print("Trainer 工厂与档位探测就绪")


In [ ]:

# ============ 8.3 训练前的两项检查（只做一次）============

def run_preflight_checks(model, data, out_dir, micro_bs=None):
    """梯度流验证 + 冒烟测试；OOM 翻译成 FitOOMError，其余原样抛出。"""

    try:
        _preflight_body(model, data, out_dir, micro_bs=micro_bs)

    except RuntimeError as e:
        if _is_oom(e):
            raise FitOOMError(str(e)[:300]) from e

        raise





def _preflight_body(model, data, out_dir, micro_bs=None):
    """梯度流验证 + 真实损失反向 + 5 步冒烟（都按探测出的 micro-batch）。"""

    if USE_GRAD_CKPT and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

        print(f"preflight：已按当前档位开启 gradient checkpointing（档位 {FIT_TIER_NAME}）")

    _mb = max(1, int(micro_bs) if micro_bs else int(BATCH_SIZE))





    model.train()

    _n_fwd = max(1, min(2, _mb))

    _gb = collator([data["train_ds"][i] for i in range(_n_fwd)])

    _gb = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in _gb.items()}

    model.zero_grad(set_to_none=True)

    _go = model(input_ids=_gb["input_ids"], attention_mask=_gb["attention_mask"])

    _gl = F.huber_loss(_go.logits, _gb["labels"], delta=HUBER_DELTA)

    _gl.backward()

    _named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]

    _with_grad = [(n, p) for n, p in _named if p.grad is not None and float(p.grad.abs().sum()) > 0]

    _head_g = float(model.head.weight.grad.abs().sum()) if model.head.weight.grad is not None else 0.0

    _bb_p = next(model.backbone.parameters())

    _bb_g = float(_bb_p.grad.abs().sum()) if _bb_p.grad is not None else 0.0

    print(f"loss                   : {_gl.item():.4f}   （NaN? {bool(torch.isnan(_gl))}）")

    print(f"可训练参数             : {len(_named)}")

    print(f"拿到非零梯度的参数     : {len(_with_grad)}  ({len(_with_grad) / max(1, len(_named)):.1%})")

    print(f"回归头权重梯度绝对值和 : {_head_g:.6g}   （必须 > 0，否则读出通路没接上）")

    print(f"主干首参数梯度绝对值和 : {_bb_g:.6g}   （必须 > 0，否则主干被冻结）")

    assert _head_g > 0, "读出通路没有梯度：回归头没接进计算图"

    assert _bb_g > 0, "主干没有梯度：主干被冻结了"

    assert not torch.isnan(_gl), "loss 出现 NaN：多半是 fp16 加载或标签里有脏值"





    try:
        _gidx = _gb.get("group_index")

        _sidx = _gb.get("subset_index")

        _v = _gb["loss_mask"][:, 0] if "loss_mask" in _gb else None

        _p0 = _go.logits[:, 0].detach()

        _y0 = _gb["labels"][:, 0].detach()

        _vf = _v if _v is not None else torch.ones_like(_p0)

        _n_g = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1

        _gsub_np = np.zeros(_n_g, dtype=np.int64)

        _gsub_np[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")

        _gsub = torch.tensor(_gsub_np, device=_p0.device, dtype=torch.long)

        _prof = profile_vectors(_p0.device, _p0.dtype)

        _n, _d = order_parts(_p0, _y0, _gidx, _sidx, ORDER_TAU, _v, _prof["order"], len(SUBSET_ORDER))

        _r = _n / _d.clamp(min=1.0)

        _n, _d = mse_parts(_p0, _y0, _vf, _sidx, _prof["mse"], len(SUBSET_ORDER))

        _ms = _n / _d.clamp(min=1.0)

        _gs0 = GroupStats(_n_g, _p0.device)

        _c, _s1, _s2, _gc, _gs1, _gs2 = _gs0.stats(_p0, _y0, _gidx, _vf)

        _sp = std_from_moments(_c, _s1, _s2, out_dtype=_p0.dtype)

        _sg = std_from_moments(_gc, _gs1, _gs2, out_dtype=_p0.dtype)

        _n, _d = guard_parts(_sp, _sg, _c, _gsub, _prof["kappa"], len(SUBSET_ORDER))

        _gd = _n / _d.clamp(min=1.0)

        _n, _d = xrank_parts(_sp, _sg, torch.unique(_gidx), _gsub, _prof["xrank"],

                             XRANK_MARGIN, XRANK_TAU,

                             torch.ones(_n_g, dtype=torch.bool, device=_p0.device),

                             len(SUBSET_ORDER))

        _xr = _n / _d.clamp(min=1.0)

        print(f"损失项体检（按子集聚合）: L_order={_r.tolist()}  L_mse={_ms.tolist()}")

        print(f"                        L_guard={_gd.tolist()}  L_xrank={_xr.tolist()}")

        assert torch.isfinite(_r).all() and torch.isfinite(_ms).all() and torch.isfinite(_gd).all() and torch.isfinite(_xr).all(), "损失项出现 NaN/Inf：先查 group_index / subset_index 的 dtype 与形状"

    except AssertionError:
        raise

    except Exception as _e:
        raise AssertionError(f"损失项体检失败（{type(_e).__name__}: {_e}）") from _e

    model.zero_grad(set_to_none=True)

    del _go

    _go = None

    _gb = None

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()





    _gc_trainer = make_trainer(model, out_dir / "_gradcheck", data,

                                max_steps=1, num_train_epochs=1,

                                per_device_train_batch_size=_mb,

                                gradient_accumulation_steps=1,

                                save_strategy="no", load_best_model_at_end=False,

                                logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})

    _n_real = max(1, min(int(_mb), len(data["train_ds"])))

    _probe_batch = collator([data["train_ds"][i] for i in range(_n_real)])

    _probe_batch = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)

                    for k, v in _probe_batch.items()}

    model.zero_grad(set_to_none=True)

    _real_loss = _gc_trainer.compute_loss(model, _probe_batch)

    _real_loss.backward()

    _bad = []

    _gmax = 0.0

    for _n, _p in model.named_parameters():
        if _p.grad is None:
            continue

        if torch.isnan(_p.grad).any() or torch.isinf(_p.grad).any():
            _bad.append(_n)

        _gmax = max(_gmax, float(_p.grad.abs().max()))

    print(f"真实损失反向          : loss={float(_real_loss):.4f} | 最大梯度={_gmax:.3e} | "

          f"非有限梯度的参数数={len(_bad)}")

    assert torch.isfinite(_real_loss), f"真实损失不是有限值（{float(_real_loss)}）"

    assert not _bad, (f"真实损失反向产生非有限梯度（前 5: {_bad[:5]}）——坏梯度会被直接写进权重，"

                      "排查：① 各分项数值；② 组内 std 的抵消误差；③ 单例目标（en-nn 有 1 个）。")

    model.zero_grad(set_to_none=True)

    del _gc_trainer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("→ 梯度流验证通过（含真实损失的反向）。")





    _st = make_trainer(model, out_dir / "_smoke", data,

                       max_steps=5, num_train_epochs=1,

                       per_device_train_batch_size=_mb,

                       gradient_accumulation_steps=1,

                       save_strategy="no", load_best_model_at_end=False,

                       logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})

    _so = _st.train()

    del _st

    gc.collect()

    assert np.isfinite(_so.training_loss), "冒烟训练 loss 非有限值"

    print("冒烟训练 loss 轨迹:", _so.training_loss)

    print("→ 冒烟测试通过（每个 Trainer 都是独立构造 args，不复用对象）。")


In [ ]:

# ============ 8.4 训练主循环：按 RUN_PLAN 自动跑完所有运行 ============





ALL_RUNS = []



for _i, (_mode, _subset) in enumerate(RUN_PLAN):


    _subsets = SUBSET_ORDER if _mode.startswith("joint") else [_subset]

    _run_name = f"{_mode}_{_subset}" if _subset else _mode

    _out_dir = OUTPUT_ROOT / MODEL_KEY / _run_name



    _arm = V12_ARMS.get(_run_name, dict(loss_mode="base", head_kind="linear"))

    LOSS_MODE = _arm["loss_mode"]

    HEAD_KIND = _arm["head_kind"]



    def _seed_all(v):
        """每次建模型前重置全局随机源：三臂的读出头初始化逐位同源（Trainer 只对齐数据顺序）。"""

        import random

        random.seed(v)

        np.random.seed(v)

        torch.manual_seed(v)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(v)



    def _build():
        _seed_all(SEED)

        return build_model()



    print(f"★ V12 臂：{_run_name} | loss_mode={LOSS_MODE} | head={HEAD_KIND} "

          f"| pool=mean")

    print("=" * 78)

    print(f"运行 {_i + 1}/{len(RUN_PLAN)}：{_run_name}  |  参与子集 {_subsets}")

    print("=" * 78)





    _data = prepare_run(_subsets, tokenizer)

    print(f"训练 {len(_data['train_ds'])} 条 | 验证 {len(_data['val_ds'])} 条 | "

          f"评测设置 {len(_data['eval_cols'])} 个")





    _run_epochs = JOINT_EPOCHS if _mode.startswith("joint") else EPOCHS

    print(f"本次运行轮数 = {_run_epochs}"

          f"（{'联合对照臂，覆盖全部评测设置' if str(_mode).startswith('joint') else '单子集诊断'}）")





    model = _build()

    n_params = sum(p.numel() for p in model.parameters())

    check_vram_budget(n_params, MODEL_REGISTRY[MODEL_KEY]["model_id"])









    if _i == 0:
        model = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()





        _tier, _bs, _peak, _sps = prepare_for_training(_build, _data)

        apply_tier(_tier)

        BATCH_SIZE = _bs







        GRAD_ACCUM = max(1, int(round(TARGET_EFFECTIVE_BATCH / _bs)))

        print(f"→ 采用档位 {_tier['name']}（{_tier['label']}）| micro-batch={BATCH_SIZE} | "

              f"梯度累积={GRAD_ACCUM} | 有效 batch={BATCH_SIZE * GRAD_ACCUM}"

              f"（目标 {TARGET_EFFECTIVE_BATCH}） | 峰值显存约 {_peak:.2f} GiB")

        print("   （档位写死为 t3 = 全参微调，不会悄悄漂到只训部分层的档位）")





        if _sps == _sps:


            _sps_cal = min(_sps, CLOUD_SPS)

            _n_rows, _tot, _hours_probe = project_train_time(_sps)

            _, _, _hours_cal = project_train_time(_sps_cal)

            print(f"训练时长投影（探针口径，偏保守）：下游共约 {_tot:,} 个样本·轮"

                  f" → 约 {_hours_probe:.1f} 小时")

            print(f"训练时长投影（云端实测吞吐 {CLOUD_SPS * 1000:.0f} ms/样本）"

                  f"：下游约 {_hours_cal:.1f} 小时（预算 {TIME_BUDGET_H} 小时）")

            if _hours_cal > TIME_BUDGET_H:
                print("=" * 78)

                print(f"⚠️ 下游投影 {_hours_cal:.1f} 小时 > 预算 {TIME_BUDGET_H} 小时！")

                print("   V12 的三个臂都是预注册必需：裁任何一臂 = 预注册完整性破坏，")

                print("   Step10 将拒绝判读（宁可失败不可无判读）。可执行的做法：")

                print("   (1) 换更快的卡（不改任何口径项）")

                print("   (2) 压缩 MAX_LENGTH 或 BATCH 分解（须记进口径并三臂同步）")

                print("=" * 78)











    _max_oom_retry = 3

    _attempt = 0

    _rand_metrics = None

    _rand_eval_loss = float("nan")

    while True:
        release_gpu_memory("正式训练前")

        model = _build()

        trainer = make_trainer(model, _out_dir, _data, epochs=_run_epochs)







        if _rand_metrics is None:
            _t0 = time.time()

            _rand_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))

            print(f"随机初始化模型基线（{time.time() - _t0:.1f}s）: "

                  f"指标1 ρ={_rand_metrics.get('合计|_指标1_ρ', float('nan')):.4f} | "

                  f"指标2 ρ={_rand_metrics.get('合计|_指标2_ρ', float('nan')):.4f} | "

                  f"RMSE={_rand_metrics.get('合计|_指标3_RMSE', float('nan')):.4f}")







            _rand_eval_loss = float(_rand_metrics.get("loss", float("nan")))

            print(f"随机初始化时的评估 loss = {_rand_eval_loss:.4f}（图 F 的 epoch 0 起点）")

            release_gpu_memory("随机基线评估后")





        _t0 = time.time()

        try:
            _res = trainer.train()

        except RuntimeError as _e:
            if not _is_oom(_e) or _attempt >= _max_oom_retry:
                raise

            _attempt += 1

            _new_bs = max(1, BATCH_SIZE // 2)

            if _new_bs == BATCH_SIZE:
                raise

            _new_ga = max(1, int(round(TARGET_EFFECTIVE_BATCH / _new_bs)))

            print("=" * 78)

            print(f"⚠️ 正式训练 OOM（第 {_attempt}/{_max_oom_retry} 次重试）: {str(_e)[:220]}")

            print(f"   micro-batch {BATCH_SIZE} → {_new_bs}，梯度累积 {GRAD_ACCUM} → {_new_ga}"

                  f"（**有效 batch 仍为 {TARGET_EFFECTIVE_BATCH}**，口径不变）")

            print("   ⇒ 重建模型与 Trainer 后重试（权重会被重新初始化，保持可比）")

            print("=" * 78)

            BATCH_SIZE = _new_bs

            GRAD_ACCUM = _new_ga

            del trainer, model

            release_gpu_memory("OOM 重试前")

            continue

        _dt = time.time() - _t0

        print(f"训练完成，用时 {_dt / 60:.1f} 分钟，loss={_res.training_loss:.4f}")

        break







    _log_hist = [dict(h) for h in trainer.state.log_history]

    _n_train_logs = sum(1 for h in _log_hist if "loss" in h and "eval_loss" not in h)

    _n_eval_logs = sum(1 for h in _log_hist if "eval_loss" in h)

    print(f"日志快照：训练点 {_n_train_logs} 个 | 评估点 {_n_eval_logs} 个")





    _actual_bs = int(trainer.args.per_device_train_batch_size)

    _actual_ga = int(trainer.args.gradient_accumulation_steps)

    _dl_len = math.ceil(len(_data["train_ds"]) / _actual_bs)

    _actual_max_steps = int(trainer.state.max_steps)

    _expected_max_steps = math.ceil(_dl_len / _actual_ga) * int(trainer.args.num_train_epochs)

    _actual_warmup = int(trainer.args.warmup_steps)

    _actual_eff_batch = _actual_bs * _actual_ga

    _actual_wf = _actual_warmup / max(1, _actual_max_steps)

    assert _actual_max_steps == _expected_max_steps, (

        f"Trainer 的总更新步 {_actual_max_steps} 与按口径算出的 {_expected_max_steps} 不一致")

    assert _actual_eff_batch == BATCH_SIZE * GRAD_ACCUM, (

        f"实际有效 batch = {_actual_eff_batch} ≠ 配置 {BATCH_SIZE * GRAD_ACCUM}")

    _wf_tol = max(0.02, 2.0 / max(1, _actual_max_steps))

    assert abs(_actual_wf - WARMUP_RATIO) <= _wf_tol, (

        f"实际 warmup 占比 {_actual_wf:.3f} ≠ 设定 {WARMUP_RATIO}")

    _loss_terms = trainer.loss_weight_snapshot()

    print(f"口径回读 ✓ 有效 batch={_actual_eff_batch} | 每轮批数={_dl_len} | "

          f"总更新步={_actual_max_steps} | warmup={_actual_warmup} 步（占 {_actual_wf:.1%}）")

    print(f"损失分项权重（自适应标定后）: {_loss_terms}")





    _pred_out = trainer.predict(_data["val_ds"])

    _trained_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))

    _diag = None

    print("--- 训练后指标 ---")

    for k, v in _trained_metrics.items():
        if isinstance(v, float) and ("指标" in k or "目标数" in k or "综合分" in k):
            print(f"   {k:34s} {v:.4f}")





    _val_df = _data["val_df"]

    _gold = ds_labels(_data["val_ds"])

    _type_level = np.zeros((len(_val_df), N_OUTPUTS))

    for _name, _cname, _j in _data["eval_cols"]:
        _src = "y0" if _j == 0 else "y1"

        _m = (_val_df["subset_name"] == _name).to_numpy()

        _type_level[_m, _j] = _val_df.groupby("group_key")[_src].transform("mean").to_numpy()[_m]

    _oracle_metrics = compute_compartment_metrics(

        _type_level, _gold, np.asarray(_data["val_ds"]["group_index"]),





        np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),

        _data["eval_cols"])









    if _run_name in SAVE_FINAL_FOR:
        trainer.save_model(str(_out_dir / "final"))

        tokenizer.save_pretrained(str(_out_dir / "final"))

        print(f"模型已保存到: {_out_dir / 'final'}")



    for _ck in sorted(_out_dir.glob("checkpoint-*")):
        shutil.rmtree(_ck, ignore_errors=True)





    _settings = dict(

        model_key=MODEL_KEY, model_id=MODEL_REGISTRY[MODEL_KEY]["model_id"],

        run_name=_run_name,

        loss_mode=LOSS_MODE, head_kind=HEAD_KIND,

        soft_bins=int(SOFT_BINS), guard_fuse_w=GUARD_FUSE_W,

        pool_mode=POOL_MODE,

        loss_profile={k: dict(v) for k, v in SUBSET_LOSS_PROFILE.items()},

        run_plan=[list(_rp) for _rp in RUN_PLAN],

        optim=OPTIM, gradient_checkpointing=USE_GRAD_CKPT,

        fit_tier=FIT_TIER_NAME, trainable_scope=FIT_SCOPE_FRAC,

        mixed_precision=("amp" if (USE_BF16 or USE_FP16) else "fp32"),

        gpu_index=GPU_INDEX,

        effective_batch=BATCH_SIZE * GRAD_ACCUM, micro_batch=BATCH_SIZE,

        grad_accum=GRAD_ACCUM, lr=LR, epochs=EPOCHS, max_length=MAX_LENGTH,

        run_epochs=int(_run_epochs), joint_epochs=int(JOINT_EPOCHS),

        metric_for_best_model=BEST_METRIC_KEY,

        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,

        huber_delta=HUBER_DELTA,

        auto_balance=bool(AUTO_BALANCE),

        order_tau=ORDER_TAU, xrank_margin=XRANK_MARGIN, xrank_tau=XRANK_TAU,

        balance_ema_beta=BALANCE_EMA_BETA,

        z_m1=Z_M1, z_m2=Z_M2, z_m3=Z_M3,

        best_metric_formula="m1/Z_M1 + m2/Z_M2 - m3/Z_M3",

        loss_finite_check_every=LOSS_FINITE_CHECK_EVERY,

        data_mode=DATA_MODE,

        stats_dtype=(str(trainer._gs[0].dtype).replace("torch.", "")

                     if trainer._gs and trainer._gs[0] is not None else "n/a"),

        actual_effective_batch=_actual_eff_batch,

        actual_max_steps=_actual_max_steps,

        actual_warmup_steps=_actual_warmup,

        actual_warmup_frac=round(_actual_wf, 4),

        dataloader_len=_dl_len,

        loss_term_weights={k: round(v, 4) for k, v in _loss_terms.items()},

        seed=SEED, n_outputs=N_OUTPUTS,

        train_subset=_subsets,

        logging_steps=int(trainer.args.logging_steps),

    )

    _set_dir = OUTPUT_ROOT / "summary"

    _set_dir.mkdir(parents=True, exist_ok=True)

    with open(_set_dir / f"run_settings_{_run_name}.json", "w", encoding="utf-8") as _f:
        json.dump(_settings, _f, ensure_ascii=False, indent=2)

    ALL_RUNS.append(dict(

        run_name=_run_name, mode=_mode, subset=_subset, subsets=list(_subsets),

        eval_cols=_data["eval_cols"], out_dir=_out_dir,

        metrics=_trained_metrics, random_metrics=_rand_metrics, oracle_metrics=_oracle_metrics,

        settings=_settings,

        preds=np.asarray(_pred_out.predictions).reshape(len(_gold), -1),

        gold=_gold,

        groups=np.asarray(_data["val_ds"]["group_index"]),

        subset_ids=np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),

        val_df=_val_df,

        train_minutes=_dt / 60,

        log_history=_log_hist,

        random_eval_loss=_rand_eval_loss,

    ))

    print(f"→ 运行 {_run_name} 完成，产物在 {_out_dir}")





    del trainer, model, _pred_out

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()



print("\n" + "=" * 78)

print(f"全部 {len(ALL_RUNS)} 次运行完成")

print("=" * 78)


## Step9 模型评估、对照图与结果分析

这一节分成**两个互不混淆的区块**：

### 区块一：评估标准（模型本身的结果）

这就是**官方评测会看到的数字**——模型在 6 个评测设置上的三项指标。
表 1 与图 A / 图 B / 图 C 全部属于这一块。

### 区块二：参考对照（不是评估标准）

两个**人造系统**，用来回答「这个分数算好吗」：

| 对照系统 | 它是什么 | 用来回答什么问题 |
|---|---|---|
| **随机初始化** | 同一个模型、同一份数据，但**没训练过** | 「模型到底学到了什么」——训练后必须显著优于它，否则等于没训 |
| **type-level oracle** | 给每个目标赋它自己的**真实均值**，完全不看语境 | 「天花板在哪」——它在指标 1 / RMSE 上能拿满分，但指标 2 必然是 `nan`（组内 std 恒为 0）。我们的模型如果指标 2 也接近 nan，说明还没真正做到「因句而异」 |

**这两个都不是提交系统的成绩**，所以它们被单独放在区块二，
不与评估标准混在同一张表、同一根柱子里。

**六张图各管一件事**：

| 图 | 内容 | 看什么 |
|---|---|---|
| 图 A | 预测 vs 真值散点（每个「子集×列」一个面板） | 点是否落在对角线上；ρ 是整体排序能力，RMSE 是数值校准 |
| 图 B | 6 个评测设置 × 三项指标 | **评估标准**本身：哪个设置强、哪个设置弱、指标 2 有没有塌成 nan |
| 图 C | 组内 std 对照散点（人工 std vs 预测 std） | 这就是指标 2 的可视化：横轴是金标准，纵轴是系统输出，ρ 即指标 2 |
| 图 D | 五次运行的合计指标横向对照 | 「联合训练」到底有没有比「单子集训练」更好 |
| 图 E | 训练后 / 随机初始化 / type-level oracle | 参照系：这个分数算好吗（**不是**评估标准） |
| 图 F | 训练损失曲线（训练集与测试集同图、不同颜色） | 训练过程健不健康：有没有收敛、有没有过拟合（泛化差距） |

**为什么图 C 最关键**：指标 2 直接决定是否出局。
如果图 C 的点挤成一条水平线（预测的组内 std 几乎不变），
说明模型没有真正对语境敏感——哪怕指标 1 很高，这份提交也会在指标 2 上拿 `nan`。

In [ ]:

# ============ 绘图工具 ============





def style_axes(ax):
    """去掉上/右边框，网格线淡化到 alpha 0.22，坐标轴置于数据之下。"""

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

    ax.grid(True, alpha=0.22, linewidth=0.6)

    ax.set_axisbelow(True)





def annotate_box(ax, x, y, text, color="#2C3E50", ha="left", va="top"):
    """带白色不透明底衬的标注框。
    【为什么不透明】半透明白底（alpha<1）挡不住后面的线条，线会穿字而过。"""

    ax.text(x, y, text, ha=ha, va=va, fontsize=8.4, color=color,

            bbox=dict(boxstyle="round,pad=0.32", facecolor="white",

                      edgecolor=color, linewidth=0.8, alpha=1.0))





def label_bars(ax, items, ylim, fontsize=7.0):
    """把数值直接标在柱子上，并自动错开互相压叠的标签。
    items: [(x, 柱高, 文本)]；ylim: (下界, 上界) 用于计算错开的间距。"""

    lo, hi = ylim

    gap = (hi - lo) * 0.058

    placed = []

    for x, y, t in sorted(items, key=lambda z: (z[0], -z[1])):
        yy = y

        for px, py in placed:
            if abs(px - x) < 0.32 and abs(py - yy) < gap:
                yy = py + gap

        placed.append((x, yy))

        va = "bottom" if y >= 0 else "top"

        ax.text(x, yy, t, ha="center", va=va, fontsize=fontsize, color="#2C3E50",

                bbox=dict(boxstyle="round,pad=0.16", facecolor="white",

                          edgecolor="none", alpha=1.0))





BLUE, ORANGE, GREY, GREEN, RED = "#3C6E9E", "#C2703D", "#8A8F98", "#2E7D5B", "#B03A2E"





MAIN_RUN = next((r for r in ALL_RUNS if str(r["mode"]).startswith("joint")), ALL_RUNS[-1])

print("主图使用的运行:", MAIN_RUN["run_name"], "| 评测设置:", len(MAIN_RUN["eval_cols"]))


In [ ]:

# ============ 表 1：评估标准（官方六项设置 × 三项指标）============





_rows = []

for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        _rows.append({

            "运行": _r["run_name"],

            "子集": _n,

            "标签列": _cn,

            "指标1 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标1_ρ", float("nan")),

            "指标2 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标2_ρ", float("nan")),

            "指标3 RMSE": _r["metrics"].get(f"{_n}|{_cn}_指标3_RMSE", float("nan")),

            "组内std退化目标数": _r["metrics"].get(f"{_n}|{_cn}_组内std退化目标数", 0),

            "样例数": _r["metrics"].get(f"{_n}|{_cn}_样例数", 0),

        })

RESULTS = pd.DataFrame(_rows)

MAIN_RESULTS = RESULTS[RESULTS["运行"] == MAIN_RUN["run_name"]].reset_index(drop=True)

print("=" * 96)

print(f"表 1  评估标准 —— {MAIN_RUN['run_name']}（联合运行覆盖全部评测设置）")

print("=" * 96)

print(MAIN_RESULTS.round(4).to_string(index=False))

print(f"\n合计（各评测设置取平均）：指标1 ρ={MAIN_RESULTS['指标1 ρ'].mean():.4f} | "

      f"指标2 ρ={MAIN_RESULTS['指标2 ρ'].mean():.4f} | "

      f"RMSE={MAIN_RESULTS['指标3 RMSE'].mean():.4f}")

print(f"best-checkpoint 选择指标（{BEST_METRIC_KEY}，= 指标1/{Z_M1} + 指标2/{Z_M2} − RMSE/{Z_M3}，"

      f"即在 z 分数空间里等权）= "

      f"{MAIN_RUN['metrics'].get(BEST_METRIC_KEY, float('nan')):.4f}")

print("→ 指标 2 若是 nan，说明该设置下模型对同一目标的所有例句给出了相同分数，"

      "会在官方评测中直接出局。")





_summary_dir = OUTPUT_ROOT / "summary"

_summary_dir.mkdir(parents=True, exist_ok=True)

RESULTS.to_csv(_summary_dir / "results_all_runs.csv", index=False)

print("\n已保存:", _summary_dir / "results_all_runs.csv")







try:
    with open(_summary_dir / "arm_config.json", "w", encoding="utf-8") as _f:
        json.dump({

            "baseline": MODEL_KEY,

            "run_plan": _RUN_PLAN_ENV or "all",

            "arms": {k: dict(v) for k, v in V12_ARMS.items()},

            "ordinal": dict(soft_bins=int(SOFT_BINS), guard_fuse_w=GUARD_FUSE_W,

                            pool=POOL_MODE,

                            readout="期望值 ŷ = Σ k·p_k（值域 [0,5] 由桶凸组合保证）"),

            "source": "V12 序数目标对照（本轮新跑）",

        }, _f, ensure_ascii=False, indent=2)

    print("已保存:", _summary_dir / "arm_config.json")

except Exception as _e:
    print("arm_config.json 写入失败（不影响训练与指标）:", _e)

except Exception as _e:
    print("arm_config.json 写入失败（不影响训练与指标）:", _e)



# ============ 口径清单：本实验的独立口径记录 ============















_DECLARED = [

    "fit_tier", "trainable_scope", "mixed_precision", "optim",

    "effective_batch", "actual_effective_batch",

    "lr", "epochs", "run_epochs", "joint_epochs",

    "warmup_ratio", "actual_warmup_frac",

    "huber_delta", "seed", "n_outputs", "max_length", "weight_decay",

    "metric_for_best_model", "auto_balance", "order_tau", "xrank_margin",

    "xrank_tau", "balance_ema_beta", "loss_finite_check_every",

]



_MECH_ITEMS = {

    "pool_mode": "本运行的池化方式（上游判读冻结为 mean = 全句掩码均值）",

    "loss_profile": "本运行的按子集损失占比表（base 臂 huber+guard；序数臂不走 balancer）",

    "loss_mode": "本运行的损失模式（base / ord / ordmix）",

    "head_kind": "本运行的读出头（linear / ordinal 6-bin）",

    "z_m1": "best 准则的 z 分数常数",

    "z_m2": "best 准则的 z 分数常数",

    "z_m3": "best 准则的 z 分数常数",

}



_FREE_ITEMS = {

    "model_key": "模型身份", "model_id": "模型身份",

    "run_name": "运行名（= 对照臂名）",

    "soft_bins": "软标签的桶数（0..5 有序桶）",

    "guard_fuse_w": "序数臂塌缩保险丝的固定权重",

    "micro_batch": "batch 分解方式（乘积 = effective_batch 即等价）",

    "grad_accum": "batch 分解方式（乘积 = effective_batch 即等价）",

    "gradient_checkpointing": "只影响速度与显存，数学上等价",

    "gpu_index": "用哪一张**同型号**卡（只影响并行调度）",

    "train_subset": "本次运行的子集",

    "actual_max_steps": "本次运行的总更新步数（随轮数变）",

    "actual_warmup_steps": "本次运行的 warmup 绝对步数",

    "dataloader_len": "本次运行每轮的批数",

    "loss_term_weights": "base 臂 = 自适应标定 EMA；序数臂 = 固定权重（_ord_fixed_terms）",

    "stats_dtype": "组统计矩的累加精度（CUDA/CPU=float64；MPS 不支持 float64）",

    "run_plan": "本次实际跑了哪几次（run_plan 裁剪后）",

    "data_mode": "数据口径（V12 固定 orig）",

    "seed": "训练种子",

    "n_outputs": "输出列数",

}

print("\n" + "=" * 78)

print("本实验的口径（独立记录；A 组里「声明值」与「实际值」必须一致）")

print("=" * 78)

_s = ALL_RUNS[0]["settings"]

print(f"\n【A. 本实验口径】{len(_DECLARED)} 项")

for _k in _DECLARED:
    print(f"  {_k:24s} = {_s.get(_k)}")

print("\n  声明值 vs 实际值核对（必须一致）：")

for _d, _a, _why in [

    ("effective_batch", "actual_effective_batch", "有效 batch：声明值 = 实际生效值"),

    ("warmup_ratio", "actual_warmup_frac", "warmup 占比：声明值 = 实际生效值"),

    ("fit_tier", "trainable_scope", "档位 = 可训层比例（t3 ⇒ 1.0 全参）"),

]:
    print(f"    {_d}={_s.get(_d)} | {_a}={_s.get(_a)}    ← {_why}")

print(f"\n【B. 机制项】{len(_MECH_ITEMS)} 项")

for _k, _why in _MECH_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")

print(f"\n【C. 实现细节 / 环境】{len(_FREE_ITEMS)} 项 —— 允许不同，不影响数值")

for _k, _why in _FREE_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")

print("\n口径文件:", _summary_dir / f"run_settings_{ALL_RUNS[0]['run_name']}.json")

print("判读规则：")

print("   ① 本实验独立成立——不要求与另一个实验口径一致；")

print("   ② A 组的「声明值」必须等于「实际生效值」；")

print("   ③ 与其它实验比较时，用这张表核对差异，不要假设两边一定一样。")


In [ ]:

# ============ 图 A：预测 vs 真值（按「子集 × 标签列」分面）============





_panels = MAIN_RUN["eval_cols"]

_pred = MAIN_RUN["preds"]

_gold = MAIN_RUN["gold"]

_groups = MAIN_RUN["groups"]

_subsets = MAIN_RUN["subset_ids"]

_nc = min(3, len(_panels))

_nr = math.ceil(len(_panels) / _nc)

fig, axes = plt.subplots(_nr, _nc, figsize=(4.8 * _nc, 4.5 * _nr), squeeze=False)

for _k, (_name, _cname, _j) in enumerate(_panels):
    ax = axes[_k // _nc][_k % _nc]

    _m = _subsets == _name

    ax.scatter(_gold[_m, _j], _pred[_m, _j], s=14, alpha=0.42, color=BLUE, edgecolors="none")

    _lo = max(0.0, float(min(_gold[_m, _j].min(), _pred[_m, _j].min())) - 0.3)

    _hi = float(max(_gold[_m, _j].max(), _pred[_m, _j].max())) + 0.3

    ax.plot([_lo, _hi], [_lo, _hi], "--", color=GREY, linewidth=1.1, zorder=1)

    ax.set_xlim(_lo, _hi); ax.set_ylim(_lo, _hi)

    ax.set_xlabel(T("gold")); ax.set_ylabel(T("pred"))

    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)

    _r = stats.spearmanr(_pred[_m, _j], _gold[_m, _j])[0]

    _e = float(np.sqrt(((_pred[_m, _j] - _gold[_m, _j]) ** 2).mean()))

    annotate_box(ax, _lo + 0.04 * (_hi - _lo), _hi - 0.04 * (_hi - _lo),

                 f"ρ = {_r:.3f}\nRMSE = {_e:.3f}")

    style_axes(ax)

for _k in range(len(_panels), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")

fig.suptitle(f"{T('figA')} ({MAIN_RUN['run_name']})", fontsize=12.5,

             fontweight="bold", y=1.02)

fig.tight_layout()

_figA = _summary_dir / "figA_pred_vs_gold.png"

fig.savefig(_figA, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figA)


In [ ]:

# ============ 图 B：评估标准 —— 六个评测设置 × 三项指标 ============





_metric_defs = [("指标1_ρ", T("rho_up"), T("m1")),

                ("指标2_ρ", T("rho_up"), T("m2")),

                ("指标3_RMSE", T("rmse_down"), T("m3"))]

_xlabels = [f"{n}\n{c}" for n, c, _j in _panels]



fig, axes = plt.subplots(1, 3, figsize=(2.0 * len(_panels) + 3.6, 5.2))

for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]

    _vals = [MAIN_RUN["metrics"].get(f"{n}|{c}_{_suffix}", float("nan")) for n, c, _j in _panels]

    _finite = [v for v in _vals if np.isfinite(v)]

    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]

    _colors = [BLUE if np.isfinite(v) else "#D8DBE0" for v in _vals]

    ax.bar(np.arange(len(_panels)), _heights, 0.6, color=_colors, edgecolor="white")





    _lo = min([0.0] + _finite) if _finite else 0.0

    _hi = max([0.0] + _finite) if _finite else 1.0

    _span = max(_hi - _lo, 1e-6)

    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.22 * _span)

    _items = [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)]

    label_bars(ax, _items, ax.get_ylim(), fontsize=7.4)

    for k, v in enumerate(_vals):
        if not np.isfinite(v):


            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",

                    fontsize=8.0, color=RED, fontweight="bold")

    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)

    ax.set_xticks(np.arange(len(_panels)))

    ax.set_xticklabels(_xlabels, fontsize=7.2)

    ax.set_ylabel(_ylabel)

    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)

    style_axes(ax)

fig.suptitle(f"{T('figB')}: {MAIN_RUN['run_name']} on {len(_panels)} {T('n_settings')}",

             fontsize=12.5, fontweight="bold", y=1.02)

fig.tight_layout()

_figB = _summary_dir / "figB_evaluation_standard.png"

fig.savefig(_figB, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figB)


In [ ]:

# ============ 图 C：组内 std 对照（指标 2 的可视化，按「子集×列」分面）============













_c_sub = list(_panels)

_nc = min(3, len(_c_sub))

_nr = math.ceil(len(_c_sub) / _nc)

fig, axes = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 4.3 * _nr), squeeze=False)

for _k, (_name, _cname, _j) in enumerate(_c_sub):
    ax = axes[_k // _nc][_k % _nc]

    _m = _subsets == _name

    _gs = pd.DataFrame({"g": _groups[_m], "p": _pred[_m, _j], "y": _gold[_m, _j]}).groupby("g")

    _ps = _gs["p"].std(ddof=1)

    _ys = _gs["y"].std(ddof=1)

    _ok = _ps.notna() & _ys.notna()

    _r2 = stats.spearmanr(_ps[_ok], _ys[_ok])[0] if _ok.sum() > 2 else float("nan")

    ax.scatter(_ys[_ok], _ps[_ok], s=20, alpha=0.55, color=ORANGE, edgecolors="none")

    _mx = float(max(_ys[_ok].max(), _ps[_ok].max())) * 1.10 if _ok.sum() else 1.0

    ax.plot([0, _mx], [0, _mx], "--", color=GREY, linewidth=1.1, zorder=1)

    ax.set_xlim(0, _mx); ax.set_ylim(0, _mx)

    ax.set_xlabel(T("gold_std"))

    ax.set_ylabel(T("pred_std"))

    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)

    annotate_box(ax, _mx * 0.03, _mx * 0.97,

                 f"{T('m2rho')} {_r2:.3f}\n{T('n_targets')} {int(_ok.sum())}\n"

                 f"{T('const_std')} {int((_ps[_ok] == 0).sum())}")

    style_axes(ax)

for _k in range(len(_c_sub), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")

fig.suptitle(f"{T('figC')}: {T('gold_axis')} ({MAIN_RUN['run_name']})",

             fontsize=12.5, fontweight="bold", y=1.02)

fig.tight_layout()

_figC = _summary_dir / "figC_group_std.png"

fig.savefig(_figC, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figC)


In [ ]:

# ============ 图 D：各次运行的横向对照（single 四个子集 vs 联合）============







_run_names = [r["run_name"] for r in ALL_RUNS]

_run_labels = []

for _r in ALL_RUNS:
    if str(_r["mode"]).startswith("joint"):
        _run_labels.append(T("joint_short"))

    else:
        _run_labels.append(T("single_short", short=short_of(_r["subset"])))



fig, axes = plt.subplots(1, 3, figsize=(2.1 * len(ALL_RUNS) + 3.2, 4.8))

for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]

    _vals = [_r["metrics"].get(f"合计|_{_suffix}", float("nan")) for _r in ALL_RUNS]

    _finite = [v for v in _vals if np.isfinite(v)]

    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]

    _colors = [ORANGE if str(_r["mode"]).startswith("joint") else BLUE for _r in ALL_RUNS]

    ax.bar(np.arange(len(ALL_RUNS)), _heights, 0.62, color=_colors, edgecolor="white")

    _lo = min([0.0] + _finite) if _finite else 0.0

    _hi = max([0.0] + _finite) if _finite else 1.0

    _span = max(_hi - _lo, 1e-6)

    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)

    label_bars(ax, [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)],

               ax.get_ylim(), fontsize=7.6)

    for k, v in enumerate(_vals):
        if not np.isfinite(v):
            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",

                    fontsize=8.0, color=RED, fontweight="bold")

    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)

    ax.set_xticks(np.arange(len(ALL_RUNS)))

    ax.set_xticklabels(_run_labels, fontsize=7.4)

    ax.set_ylabel(_ylabel)

    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)

    style_axes(ax)

_handles = [plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)]

fig.legend(_handles, [T("single"), T("joint")],

           loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=9, frameon=False)

fig.suptitle(T("figD"), fontsize=12.5, fontweight="bold", y=1.06)

fig.tight_layout()

_figD = _summary_dir / "figD_run_comparison.png"

fig.savefig(_figD, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figD)


In [ ]:

# ============ 区块二：参考对照（不是评估标准）============











print("=" * 96)

print("区块二  参考对照（不是评估标准，仅用于解读上面的数字）")

print("=" * 96)

_cmp_rows = []

for _r in ALL_RUNS:
    for _sys, _m in [("训练后模型", _r["metrics"]),

                     ("随机初始化", _r["random_metrics"]),

                     ("type-level oracle", _r["oracle_metrics"])]:
        _cmp_rows.append({

            "运行": _r["run_name"], "对照系统": _sys,

            "指标1 ρ": _m.get("合计|_指标1_ρ", float("nan")),

            "指标2 ρ": _m.get("合计|_指标2_ρ", float("nan")),

            "指标3 RMSE": _m.get("合计|_指标3_RMSE", float("nan")),

        })

COMPARE = pd.DataFrame(_cmp_rows)

print(COMPARE.round(4).to_string(index=False))

COMPARE.to_csv(_summary_dir / "reference_comparison.csv", index=False)



print("\n--- 三个系统分别是什么 ---")

print("随机初始化      ：同一模型、同一份数据，但一步都没训练。"

      "它是「瞎猜」的下界；训练后模型必须显著高于它，否则等于没训。")

print("type-level oracle：给每个目标赋它自己的真实均值（不看语境）。"

      "它在指标 1 / RMSE 上接近满分，但指标 2 必然是 nan"

      "（同一目标的所有例句拿到同一个分数 → 组内 std 恒为 0 → 相关系数未定义）。")

print("→ 我们的模型如果指标 2 也接近 nan，说明还没有真正做到「因句而异」。")


In [ ]:

# ============ 图 E：参考对照（训练后 / 随机初始化 / type-level oracle）============



_cmp_systems = [(T("trained"), "metrics", BLUE),

                (T("random_init"), "random_metrics", GREY),

                (T("oracle"), "oracle_metrics", ORANGE)]

fig, axes = plt.subplots(1, 3, figsize=(11.0, 4.6))

for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]

    _items = []

    for _si, (_sys, _key, _c) in enumerate(_cmp_systems):
        _v = MAIN_RUN[_key].get(f"合计|_{_suffix}", float("nan"))

        _x = _si

        if np.isfinite(_v):
            ax.bar(_x, _v, 0.55, color=_c, edgecolor="white")

            _items.append((_x, _v, f"{_v:.3f}"))

        else:
            ax.bar(_x, 0.0, 0.55, color="#D8DBE0", edgecolor="white")

    _finite = [y for _x, y, _t in _items]

    _lo = min([0.0] + _finite) if _finite else 0.0

    _hi = max([0.0] + _finite) if _finite else 1.0

    _span = max(_hi - _lo, 1e-6)

    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)

    label_bars(ax, _items, ax.get_ylim(), fontsize=8.4)



    if _suffix == "指标2_ρ":
        ax.text(2, _lo + 0.02 * _span, "nan", ha="center", va="bottom",

                fontsize=9.0, color=RED, fontweight="bold")

    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)

    ax.set_xticks(range(len(_cmp_systems)))

    ax.set_xticklabels([s for s, _k, _c in _cmp_systems], fontsize=8.2)

    ax.set_ylabel(_ylabel)

    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)

    style_axes(ax)

fig.suptitle(f"{T('figE')} ({T('not_standard')}): {MAIN_RUN['run_name']}",

             fontsize=12.0, fontweight="bold", y=1.03)

fig.tight_layout()

_figE = _summary_dir / "figE_reference_comparison.png"

fig.savefig(_figE, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figE)


In [ ]:

# ============ 图 F：训练损失曲线（训练集与测试集在同一幅图里、不同颜色）============



















def _loss_series(hist):
    """把 log_history 拆成 (训练点, 评估点) 两条序列。
    【为什么要拆开】一条记录里要么有 'loss'（训练），要么有 'eval_loss'（评估），
       混在一起画会出现「评估点被当成训练点」的错位。"""

    tr = [(float(h.get("epoch", 0.0)), float(h["loss"]))

          for h in hist if "loss" in h and "eval_loss" not in h]

    ev = [(float(h.get("epoch", 0.0)), float(h["eval_loss"]))

          for h in hist if "eval_loss" in h]

    tr.sort()

    ev.sort()

    return tr, ev





_RUN_COLORS = [BLUE, GREEN, ORANGE, "#7A5195", "#3F7F93"]

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.7))





ax = axes[0]

for _ri, _r in enumerate(ALL_RUNS):
    _tr, _ev = _loss_series(_r.get("log_history", []))

    if not _tr:
        continue

    _lbl = (T("joint_short").replace("\n", " ") if str(_r["mode"]).startswith("joint")

            else short_of(_r["subset"]))

    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], marker="o", markersize=3.0,

            linewidth=1.5, color=_RUN_COLORS[_ri % len(_RUN_COLORS)], label=_lbl)

ax.set_xlabel(T("epoch"))

ax.set_ylabel(T("loss"))

ax.set_title(f"(a) {T('loss_all_runs')}", fontsize=10.5)

ax.legend(fontsize=8.2, frameon=False)

style_axes(ax)





ax = axes[1]

_tr, _ev = _loss_series(MAIN_RUN.get("log_history", []))





_rl = MAIN_RUN.get("random_eval_loss", float("nan"))

if np.isfinite(_rl):
    _ev = [(0.0, float(_rl))] + _ev

if _tr:
    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], color=BLUE, linewidth=1.6,

            marker="o", markersize=3.2, label=T("train_loss"))

if _ev:
    _ex = [p[0] for p in _ev]

    _ey = [p[1] for p in _ev]

    ax.plot(_ex, _ey, color=RED, linewidth=1.8, marker="s", markersize=6.0,

            label=T("eval_loss"))

    for _i0, (_x0, _y0) in enumerate(zip(_ex, _ey)):




        _va = "bottom" if _i0 == 0 else "top"

        annotate_box(ax, _x0, _y0, f"{_y0:.3f}", color=RED, ha="center", va=_va)







    if _ex and abs(_ex[0]) < 1e-9:
        ax.annotate(T("untrained"), xy=(_ex[0], _ey[0]), xytext=(0.30, 0.52),

                    textcoords="axes fraction", fontsize=8.2, color=RED,

                    arrowprops=dict(arrowstyle="->", color=RED, linewidth=0.9))

ax.set_xlabel(T("epoch"))

ax.set_ylabel(T("loss"))

ax.set_title(f"(b) {T('loss_main')}", fontsize=10.5)

ax.legend(fontsize=8.6, frameon=False)

style_axes(ax)



fig.suptitle(f"{T('figF')}: {MAIN_RUN['run_name']}", fontsize=12.5,

             fontweight="bold", y=1.02)

fig.tight_layout()

_figF = _summary_dir / "figF_loss_curves.png"

fig.savefig(_figF, dpi=160, bbox_inches="tight")

plt.show()

plt.close(fig)

print("已保存:", _figF)


## 附录：推理与提交（`*-pred.tsv` 导出）

**为什么把导出放在最后**：提交格式是本任务**最容易在最后一天翻车**的环节。
下面这份导出函数已经把所有已知的坑写成了断言——任何一条不满足都直接 `raise`，
而不是让它静默走到提交那天。

**格式规范（官方）**：

- 提交物是**一个 zip**，内含 4 个文件：`en-nn-pred.tsv` / `en-pv-pred.tsv` /
  `de-nn-pred.tsv` / `de-pv-pred.tsv`
- **无表头**，制表符分隔
- 名词复合词每行 **3** 列：`ContextID` ⇥ `修饰语预测` ⇥ `中心语预测`（**修饰语在前**）
- 小品词动词每行 **2** 列：`ContextID` ⇥ `单一预测`
- 测试集**不含** `*Avg` / `*Std` 列，载入逻辑必须容忍缺列
- 预测值必须落在 **[0, 5]**；输出**不得是常数**（否则指标 2 = nan）

In [ ]:

# ============ 导出函数 ============

def export_pred(tsv_path, preds, lang, task):
    """把预测导出成官方格式的 *-pred.tsv，并做全部自检。
    【为什么每条都写成断言而不是 print】提交脚本一旦出错，
       官方评测器**不会报格式错误**，只会静默给出离谱的分数。
       与其提交后怀疑人生，不如在这里就把红线画出来。"""

    test = pd.read_csv(tsv_path, sep="\t")

    ids = test["ContextID"].astype(str)

    preds = np.asarray(preds, dtype=float)

    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)



    if task == "nn":
        assert preds.shape[1] == 2, f"名词复合词需要 2 列（修饰语、中心语），收到 {preds.shape[1]}"

        body = pd.DataFrame({"ContextID": ids, "mod": preds[:, 0], "head": preds[:, 1]})

    else:
        assert preds.shape[1] == 1, f"小品词动词需要 1 列，收到 {preds.shape[1]}"

        body = pd.DataFrame({"ContextID": ids, "pred": preds[:, 0]})



    assert len(body) == len(test), "行数与测试集不一致"

    assert set(body["ContextID"]) == set(ids), "ContextID 集合与测试集不一致"

    _v = body.select_dtypes("number").to_numpy()

    assert np.isfinite(_v).all(), "预测里出现 NaN/inf"

    assert _v.min() >= 0.0 and _v.max() <= 5.0, f"预测超出 [0,5]：[{_v.min():.3f}, {_v.max():.3f}]"



    out = OUTPUT_ROOT / f"{lang}-{task}-pred.tsv"

    body.to_csv(out, sep="\t", index=False, header=False)

    return out





# ============ 演示：按运行逐个子集导出，检查格式 ============







_exported = []

for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        if _cn != SUBSET_SPECS[_n]["cols"][0]:
            continue

        _sp = SUBSET_SPECS[_n]

        _m = _r["subset_ids"] == _n

        _cols = [_r["preds"][_m, jj] for jj in range(len(_sp["cols"]))]

        _tsv = OUTPUT_ROOT / f"_demo_test_{_n}.tsv"

        pd.DataFrame({"ContextID": [f"{i:05d}" for i in range(int(_m.sum()))],

                      "Context": _r["val_df"].loc[_m, "Context"].to_numpy()

                      }).to_csv(_tsv, sep="\t", index=False)

        _out = export_pred(_tsv, np.stack(_cols, axis=1), _sp["lang"], _sp["task"])

        _exported.append((_r["run_name"], _n, _out))

        print(f"[{_r['run_name']} / {_n}] -> {_out.name}")

print("\n→ 无表头、列数正确、值域 [0,5]、ContextID 原样透传。")

print("→ 正式提交时把 4 个 *-train.tsv 换成官方测试 TSV，文件名保持 en-nn-pred.tsv 等 4 个。")


## Step10 判读（预注册：跑之前定死阈值与读法）

**输入**：`ALL_RUNS` 里三个联合臂（同切分 ⇒ 验证集逐行对齐，配对差值合法）。
**阈值**：合计口径 `analysis/metric_noise.json`（指标1 **0.0652** / 指标2 **0.1526** /
RMSE **0.0463**）。**判读顺序**：R4 红线 → 反向否决 → 采纳规则。
**参照**：冻结配方在 seed 42 的实测 **0.5825**（上游）/ **0.5782**（V11 复测，同配方同切分）、
上游六次均值 **0.5853**、v4.4 0.4982。

In [ ]:

# ============ Step10.1 三臂汇总 + 逐轮轨迹 + 红线 ============

V12_MDE = {"m1": 0.0652, "m2": 0.1526, "rmse": 0.0463}

V12_REF = dict(frozen_v10=0.5825, frozen_rep=0.5782, sweep_mean=0.5853, v4_4=0.4982)

V12_JOINTS = [r for r in ALL_RUNS if str(r["mode"]).startswith("joint")]

assert V12_JOINTS, "ALL_RUNS 里没有联合臂，Step10 无从分析"





def _v12_metrics(run):
    """一个臂的合计三指标 + 组内 std 退化目标数。"""

    m = run["metrics"]

    return dict(m1=float(m["合计|_指标1_ρ"]), m2=float(m["合计|_指标2_ρ"]),

                rmse=float(m["合计|_指标3_RMSE"]),

                ndeg=int(m["合计|_组内std退化目标数"]))





def _v12_traj(run, key="指标1"):
    """逐轮合计指标轨迹。

    键必须带 eval_ 前缀（log_history 是 trainer.state 原始快照，Trainer.evaluate()
    写入的键全部带前缀）；首点 epoch≈0 是随机初始化评估 ⇒ 跳过；同 epoch 去重
    保留原序后者（best 重评；不能按数值排序取大）。"""

    k = f"eval_合计|_{key}_ρ"

    pts = [(float(h["epoch"]), float(h[k]))

           for h in run["log_history"] if k in h and h.get("epoch") is not None]

    pts = [p for p in pts if p[0] > 1e-6]

    out = []

    for ep, v in pts:
        if out and abs(ep - out[-1][0]) < 1e-6:
            out[-1] = (ep, v)

        else:
            out.append((ep, v))

    return out





print("=" * 96)

if SMOKE:
    print("⚠️ SMOKE=1：以下全部判读仅为链路验证（tiny 模型 / 1 轮 / 400 行），不是证据；"

          "云端全量判读为准。")

print("表 1  三臂汇总（best-checkpoint；参照 冻结配方 0.5825/0.5782 / 六次均值 0.5853 / v4.4 0.4982）")

print("=" * 96)

print(f"{'臂':14s} {'loss_mode':9s} {'指标1ρ':>8s} {'Δm1(vs 0.5825)':>14s} "

      f"{'指标2ρ':>8s} {'RMSE':>8s} {'退化':>4s} {'训练min':>8s}")

V12_ROWS = []

for r in V12_JOINTS:
    t = _v12_metrics(r)

    arm = V12_ARMS.get(r["run_name"], dict(loss_mode="?", head_kind="?"))

    d1 = t["m1"] - V12_REF["frozen_v10"]

    assert t["ndeg"] == 0, (f"[{r['run_name']}] 指标2 红线破坏：退化目标数 {t['ndeg']} ≠ 0，"

                            "停止分析")

    assert np.isfinite(t["m2"]), (f"[{r['run_name']}] 指标2 非有限"

                                  "（ndeg=0 但跨目标预测 std 恒等？）防御断言")

    tr = _v12_traj(r)

    assert tr, f"[{r['run_name']}] 逐轮轨迹为空（eval_ 前缀读取失败？）"

    print(f"{r['run_name']:14s} {arm['loss_mode']:9s} {t['m1']:8.4f} {d1:+14.4f} "

          f"{t['m2']:8.4f} {t['rmse']:8.4f} {t['ndeg']:4d} {r['train_minutes']:8.1f}")

    V12_ROWS.append(dict(run=r["run_name"], loss_mode=arm["loss_mode"], **t,

                         train_min=round(float(r["train_minutes"]), 1),

                         traj=[[ep, v] for ep, v in tr]))

V12_SUM = pd.DataFrame(V12_ROWS)



print("\n逐臂轨迹（合计指标1）：")

for _, row in V12_SUM.iterrows():
    line = " ".join(f"e{int(ep)}:{v:.4f}" for ep, v in row["traj"])

    print(f"  {row['run']:14s} {line}")


In [ ]:

# ============ Step10.2 预注册判读（反向否决 → RMSE 假设 → m1 假设）============

print("=" * 96)

print("表 2  预注册配对判读（阈值 0.0652/0.1526/0.0463；同一数据切分 ⇒ 配对差值合法）")

print("=" * 96)

_g = V12_SUM.set_index("run")["m1"].to_dict()

_g2 = V12_SUM.set_index("run")["m2"].to_dict()

_gr = V12_SUM.set_index("run")["rmse"].to_dict()

_need = ["joint_base", "joint_ord", "joint_ordmix"]

_missing = [n for n in _need if n not in _g]

assert not _missing, f"缺少预注册臂（被 RUN_PLAN 裁剪？）：{_missing}"



V12_ALIGN_OK = all(

    (r["val_df"]["group_key"].to_numpy()

     == V12_JOINTS[0]["val_df"]["group_key"].to_numpy()).all()

    and np.allclose(np.asarray(r["gold"]), np.asarray(V12_JOINTS[0]["gold"]))

    for r in V12_JOINTS)

assert V12_ALIGN_OK, "验证集未逐行对齐（数据切分种子被改动？）⇒ 配对差值不合法"

print("验证集逐行对齐检验：通过（数据切分种子恒为 42）")



V12_DELTA = {}

for _a in ("joint_ord", "joint_ordmix"):
    V12_DELTA[_a] = dict(m1=_g[_a] - _g["joint_base"], m2=_g2[_a] - _g2["joint_base"],

                         rmse=_gr[_a] - _gr["joint_base"])

    d = V12_DELTA[_a]

    print(f"{_a} vs base: Δm1 = {d['m1']:+.4f} | Δm2 = {d['m2']:+.4f} | ΔRMSE = {d['rmse']:+.4f}")



_veto = [a for a in ("joint_ord", "joint_ordmix")

         if V12_DELTA[a]["m1"] < -V12_MDE["m1"] or V12_DELTA[a]["m2"] < -V12_MDE["m2"]]

for a in _veto:
    print(f"反向否决：{a}（m1 或 m2 显著变差）⇒ 出局")

_alive = [a for a in ("joint_ord", "joint_ordmix") if a not in _veto]



V12_DECISION = "base"

if _alive:
    _best = min(_alive, key=lambda a: V12_DELTA[a]["rmse"])

    d = V12_DELTA[_best]

    print(f"幸存臂中 RMSE 最低：{_best}（ΔRMSE = {d['rmse']:+.4f}）")

    if d["rmse"] <= -V12_MDE["rmse"]:
        V12_DECISION = _best

        print(f"预注册决策：采纳 {_best}（RMSE 假设成立：ΔRMSE ≤ −0.0463 且未触发反向否决）")

    elif d["m1"] >= V12_MDE["m1"]:
        V12_DECISION = _best

        print(f"预注册决策：采纳 {_best}（m1 假设成立：Δm1 ≥ +0.0652 且未触发反向否决）")

    else:
        print("预注册决策：序数目标无可测收益 ⇒ 不采纳（提交配方保持冻结版）")

else:
    print("预注册决策：两个序数臂全部被反向否决 ⇒ 不采纳（提交配方保持冻结版）")



_r = {r["run_name"]: r for r in V12_JOINTS}

for _a in ("joint_ord", "joint_ordmix"):
    if _a not in _g:
        continue

    _lines = []

    for n, cname, _j in _r["joint_base"]["eval_cols"]:
        _k = f"{n}|{cname}_指标1_ρ"

        _dv = float(_r[_a]["metrics"][_k]) - float(_r["joint_base"]["metrics"][_k])

        _lines.append(f"{n}|{cname} {_dv:+.4f}")

    print(f"逐设置 Δm1（{_a} − base；仅作参考，预注册判读只看合计）:", " | ".join(_lines))

if SMOKE:
    print("⚠️ SMOKE=1：以上决策仅为链路验证输出（tiny 模型 / 1 轮），不是证据；云端全量判读为准。")



print(f"\n复现性参照：冻结配方上游实测 0.5825 / 上一版同配方复测 0.5782；"

      f"joint_base 与 0.5825 的差 = {_g['joint_base'] - V12_REF['frozen_v10']:+.4f}"

      "（同一配方跨会话的复现噪声，非差异证据）")

_best_row = V12_SUM.loc[V12_SUM["m1"].idxmax()]

print(f"本计划最佳臂：{_best_row['run']}  "

      f"{_best_row['m1']:.4f}/{_best_row['m2']:.4f}/{_best_row['rmse']:.4f}")

print("注：单次运行对照噪声较大，判读只按预注册阈值执行；「无差异」时按精简原则取冻结侧。")


In [ ]:

# ============ Step10.3 图 J/K/L + 落盘 ============

_out_v12 = OUTPUT_ROOT / "summary"

_out_v12.mkdir(parents=True, exist_ok=True)



_figJ, _axJ = plt.subplots(figsize=(10.5, 4.6))

_arm_colors = {"joint_base": "#7F8C8D", "joint_ord": "#2980B9", "joint_ordmix": "#27AE60"}

_xj = np.arange(len(V12_SUM))

_axJ.bar(_xj, V12_SUM["m1"], color=[_arm_colors.get(r, "#95A5A6") for r in V12_SUM["run"]])

for _x, _v in zip(_xj, V12_SUM["m1"]):
    _axJ.annotate(f"{_v:.4f}", (_x, _v), textcoords="offset points", xytext=(0, 3),

                  ha="center", fontsize=7)

_axJ.axhline(V12_REF["sweep_mean"], color="#7F8C8D", lw=1.1, ls="--",

             label=f"upstream sweep mean = {V12_REF['sweep_mean']:.4f}")

_axJ.axhline(V12_REF["frozen_v10"], color="#27AE60", lw=1.0, ls=":",

             label=f"frozen recipe = {V12_REF['frozen_v10']:.4f}")

_axJ.axhline(V12_REF["v4_4"], color="#B03A2E", lw=1.0, ls=":",

             label=f"v4.4 = {V12_REF['v4_4']:.4f}")

_axJ.set_xticks(_xj, [r.replace("joint_", "") for r in V12_SUM["run"]], fontsize=8)

_axJ.set_ylabel("Metric1 rho (total)")

_axJ.set_ylim(min(0.0, float(V12_SUM["m1"].min()) - 0.02),

              max(float(V12_SUM["m1"].max()) + 0.02, 0.62))

_axJ.set_title("Fig J  Ordinal target vs point regression (Metric1)", fontsize=10)

_axJ.legend(fontsize=8)

_figJ.tight_layout()

_figJ.savefig(_out_v12 / "figJ_v12_ordinal_arms.png", dpi=150)

plt.show()



_figK, _axK = plt.subplots(figsize=(10.5, 4.6))

for _, _row in V12_SUM.iterrows():
    _tr = _row["traj"]

    _axK.plot([p[0] for p in _tr], [p[1] for p in _tr], marker="o", ms=4, lw=1.4,

              color=_arm_colors.get(_row["run"], "#95A5A6"), label=_row["run"])

_axK.set_xlabel("epoch")

_axK.set_ylabel("Metric1 rho (total)")

_axK.set_ylim(0, max(0.65, max(p[1] for _, r in V12_SUM.iterrows()

                               for p in r["traj"]) + 0.03))

_axK.set_title("Fig K  Per-arm epoch trajectories", fontsize=10)

_axK.legend(fontsize=8)

_figK.tight_layout()

_figK.savefig(_out_v12 / "figK_v12_trajectories.png", dpi=150)

plt.show()



_soft_cases = [(1.0, 0.3), (2.5, 0.2), (3.8, 0.9), (4.5, 0.0)]

_figL, _axesL = plt.subplots(1, len(_soft_cases), figsize=(11.5, 3.2), sharey=True)

for _axL, (_m, _s) in zip(_axesL, _soft_cases):
    _q = _soft_target(_m, _s)

    _ev_q = sum(k * v for k, v in enumerate(_q))

    _axL.bar(range(SOFT_BINS), _q, color="#2980B9", alpha=0.75)

    _axL.axvline(_m - 0.5 + 0.5, color="#27AE60", lw=1.2, ls=":",

                 label=f"m = {_m}")

    _axL.axvline(_ev_q - 0.5 + 0.5, color="#C0392B", lw=1.2, ls="--",

                 label=f"EV(q) = {_ev_q:.2f}")

    _axL.set_title(f"m={_m}, s={_s}", fontsize=9)

    _axL.set_xlabel("rating bin")

    _axL.set_ylim(0, 1.0)

    _axL.legend(fontsize=7)

_axesL[0].set_ylabel("soft target mass")

_figL.suptitle("Fig L  Soft label distributions; dotted=m, dashed=EV(q) (binning pulls EV toward edges)", fontsize=10)

_figL.tight_layout()

_figL.savefig(_out_v12 / "figL_v12_soft_targets.png", dpi=150)

plt.show()



_dump = dict(

    ref=V12_REF, thresholds=V12_MDE,

    arms={k: dict(v) for k, v in V12_ARMS.items()},

    ordinal=dict(soft_bins=int(SOFT_BINS), guard_fuse_w=GUARD_FUSE_W,

                 reconstruction="truncated-normal binning from (mean, std, n=10)"),

    per_run=V12_SUM.drop(columns=["traj"]).to_dict(orient="records"),

    trajectories={row["run"]: [list(p) for p in row["traj"]] for _, row in V12_SUM.iterrows()},

    readout={a: {k: float(v) for k, v in d.items()} for a, d in V12_DELTA.items()},

    decision=V12_DECISION,

    decision_note=("SMOKE=1 chain-validation only, not evidence" if SMOKE

                   else "cloud full-data judgment"),

    run_settings={r["run_name"]: dict(loss_mode=r["settings"]["loss_mode"],

                                      head_kind=r["settings"]["head_kind"],

                                      seed=int(r["settings"]["seed"]),

                                      run_epochs=int(r["settings"]["run_epochs"]),

                                      effective_batch=int(r["settings"]["effective_batch"]))

                  for r in V12_JOINTS},

)

with open(_out_v12 / "v12_ordinal.json", "w", encoding="utf-8") as _f:
    json.dump(_dump, _f, ensure_ascii=False, indent=1, default=str)

V12_SUM.drop(columns=["traj"]).to_csv(_out_v12 / "v12_ordinal_runs.csv", index=False)

print("已写出 v12_ordinal.json / v12_ordinal_runs.csv / figJ / figK / figL")

print("判读顺序：R4 红线 → 反向否决 → 采纳规则")


## 结论与延伸

### 这份实验的运行前置条件


> 本文件是 **XLM-RoBERTa-large（V12：冻结架构 + 序数/软标签目标）** 的**受控对照实验**，由共享真源（`tools/nbgen/cells_a/b/c.py`，
> **只读**）经 `v12/nbgen/build_v12.py` 的 overlay 补丁生成：数据管线、切分、指标、
> 档位与 v8–v11 逐字同源。对照臂 = 上游判读冻结配方（mean 池化 + huber+guard + 4 轮），
> 一字不改；本实验唯一变量 = 训练目标（点回归 → 序数软标签分布）与读出头。
>
> **本文件验证的机制**：**序数/软标签目标（B4）在冻结架构上的 3 臂受控对照（4 轮，seed 42）**。

| 项 | 要求 | 说明 |
|---|---|---|
| Python | ≥ 3.10 | 已验证 3.12 |
| torch | ≥ 2.4 | 已验证 2.14 |
| transformers | ≥ 5.0 | **5.x 的 API 与 4.x 不同**：`eval_strategy`、`processing_class`、`warmup_steps`、`dtype=` 都是 5.x 写法；代码已做双兼容（签名探测自动切换） |
| datasets | ≥ 3.0 | 已验证 3.6 |
| 网络 | 云端 GPU 可直连 `huggingface.co`；本机通常不通 | Step1 会**自动探测**官方源，不通就换 `hf-mirror.com`，下载时再兜一层重试。也可设 `COMPARTMENT_HF_MODE=official\|mirror\|cache` 强制指定 |
| **显存** | **单卡 ≥ 12GiB 可用**（T4 14.56GiB 即可） | 训练「档位」默认写死 t3（全参微调 · Adafactor · 关 AMP），共 7 档且固定开销单调递减。Step8 会用真实模型真跑 2 步验证，OOM 自动降一档 |
| 内存 | ≥ 16GB | 主干 fp32 权重约 3.6GB / 6.2GB |
| 磁盘 | ≥ 10GB 可用 | 5 次运行只保留**联合运行**的权重（~3.6GB），训练用 checkpoint 训完即删 |
| 时间 | 5 次运行（4 个单子集 + 1 个联合） | 轮数 `EPOCHS=JOINT_EPOCHS`（注入档可覆盖）；总样本·轮 = 0.8×总行数×(EPOCHS+JOINT_EPOCHS)。Step8 开跑前会打印**时长投影**，超预算会报警 |

### 显存够不够：先看这张表

全参微调 + AdamW 的**固定开销**（权重 fp32 4 B/param + 梯度 fp32 4 B/param
+ AdamW 的 m/v 8 B/param = **16 B/param**，再加上开混合精度时 AMP 的 fp16 权重副本 2 B/param
→ **18 B/param**）**与 batch 大小无关**，所以「降 batch / 加梯度累积」救不了它。
真正能降下来的只有三样：**优化器状态**、**AMP 的 fp16 副本**、**可训练参数的范围**。

| 模型 | 参数量 | fp32 AdamW+AMP | 8bit+AMP | Adafactor+AMP | Adafactor 无 AMP（t3） |
|---|---|---|---|---|---|
| `microsoft/deberta-v2-xlarge` | 885M | 14.8 GiB | 9.9 GiB | 8.4 GiB | 6.8 GiB ✓ |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1544M | **25.9 GiB** | 17.3 GiB | 14.7 GiB ✗ | **11.8 GiB ✓** |

（表中为「固定开销」，不含激活；激活 + 显存碎片 + CUDA context 另计约 3 GiB。）

**T4 只有 14.56 GiB**，所以在 T4 上能保住「全参微调」语义的档位是
**t3（Adafactor + 关 AMP，8.2 B/param ≈ 11.8 GiB）** —— 没有任何参数被冻结。

### 为什么「9B 都能全量微调，1.5B 却超了」

这两句话说的**不是同一件事**，差别在三个地方：

| | 9B 全参微调（业界常见） | 本项目 1.5B 全参微调 |
|---|---|---|
| 硬件 | 2 × A100 **80GB**（合计 160GB） | 2 × T4 **14.56GB**（合计 29.1GB） |
| 固定开销 | 9B × 18 B = **162GB**，靠 ZeRO-3 分片塞进 160GB（还要再省） | 1.5B × 18 B = **25.9GB**，单卡 14.56GB 装不下 |
| 是否分片 | 用 ZeRO-3 / FSDP 把参数切到多卡 | notebook 是**单进程**，每卡都要放完整一份 |

关键在于：**DDP（数据并行）不省显存** —— 每张卡仍要放一整份模型+梯度+优化器状态，
只是把数据分半（提速用）。只有 **ZeRO-3 / FSDP 参数分片**才会把 25.9GB 摊到两张卡
（13GB/卡），而那需要 `accelerate launch --num_processes 2`，
**在 papermill 的单进程里跑不了**；而且本 notebook 自定义了分组采样器
（让同一目标的例句落在同一个 batch），DDP 下若不改成分布式采样器，
两个进程会取到**同一批样本** → 训练结果悄悄错掉。

所以本 notebook 的选择是：**单卡 + 把固定开销从 18 B/param 降到 8.2 B/param
（Adafactor + 关 AMP）**，这样 1.5B 也能在 T4 上做**完整的全参微调**。

### 训练档位（tier）：默认写死 t3（全参微调）

> 档位写死 `FIT_MODE = "t3"`：降档会改变训练范围（t4/t5 只训顶部若干层），
> 而机制实验要求主干整体可学 ⇒ 写死后训练范围可复现、不会悄悄漂移；
> 真跑不通就直接报错并给出诊断。两条基线是两个独立的机制实验，
> 各自锁定 t3，口径记录在「口径清单」里。
> 下面的档位表仍保留：它是 `FIT_MODE="auto"` 时的选档逻辑，也是降档链的依据。

**显存模型**：

```
每参数开销 = 权重 fp32 4
           + （开 AMP 时）fp16 权重副本 2   ← 反向传播要用它算 grad_input，
                                               被 autograd 存到 backward 结束
           + 梯度 fp32 4 × 可训比例
           + 优化器状态 × 可训比例   AdamW 8 / 8bit AdamW 2 / Adafactor ≈0.2
```

（AMP 的 fp16 副本 2 B/param 不能漏算：漏算会低估约 3 GiB，把「装不下」误判成「装得下」。）

| 档位 | 含义 | optim | AMP | 可训层 | 固定开销 | 什么时候选它 |
|---|---|---|---|---|---|---|
| t0 | 全参微调 · fp32 AdamW | `adamw_torch` | 开 | 100% | 18 B/param | 40GB+ 的卡（A100 等） |
| t1 | 全参微调 · 8bit AdamW | `adamw_bnb_8bit` | 开 | 100% | 12 B/param | 装了 bitsandbytes 且 ≥24GB 卡 |
| t2 | 全参微调 · Adafactor | `adafactor` | 开 | 100% | 10.2 B/param | 16GB+ 的卡（L4 24GB 会选 t1） |
| **t3** | **全参微调 · Adafactor · 关 AMP** | `adafactor` | 关 | 100% | **8.2 B/param** | **T4 14.56GB 就是这一档**（仍全参微调，只是不用混合精度） |
| t4 | 顶部 50% 层 · Adafactor | `adafactor` | 开 | 50% | 8.1 B/param | 想要 AMP 速度、可以接受冻一半层时 |
| t5 | 顶部 50% 层 · Adafactor | `adafactor` | 关 | 50% | 6.1 B/param | 更省显存 |
| t6 | 顶部 25% 层 · Adafactor | `adafactor` | 关 | 25% | 5.05 B/param | 兜底 |

> ⚠️ **上表的前提是「单卡模式」**。若进程能看见 2 张卡，transformers 的 Trainer 会自动套
> `nn.DataParallel`，其反向传播要**再复制一整份梯度**（Qwen-1.5B 多 5.75 GiB）→ 必然 OOM。
> Step1 用 `CUDA_VISIBLE_DEVICES` 把进程限制成单卡，`build_training_args` 里再钉 `args._n_gpu = 1` 兜底。

**选档规则（仅 `FIT_MODE="auto"` 时启用）**：用**两条基线中较大者的参数量**
（`FIT_REF_PARAMS = 1.55e9`）作参考，配合本机显存选第一个装得下的档位
（判据用「micro-batch=1 的最小激活」，因为激活是 batch 的事、由探测阶段逐档降）。
若估算仍偏乐观，Step8 会用**真实模型真跑 2 步**验证，OOM 就自动降一档重来；
全部档位都失败时给出精确诊断（而不是一句 out of memory）。

> Qwen 的有效 batch 是 8（micro 8 × accum 1）：梯度累积 ≥2 会让绑定 embedding 的
> 稠密梯度在反向时同时存在两份（+890 MiB）⇒ t3 装不下；
> accum=1 时 autograd 直接复用那块临时缓冲（全程只 1 份）⇒ 装得下。

> t3 档为什么「关 AMP」：开 AMP 时 PyTorch 会把权重的 fp16 副本留到反向传播用，
> 多占 2 B/param。Qwen-1.5B 正是差这 2 B/param（3.1 GiB）而装不下 14.56 GiB 的卡。
> 关掉 AMP 就省出这 3.1 GiB，代价是纯 fp32 训练慢一些（T4 没有 TF32）。

### 「两张 T4 能不能凑成 30G 用？」——不能

**结论先说**：CUDA 显存**不能跨卡合并**。一个张量只能待在一张卡上，
「2 × 14.56 GiB」不会变成「一张 30 GiB 的卡」。

**更糟的是**：只要进程能看见 2 张卡，transformers 的 Trainer 就会**自动**把模型套上
`nn.DataParallel`：

```python
# transformers/trainer.py :: _wrap_model()
if self.args.n_gpu > 1 and not is_loaded_in_8bit and not is_loaded_in_4bit:
    model = nn.DataParallel(model)
```

`nn.DataParallel` 的反向传播要经过 `Broadcast` → `ReduceAddCoalesced`，
**额外分配并复制一整份梯度** → 梯度显存翻倍。Qwen-1.5B 多 5.75 GiB → 14.56 GiB 的卡必然 OOM。

所以本项目的做法是：**用 `CUDA_VISIBLE_DEVICES` 把进程限制成只看见 1 张卡**
（Step1 已设，`build_training_args` 里再钉一次 `args._n_gpu = 1` 兜底），
让 Trainer 走单卡路径，梯度显存不翻倍。

| 想做的事 | 能不能把两张卡的显存合起来用 | 说明 |
|---|---|---|
| **DDP（数据并行）** | **不能** | 每张卡各存**一整份**模型+梯度+优化器状态，只是把 batch 分半 → 提速，不省显存 |
| **nn.DataParallel（Trainer 自动套的）** | **不能，而且更糟** | 反向传播还要**再复制一份梯度** → 显存 ×2 |
| **ZeRO-3 / FSDP（参数分片）** | **能** | 每卡只放 1/2 → 这才是「29 GiB 一起用」 |
| **两条基线各占一张卡** | 不需要合并 | 两个独立实验各跑各的，墙钟时间直接减半 |

**为什么本 notebook 不直接上 ZeRO-3/FSDP**（三个原因）：

1. 需要**多进程**启动：`accelerate launch --num_processes 2` 或 `torchrun`。
   papermill / `jupyter nbconvert` 起的是**普通单进程**，不会自动开 DDP。
2. 本 notebook 自定义了 `TargetGroupedSampler`（让同一目标的例句落在同一个 batch，
   否则组内损失形同虚设）。DDP 下若不改成分布式采样器，
   **两个进程会取到同一批样本** → 训练结果悄悄错掉，而且**不报错**。
3. 改造要动的地方多（采样器、Trainer 子类、跨进程结果汇总），属于大改，
   而当前阻塞点（Qwen 装不下）用单卡 + 档位机制已经能解。

**最省事的方案**：

```bash
COMPARTMENT_GPU=0   # DeBERTa 版（单卡，梯度显存不翻倍）
COMPARTMENT_GPU=1   # Qwen 版（同一台机器、两条基线各占一张卡，各自独立运行）
```

两条基线各占一张卡同时跑，**两张卡的算力都用满**，墙钟时间减半（同型号卡对数值无影响）；
「29 GiB 合成一张卡」等以后真的要训 7B/9B 时再用 FSDP 脚本形态来做。

### 多卡怎么用：先分清「提速」和「省显存」

| 用法 | 每卡上放什么 | 能降低单卡显存需求吗 | 能提速吗 |
|---|---|---|---|
| 单进程 | 全部在一张卡 | — | 基准 |
| **DDP（数据并行）** | **每卡各存一整份**模型+梯度+优化器状态，只是数据分半 | **不能**（每卡仍是完整那一份） | 能（约 1.8×） |
| **ZeRO-3 / FSDP（参数分片）** | 参数/梯度/优化器状态**切分**到多卡 | **能**（这才是用满多卡显存的办法） | 能 |
| 两条基线各占一张卡 | 两条基线是**互不相干的实验**，各占一卡同时跑 | 不需要（各跑各的） | 能（墙钟减半） |

**对 Qwen2.5-1.5B 具体意味着什么**：DDP **救不了**它 —— 每卡仍要装 23.0 GiB 的固定开销。
本 notebook 的做法是走 **t3 档**（全参微调 + Adafactor + 关 AMP = 8.2 B/param ≈ 11.8 GiB，单卡就够）；
真想用满 29 GiB 训更大的模型，则要上 ZeRO-3/FSDP 脚本形态（见上一节的三条理由）。

**注意**：papermill / `jupyter nbconvert` 起的是**普通单进程**，**不会自动开 DDP**。
真要用 DDP 得显式 `accelerate launch --num_processes 2 notebook.ipynb`，
而且必须把自定义采样器改成分布式采样器，否则两个进程会取到同一批样本。

### 训练时间够不够：开跑前就有投影

Step8 探测时量出「每样本耗时」，按**本计划（3 个联合臂 × 4 轮）**投影总时长并打印；
探针口径偏保守 ⇒ 预算判定用「探针值与云端实测值（CLOUD_SPS）的较小者」。
超预算按内置降级阶梯裁臂（先去 joint_ordmix 混合检验臂，见投影处的打印）。


### 图上的文字：一律英文

Kaggle 等云端镜像的默认字体（DejaVu Sans）**不含中文字形**，中文会静默变成方块且不报错
⇒ 图上文字固定英文（`PLOT_LANG="en"`），字体只挑能显示拉丁字母与德语变音的通用字体。

### 自动跑了哪些运行（V12 三臂对照）

`RUN_PLAN` / `V12_ARMS`（Step1）是唯一的开关，三臂全部为联合训练（覆盖全部 6 个评测设置）：

| 臂 | 读出头 | 损失 | 检验什么 |
|---|---|---|---|
| `joint_base` | Linear(1024→2) | huber+guard（冻结配方） | 对照 |
| `joint_ord` | Linear(1024→12)→6 bin | 软 CE + guard 保险丝 | 序数包 vs 点回归 |
| `joint_ordmix` | 同上 | 软 CE + huber + guard | 混合能否兼得 |

数据切分恒为 seed 42 ⇒ 三臂验证集逐行对齐，配对差值合法。

**想少跑几次**：把 `COMPARTMENT_RUN_PLAN` 设成臂名子集（逗号分隔），其余代码一行不用动。

### 云端跑法（推荐命令）

```bash
# 1) 拉代码与数据（数据放在仓库的 train/ 下）
git clone <your-repo> ComPartMENT && cd ComPartMENT

# 2) 环境变量（都可省略，notebook 会自动探测）
export COMPARTMENT_TRAIN_DIR=/kaggle/input/datasets/kintsugi0v0/compartment-data-v2/train_data_v2
export COMPARTMENT_OUTPUT=/kaggle/working/outputs
export COMPARTMENT_HF_MODE=auto        # auto=先探测官方源，不通换镜像
export TOKENIZERS_PARALLELISM=false

# 3) 依赖
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" \
            scikit-learn pandas matplotlib seaborn scipy jupyter nbconvert

# 4) 先用 30 秒跑通整条链路（2 层随机主干，不下权重）
#    用环境变量开冒烟模式（COMPARTMENT_SMOKE=1，2 层随机主干，不下权重）
COMPARTMENT_SMOKE=1 jupyter nbconvert --to notebook --execute \
  "v12/ComPartMENT(XLM-RoBERTa-large-V12序数软标签目标)/ComPartMENT_XLM-RoBERTa-large_V12.ipynb" \
  --ExecutePreprocessor.timeout=1800

# 5) 正式跑（不设 COMPARTMENT_SMOKE 即为正式模式）
nohup jupyter nbconvert --to notebook --execute \
  "v12/ComPartMENT(XLM-RoBERTa-large-V12序数软标签目标)/ComPartMENT_XLM-RoBERTa-large_V12.ipynb" \
  --ExecutePreprocessor.timeout=7200 > train.log 2>&1 &
```

**数据路径不用改代码**：notebook 会按「环境变量 `COMPARTMENT_TRAIN_DIR` → Kaggle 数据集
`compartment-data-v2`（原始 TSV 在 `train_data_v2/` 根目录）→ 仓库内 `train/`」的顺序自动探测，
找不到时再用递归 glob 兜底（排除 `filtered/` 下的合成数据）。Kaggle 上挂上
`compartment-data-v2` dataset 直接跑即可，不用改任何一行。

**云端唯一必须改的地方**（Step1）：

| 位置 | 改什么 | 为什么 |
|---|---|---|
| `SMOKE` | 不设 `COMPARTMENT_SMOKE`（默认正式模式） | 冒烟模式（`COMPARTMENT_SMOKE=1`）不下载权重、不产出可用模型 |
| `HF_MODE` | 一般不用改 | `auto` 会先探测官方源；海外云端直通用它最稳 |

### 显存不够时的降级顺序（代价从小到大）

Step8 的**训练档位机制**已经把这套顺序自动化了（见上文的档位表），
这里只列「为什么是这个顺序」：

1. **降 `per_device_train_batch_size`，用 `gradient_accumulation_steps` 把有效 batch 补回去**
   —— 数学等价、几乎无损，第一选择（显存探测会自动做这件事）
   —— ⚠️ 但只省**激活**显存，救不了「权重+梯度+优化器状态」这块固定开销
2. **开 `gradient_checkpointing`**（记得同时 `model.config.use_cache = False`）
   —— 激活降到约 1/N，慢约 30%
3. **换优化器**（`adafactor` 把 m/v 从 8 B/param 压到 ≈0.2；`adamw_bnb_8bit` 压到 2）
   —— 唯一能显著降低**固定开销**的办法
4. **关掉混合精度**（AMP 的 fp16 权重副本占 2 B/param，Qwen-1.5B 就是差这 3.1 GiB）
   —— 代价是纯 fp32 训练慢一些
5. **只训顶部若干层**（`apply_fit_scope`）—— 会冻结底层与 embedding，是**方法层面的改变**，
   必须记进口径清单（本版两条基线各自独立，改动只影响本条基线）
6. **降 `MAX_LENGTH`** —— 最有效，但要改全局配置并**从头重跑**（分词在 Step3 已固化）
7. **换更小的同族模型** —— `microsoft/deberta-v3-base` / `Qwen2.5-0.5B-Instruct`，
   管线一行不用改

### 三个判断「实验有没有做对」的指标

1. **梯度流验证必须通过**：回归头与主干的梯度绝对值和都 > 0，且 loss 不是 NaN。
   否则模型可能训了十几分钟而参数纹丝不动。
2. **指标 2 不能是 `nan`**：组内预测 std 恒为 0 会让相关系数未定义。
   图 C 上如果点全挤在水平线上，就说明模型没有真正对语境敏感。
3. **训练后必须显著优于随机初始化**（参考对照区）：「训练后模型」的指标 1 要明显高于
   「随机初始化」，否则模型没有学到任何东西（此时优先查：标签列是否对齐、
   输入模板是否恒定、学习率是否过大）。

### 后续可以做什么（对应本项目的建模决策）

| 方向 | 做法 | 开关位置 |
|---|---|---|
| 序数桶数扫描 | 若序数臂过关，扫 SOFT_BINS ∈ {6, 10}（桶数 = 评分粒度） | Step1 |
| 混合比例扫描 | ordmix 过关后扫 CE:huber 权重 | `_ordinal_loss` |
| σ-头（异方差 NLL） | 本项目的 B2 候选，与序数头同族可叠加 | — |
| 恢复 order / mse / xrank | 上游判读已判负（噪声内）⇒ 不建议重启 | — |
| 换模型规模 / 主干 | `MODEL_REGISTRY`（V12 只注册了 xlmr） | Step1 |
| 帕累托前沿 | 输出缩放系数当旋钮画「指标1 vs 指标2」前沿 | Step10 之后 |

### 复现命令

```bash
cd ComPartMENT
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" scikit-learn pandas matplotlib seaborn scipy
jupyter nbconvert --to notebook --execute \
  "v12/ComPartMENT(XLM-RoBERTa-large-V12序数软标签目标)/ComPartMENT_XLM-RoBERTa-large_V12.ipynb"
```